# Notebook B - annual participation, data-type evolution, and published-study analysis recipes

Run A -> B -> C. Reads saved sources without making API requests. Fourteen main discussion slides; detailed paper comparisons are exported separately. Historical submission events, actual downloads, and request-linked publications are never inferred from current metadata.


**Slide 4 / 7 revision:** source MeSH is visible even without ontology resolution; unknown validation is not displayed as a zero. Contributor/accessor columns each retain annual PI/institution charts and have an all-PI ranked volume plot.


## Settings

In [ ]:
# @title Settings -- local analysis only
DRIVE_ROOT = '/content/drive/MyDrive/dbgap_analysis'
ANALYSIS_MODE = 'reuse'  # 'reuse' or 'rebuild'; never changes source data
ACCESSION_DATE_BASIS = 'ftp_artifact'  # 'ftp_artifact' or 'local_first_observed'
DATA_CUTOFF = ''  # Use an actual acquisition cutoff, never silently the render date.
SHOW_FIGURES = True
INSTALL_MISSING = True
MOUNT_DRIVE = True
OUTPUT_LABEL = ''  # Test fixtures use SYNTHETIC; never use them as findings.
GROWTH_START_YEAR = 2006
GROWTH_END_YEAR = 2026
SEMANTIC_INCLUDE_REVIEWED = False  # Optional separate lane; never accepts unreviewed candidates.
SEMANTIC_ANCESTOR_DECAY = 0.5  # Descriptive fingerprint, not calibrated RCDC weights.
SEMANTIC_MAX_ANCESTOR_HOPS = 6
SEMANTIC_MIN_BRIDGE_DEPTH = 1  # Broad fallbacks retained but explicitly labelled.
SEMANTIC_THEME_DEPTH = 2
MODEL_ENABLED = False  # Optional diagnostic only; not displayed on slide 5
MODEL_COVARIATES = ['subjects','variables','consent_codes','data_versions','BDC','SRA','is_collection','discovery_year','admin_ic']
MODEL_MIN_COVERAGE = 0.50   # Availability screen, never an association/p-value screen.
MODEL_MIN_OBSERVED = 30
MODEL_MIN_N = 80
MODEL_MIN_CLASS_PER_PARAMETER = 10  # Configurable sparse-fit safeguard, not a universal rule.
MODEL_MIN_CATEGORY = 30
MODEL_MAX_CATEGORIES = 6
MODEL_MIN_CLUSTERS = 30
# Actual submitters, when available, take precedence over public study-PI attribution.
# No sponsor, grant PI, or manuscript author is silently substituted for a submitter.
SUBMITTER_MODE = 'auto'  # auto or verified_only; auto falls back to labelled public PI proxy

import os, sys, re, ast, json, math, hashlib, unicodedata, textwrap, itertools
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import importlib.util, subprocess
_needed = [p for p in (['numpy','pandas','matplotlib','pyarrow'] + (['statsmodels'] if MODEL_ENABLED else [])) if importlib.util.find_spec(p) is None]
if _needed and INSTALL_MISSING:
    subprocess.run([sys.executable,'-m','pip','install','-q',*_needed],check=True)
if MOUNT_DRIVE and DRIVE_ROOT.startswith('/content/drive/'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError: pass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, StrMethodFormatter
from IPython.display import display
BASE=Path(DRIVE_ROOT)
OUT=BASE/'preliminary14_analysis'
OUT.mkdir(parents=True,exist_ok=True)
RECIPE='preliminary-fourteen-contributors-accessors-v2'
RECIPE_SHA256='618f6b1f6f3173ebbe251abd5d0217c035ba58b0e8f175e19ebf25626aa2fa50'
SOURCES={
 'roots':['joined_accessions_dar','dbgap_root_accessions'],
 'dar':['dar_records'], 'links':['dbgap_study_pubmed_links_full'],
 'pubmed':['pubmed_metadata_full'], 'icite':['icite_rcr_metrics'],
 'trials':['dbgap_ctgov_direct_links'], 'sstr':['sstr_study_summary','sstr_summary_analysis_ready'],
 'page':['study_page_metadata_analysis_ready','study_page_metadata'],
 'auth':['dbgap_authorized_access_api_metadata'],
 'attribution':['dbgap_public_attribution'], 'grants':['pubmed_grant_acknowledgements'],
 'submission_entities':['submission_entities'], 'entity_aliases':['entity_aliases'],
 'ftp':['accession_timing_ftp_root_results'], 'ledger':['dbgap_first_observed_ledger'],
 'reporter':['reporter_pub_links_full'],
 'counts':['study_counts_analysis_ready','study_counts'],
 'versions_raw':['dbgap_studies_raw'],
 'families':['documented_study_families'],
 'submission_events':['replication_inputs/submission_events','submission_events'],
 'assay_submission_events':['replication_inputs/assay_submission_events','assay_submission_events'],
 'rep_lifecycle':['replication_inputs/study_lifecycle'],
 'rep_access':['replication_inputs/access_events'],
 'rep_publications':['replication_inputs/publication_events'],
 'rep_institutions':['replication_inputs/institutions'],
 'rep_samples':['replication_inputs/sample_assays'],
 'rep_topics':['replication_inputs/publication_topics'],
 'rep_metrics':['replication_inputs/publication_metrics']}
TABLES={};INPUTS=[];GAPS=[];CHECKS=[];OUTPUTS={}
PALETTE={'blue':'#235A85','teal':'#137D83','amber':'#BA7D24','grey':'#8C9BA8','text':'#21364B'}
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':180,'font.family':'DejaVu Sans',
 'font.size':12,'axes.titlesize':15,'axes.labelsize':12,'xtick.labelsize':10,'ytick.labelsize':10,
 'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})
assert ANALYSIS_MODE in {'reuse','rebuild'}
assert ACCESSION_DATE_BASIS in {'ftp_artifact','local_first_observed'}

assert SUBMITTER_MODE in {'auto','verified_only'}

assert (GROWTH_START_YEAR,GROWTH_END_YEAR)==(2006,2026)

# Local interactive companion; does not change analytical tables or static slides.
BUILD_INTERACTIVE_EXPLORER = True


## Shared helpers and inputs

In [ ]:
# @title Shared read-only input handling and simple checks

def text(x):
    if x is None or x is pd.NA or x is pd.NaT:return ''
    if isinstance(x,(float,np.floating)) and not np.isfinite(x):return ''
    return str(x).strip()
def rootid(x):
    m=re.fullmatch(r'(phs\d+)(?:\.v\d+(?:\.p\d+)?)?',text(x).lower())
    return m.group(1) if m else None
def pmidid(x):
    s=text(x)
    if isinstance(x,(bool,np.bool_)):return None
    if re.fullmatch(r'\d+\.0',s):s=s[:-2]
    return str(int(s)) if s.isdigit() and int(s)>0 else None
def tri(x):
    s=text(x).lower()
    if s in {'true','1','1.0','yes','y'}:return True
    if s in {'false','0','0.0','no','n'}:return False
    return pd.NA
def boolean(values):return pd.Series(values).map(tri).astype('boolean')
def value(frame,name,default=pd.NA):return frame[name].copy() if name in frame else pd.Series(default,index=frame.index)
def number(frame,name):return pd.to_numeric(value(frame,name),errors='coerce')
def decoded(x):
    if isinstance(x,(list,tuple,dict,np.ndarray)):return x
    s=text(x)
    if not s:return None
    for parser in (json.loads,ast.literal_eval):
        try:return parser(s)
        except (ValueError,SyntaxError,TypeError):pass
    return None
def canonical_label(x):return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',text(x))).strip().casefold()
def safe(x):
    if isinstance(x,dict):return {str(k):safe(v) for k,v in x.items()}
    if isinstance(x,(list,tuple,np.ndarray,set)):return [safe(v) for v in x]
    if isinstance(x,pd.DataFrame):return safe(x.to_dict('records'))
    if isinstance(x,pd.Series):return safe(x.tolist())
    if x is None or x is pd.NA or x is pd.NaT:return None
    if isinstance(x,(pd.Timestamp,datetime)):return x.isoformat()
    if isinstance(x,(np.integer,)):return int(x)
    if isinstance(x,(np.bool_,)):return bool(x)
    if isinstance(x,(float,np.floating)):return float(x) if np.isfinite(x) else None
    return x
def hashfile(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
    return h.hexdigest()
def write_json(p,obj):
    content=json.dumps(safe(obj),indent=2,sort_keys=True,allow_nan=False)
    if p.exists() and p.read_text()==content:return
    tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(content);os.replace(tmp,p)
def export_table(name,frame):
    # New B-owned CSVs; raw A parquet files remain untouched.
    p=OUT/(name+'.csv'); content=frame.to_csv(index=False)
    if not p.exists() or p.read_text()!=content:
        tmp=p.with_suffix('.tmp');tmp.write_text(content);os.replace(tmp,p)
    OUTPUTS[p.name]=hashfile(p)
def gap(section,message):
    row={'slide':section,'reason':message}
    if row not in GAPS:GAPS.append(row)
    print(f'[{section}] {message}')
def check(name,ok,details=''):
    CHECKS.append({'check':name,'passed':bool(ok),'details':str(details)})
    if not ok:raise ValueError(f'{name}: {details}')
def read_input(key,stems):
    for stem in stems:
        for suffix in ('.parquet','.csv'):
            p=BASE/(stem+suffix)
            if p.is_file():
                f=pd.read_parquet(p) if suffix=='.parquet' else pd.read_csv(p,dtype={'root_accession':'string','pmid':'string','wlid':'string'})
                INPUTS.append({'key':key,'path':str(p),'sha256':hashfile(p),'rows':len(f),'columns':list(f)})
                return f
    return pd.DataFrame()
def with_root(f):
    f=f.copy()
    c=next((c for c in ['root_accession','queried_accession','accession','latest_accession','raw_accession','index'] if c in f),None)
    if c:f['root_accession']=f[c].map(rootid)
    return f
def current_root_rows(f,rootframe):
    f=with_root(f)
    if 'root_accession' not in f:return f
    # Resolve multiple retained versions by explicit current accession, else numeric version order.
    accession_field=next((k for k in ['accession','accession_sstr','latest_accession_used','raw_accession'] if k in f),None)
    if accession_field is not None:
        latest=rootframe.set_index('root_accession').get('latest_accession',pd.Series(dtype='string'))
        f['_is_current']=f[accession_field].astype(str).eq(f.root_accession.map(latest).astype(str))
        f['_version']=pd.to_numeric(f[accession_field].astype(str).str.extract(r'\.v(\d+)')[0],errors='coerce').fillna(-1)
        f['_participant']=pd.to_numeric(f[accession_field].astype(str).str.extract(r'\.p(\d+)')[0],errors='coerce').fillna(-1)
        f=f.sort_values(['_is_current','_version','_participant']).drop_duplicates('root_accession',keep='last').drop(columns=['_is_current','_version','_participant'])
    elif f.root_accession.duplicated().any():
        # Identical rows okay; conflicting authoritative root rows are not silently selected.
        fingerprint=f.apply(lambda row:json.dumps(safe(row.to_dict()),sort_keys=True,default=str),axis=1)
        f=f.loc[~fingerprint.duplicated()].copy()
        check('One row per root in root-level source',not f.root_accession.duplicated().any())
    return f

def status(x):
    s=text(x).lower()
    if s in {'ok','success','complete','resolved','ok_unparsed'} or s.endswith('_fallback_ok'):return 'ok'
    if s in {'no_requesters','no_publications','empty'} or s.endswith('_fallback_no_requesters'):return 'empty'
    if s in {'http_404','not_found'}:return 'not_found'
    if any(t in s for t in ['error','failed','timeout','exceeded','non_json','http_5','http_4']):return 'failed'
    return 'unknown'

def year(x):
    if isinstance(x,(pd.Timestamp,datetime,np.datetime64)):
        return pd.Timestamp(x).year if pd.notna(x) else np.nan
    m=re.search(r'(?<!\d)((?:18|19|20|21)\d{2})(?!\d)',text(x))
    return int(m.group(1)) if m else np.nan

def histogram_counts(values,edges=None):
    x=pd.to_numeric(pd.Series(values),errors='coerce').dropna()
    x=x[np.isfinite(x)&(x>=0)]
    if edges is None:edges=[-0.5,0.5,1.5,4.5,9.5,24.5,49.5,99.5,np.inf]
    labels=['0','1','2-4','5-9','10-24','25-49','50-99','100+']
    counts,_=np.histogram(x,edges)
    return {'labels':labels,'counts':counts.tolist(),'n':len(x),'max':float(x.max()) if len(x) else None}

def cumulative(values,label):
    """Fixed 2006-2026 display; cumulative bars retain the pre-2006 baseline."""
    v=pd.Series(values);y=v.map(lambda x:int(text(x)) if re.fullmatch(r'\d{4}',text(x)) else year(x));missing=int(y.isna().sum())
    upper=min(GROWTH_END_YEAR,pd.Timestamp(DATA_CUTOFF).year) if DATA_CUTOFF else GROWTH_END_YEAR
    bad=y.notna()&((y<1800)|(y>upper))
    # Apply an exact cutoff when the source contains parseable dates; year-only values stay year-granular.
    if DATA_CUTOFF:
        cutoff=pd.to_datetime(DATA_CUTOFF,utc=True)
        for idx,item in v.items():
            if re.fullmatch(r'\d{4}',text(item)):continue
            dt=pd.to_datetime(text(item),errors='coerce',utc=True)
            if pd.notna(dt) and dt>cutoff:bad.loc[idx]=True
    valid=y[y.notna()&~bad].astype(int);pre=int((valid<GROWTH_START_YEAR).sum())
    counts=valid.value_counts();running=pre;rows=[]
    for t in range(GROWTH_START_YEAR,GROWTH_END_YEAR+1):
        if t>upper or not len(valid):a=None;c=None
        else:a=int(counts.get(t,0));running+=a;c=running
        rows.append({'year':t,'annual':a,'cumulative':c})
    return {'rows':rows,'n':len(valid),'label':label,'before_window':pre,
            'within_window':int(valid.between(GROWTH_START_YEAR,GROWTH_END_YEAR).sum()),
            'missing_dates':missing,'excluded_invalid_or_future':int(bad.sum()),
            'window_start':GROWTH_START_YEAR,'window_end':GROWTH_END_YEAR,
            'cutoff':DATA_CUTOFF or None,'latest_year_may_be_incomplete':True,
            'note':'Cumulative includes valid pre-2006 observations; annual bars show only 2006-2026. Missing dates excluded; not a historical inventory of repository contents.'}

def gini(values):
    x=np.sort(np.asarray(values,dtype=float));x=x[np.isfinite(x)&(x>=0)]
    return float(2*np.dot(np.arange(1,len(x)+1),x)/(len(x)*x.sum())-(len(x)+1)/len(x)) if len(x) and x.sum()>0 else None

def rank_records(f,key,label,kind):
    if f.empty or key not in f:return {'rows':[],'unique':None,'gini':None,'unit':kind,'covered_roots':0}
    f=f.dropna(subset=[key,'root_accession']).copy();f=f[f[key].map(text).ne('')]
    # Count distinct root-entity links, not versions or duplicate evidence lines.
    pairs=f.drop_duplicates(['root_accession',key])
    counts=pairs.groupby(key).size().sort_values(ascending=False)
    labels=pairs.groupby(key)[label].first() if label in pairs else pd.Series(counts.index,index=counts.index)
    rows=[{'entity_key':text(k),'label':text(labels.get(k,k)),'accessions':int(v)} for k,v in counts.items()]
    return {'rows':rows,'unique':len(counts),'gini':gini(counts.values),'unit':kind,
            'covered_roots':int(pairs.root_accession.nunique()),'root_entity_links':len(pairs)}

TABLES={k:read_input(k,v) for k,v in SOURCES.items()}
for k in list(TABLES):TABLES[k]=with_root(TABLES[k])
check('Root accession table present',not TABLES['roots'].empty)
R=TABLES['roots'].copy();check('Root accession key unique',not R.root_accession.duplicated().any())
check('Root accession syntax',R.root_accession.notna().all())
R=R.sort_values('root_accession').reset_index(drop=True);N=len(R);rootset=set(R.root_accession)
for k,f in list(TABLES.items()):
    if 'root_accession' in f and k!='roots':
        out=~f.root_accession.isin(rootset)
        if out.any():gap('inventory',f'{k}: {int(out.sum())} rows outside current root universe excluded from working copy.')
        TABLES[k]=f.loc[~out].copy()
cp={}
p=BASE/'dar_fetch_checkpoint.json'
if p.exists():
    raw=json.loads(p.read_text());c=raw.get('done_accessions',{})
    if isinstance(c,dict):
        # Use only the current version's checkpoint (or an explicit root key).
        cp={}
        for row in R.to_dict('records'):
            root=row['root_accession'];current=text(row.get('latest_accession'))
            if current in c:cp[root]=status(c[current])
            elif root in c:cp[root]=status(c[root])
    INPUTS.append({'key':'dar_checkpoint','path':str(p),'sha256':hashfile(p)})
# Include native recovery outputs, raw/parsed vocabularies, and optional human decisions in the input signature.
_sem_dir=BASE/'semantic_sources'
_sem_manifest=_sem_dir/'manifest.json'
if _sem_manifest.exists():
    _sm=json.loads(_sem_manifest.read_text())
    _sem_paths=[_sem_manifest]+[_sem_dir/name for name in _sm.get('outputs',{})]
    for _v in _sm.get('vocabularies',[]):_sem_paths.extend([Path(_v['raw_path']),Path(_v['parsed_path'])])
    for _p in _sem_paths:
        if not _p.is_file():raise FileNotFoundError('Semantic source missing: '+str(_p))
        INPUTS.append({'key':'semantics.'+_p.name,'path':str(_p),'sha256':hashfile(_p)})
if (BASE/'semantic_mesh_adjudications.csv').is_file():
    _p=BASE/'semantic_mesh_adjudications.csv';INPUTS.append({'key':'semantic_reviews','path':str(_p),'sha256':hashfile(_p)})
SETTINGS={k:globals()[k] for k in ['ACCESSION_DATE_BASIS','DATA_CUTOFF','OUTPUT_LABEL','SUBMITTER_MODE','GROWTH_START_YEAR','GROWTH_END_YEAR','MODEL_ENABLED','MODEL_COVARIATES',
 'SEMANTIC_INCLUDE_REVIEWED','SEMANTIC_ANCESTOR_DECAY','SEMANTIC_MAX_ANCESTOR_HOPS','SEMANTIC_MIN_BRIDGE_DEPTH','SEMANTIC_THEME_DEPTH','MODEL_MIN_COVERAGE','MODEL_MIN_OBSERVED','MODEL_MIN_N','MODEL_MIN_CLASS_PER_PARAMETER','MODEL_MIN_CATEGORY','MODEL_MAX_CATEGORIES','MODEL_MIN_CLUSTERS']}
import importlib.metadata
SOFTWARE={name:importlib.metadata.version(name) for name in ['numpy','pandas','matplotlib','statsmodels'] if importlib.util.find_spec(name)}
SETTINGS['software']=SOFTWARE
SIGNATURE=hashlib.sha256(json.dumps({'recipe':RECIPE_SHA256,'inputs':INPUTS,'settings':SETTINGS},sort_keys=True).encode()).hexdigest()
oldmanifest=json.loads((OUT/'manifest.json').read_text()) if (OUT/'manifest.json').exists() else {}
REUSE=ANALYSIS_MODE=='reuse' and oldmanifest.get('signature')==SIGNATURE and (OUT/'summary.json').exists()
if REUSE:
    REUSE=all((OUT/p).is_file() and hashfile(OUT/p)==h for p,h in oldmanifest.get('outputs',{}).items())
RESULTS=json.loads((OUT/'summary.json').read_text()) if REUSE else {'version':RECIPE,'input_signature':SIGNATURE,
        'mode':'saved_table_analysis','data_cutoff':DATA_CUTOFF or None,'label':OUTPUT_LABEL,'n_roots':N}
print('Reusing matching derived outputs.' if REUSE else 'Calculating new preliminary-deck summaries.')
display(pd.DataFrame([{'source':k,'rows':len(f)} for k,f in TABLES.items()]))


## Canonical working views

In [ ]:
# @title Canonical analysis views -- independent of slide rendering
HAVE_LINKS=any(i['key']=='links' for i in INPUTS)
D=TABLES['dar'].copy();L=TABLES['links'].copy();P=TABLES['pubmed'].copy();I=TABLES['icite'].copy();T=TABLES['trials'].copy()
for f in [L,P,I]:
    if 'pmid' in f:f['pmid']=f.pmid.map(pmidid)
if 'pmid' not in L:L=pd.DataFrame(columns=['root_accession','pmid'])
L=L.dropna(subset=['root_accession','pmid']).drop_duplicates(['root_accession','pmid'])
for f,name in [(P,'PubMed'),(I,'iCite')]:
    if 'pmid' in f:
        check(name+' PMID key unique',not f.dropna(subset=['pmid']).pmid.duplicated().any())
# Metadata in the link table can supplement a missing standalone PubMed record table.
if P.empty and 'pubmed_fetch_note' in L:P=L.drop_duplicates('pmid').copy()
P=P[P.pmid.isin(set(L.pmid))].copy() if 'pmid' in P else pd.DataFrame(columns=['pmid'])
I=I[I.pmid.isin(set(L.pmid))].copy() if 'pmid' in I else pd.DataFrame(columns=['pmid','rcr'])
p_ok=set(P.loc[value(P,'pubmed_fetch_note').map(status).eq('ok'),'pmid'])
if not p_ok and 'pubmed_fetch_note' in L:p_ok=set(L.loc[L.pubmed_fetch_note.map(status).eq('ok'),'pmid'])
PL=L[L.pmid.isin(p_ok)].copy()
S=R[['root_accession']].copy().set_index('root_accession')
page=current_root_rows(TABLES['page'],R)
page=page.set_index('root_accession') if 'root_accession' in page else pd.DataFrame()
rawcounts=D.groupby('root_accession').size() if 'root_accession' in D else pd.Series(dtype=float)
S['dar_count']=number(R,'n_dar_total').set_axis(R.root_accession).reindex(S.index)
S.loc[rawcounts.index,'dar_count']=rawcounts
S['dar']=pd.Series(pd.NA,index=S.index,dtype='boolean')
S.loc[S.dar_count.notna(),'dar']=S.loc[S.dar_count.notna(),'dar_count'].gt(0)
for r,st in cp.items():
    if r in S.index:
        if st=='failed' and not (pd.notna(S.loc[r,'dar_count']) and S.loc[r,'dar_count']>0):S.loc[r,'dar']=pd.NA;S.loc[r,'dar_count']=np.nan
        elif st=='empty':S.loc[r,'dar']=False;S.loc[r,'dar_count']=0
S['publication']=pd.Series(pd.NA,index=S.index,dtype='boolean')
S['paper_count']=np.nan
pc=PL.groupby('root_accession').pmid.nunique()
S.loc[pc.index,'paper_count']=pc;S.loc[pc.index,'publication']=True
# Complete captured source needed to conclude no linked PubMed record.
for r in S.index:
    row=R.loc[R.root_accession.eq(r)].iloc[0]
    pr=page.loc[r] if r in page.index else row
    st=status(pr.get('publications_fetch_note',row.get('publications_fetch_note')))
    roots_links=L[L.root_accession.eq(r)]
    if st in {'ok','empty'}:
        if roots_links.empty and HAVE_LINKS:S.loc[r,['publication','paper_count']]=[False,0]
        elif not roots_links.empty and set(roots_links.pmid).issubset(p_ok):S.loc[r,'paper_count']=len(roots_links)
        # A failed PubMed lookup leaves count incomplete, but does not erase a positive.
        else:S.loc[r,'paper_count']=np.nan
    elif pd.notna(row.get('n_publications_corrected')) and row.get('n_publications_corrected')==0 and st!='failed':
        S.loc[r,['publication','paper_count']]=[False,0]
S['mesh']=pd.Series(pd.NA,index=S.index,dtype='boolean')
for r,row in R.set_index('root_accession').iterrows():
    terms=decoded(row.get('mesh_terms'))
    if isinstance(terms,(list,tuple,dict,np.ndarray)):S.loc[r,'mesh']=len(terms)>0
    elif pd.notna(row.get('n_mesh_terms')):S.loc[r,'mesh']=float(row['n_mesh_terms'])>0
S['nct']=pd.Series(pd.NA,index=S.index,dtype='boolean')
if 'direct_ctgov_link_present' in R:S['nct']=R.set_index('root_accession').direct_ctgov_link_present.map(tri).astype('boolean')
if 'n_direct_nct_links' in R:
    c=R.set_index('root_accession').n_direct_nct_links
    S.loc[c.notna(),'nct']=pd.to_numeric(c[c.notna()],errors='coerce').gt(0)
if 'nct_id' in T:
    T['nct_id']=T.nct_id.astype('string').str.upper()
    T=T[T.nct_id.str.fullmatch(r'NCT\d{8}',na=False)].copy()
    S.loc[list(set(T.root_accession)&set(S.index)),'nct']=True
S['sstr']=pd.Series(pd.NA,index=S.index,dtype='boolean');S['sstr_count']=np.nan
ss=TABLES['sstr'];ss=current_root_rows(ss,R) if 'root_accession' in ss else ss
if len(ss):
    for row in ss.to_dict('records'):
        r=row['root_accession'];st=status(row.get('sstr_fetch_note'))
        if r not in S.index:continue
        if st=='ok':S.loc[r,['sstr','sstr_count']]=[True,1]
        elif st=='not_found':S.loc[r,['sstr','sstr_count']]=[False,0]
else:
    for r,row in R.set_index('root_accession').iterrows():
        st=status(row.get('sstr_fetch_note'))
        if st=='ok':S.loc[r,['sstr','sstr_count']]=[True,1]
        elif st=='not_found':S.loc[r,['sstr','sstr_count']]=[False,0]
S['nih_ack']=pd.Series(pd.NA,index=S.index,dtype='boolean')
attr=current_root_rows(TABLES['attribution'],R)
if 'root_accession' in attr and 'nih_funding_acknowledgement' in attr:
    v=attr.set_index('root_accession').nih_funding_acknowledgement.map(tri)
    S.loc[v.index,'nih_ack']=v.astype('boolean')
elif 'nih_funding_acknowledgement' in R:
    S['nih_ack']=R.set_index('root_accession').nih_funding_acknowledgement.map(tri).astype('boolean')
# Alternate paper-level GrantList evidence is retained separately, not substituted.
S['nih_paper_ack']=pd.Series(pd.NA,index=S.index,dtype='boolean')
g=TABLES['grants']
if 'pmid' in g:
    g=g.copy();g['pmid']=g.pmid.map(pmidid);g=g.set_index('pmid')
    for r,links in PL.groupby('root_accession'):
        vals=g.reindex(links.pmid).get('has_nih_grantlist',pd.Series(dtype='boolean')).map(tri)
        if vals.fillna(False).any():S.loc[r,'nih_paper_ack']=True
        elif len(vals) and vals.notna().all():S.loc[r,'nih_paper_ack']=False
AA=current_root_rows(TABLES['auth'],R)
for platform,field in [('SRA','any_consent_group_sra_available'),('BDC','platform_biodata_catalyst'),('AnVIL','platform_anvil')]:
    S[platform]=pd.Series(pd.NA,index=S.index,dtype='boolean')
    if field in AA:
        a=AA.set_index('root_accession');v=a[field].map(tri).astype('boolean')
        # Explicit true can remain observed; false needs a successfully resolved API.
        resolved=value(a,'authorized_access_fetch_note').map(status).eq('ok')
        v=v.where(v.fillna(False)|resolved,pd.NA)
        S.loc[v.index,platform]=v
S=S.reset_index()
check('Canonical signals preserve root universe',len(S)==N and S.root_accession.is_unique)
if not REUSE:export_table('root_signals',S)


## Slide 1: overview

In [ ]:
# @title Slide 1 -- totals and six accession-level coverage measures
if not REUSE:
    cov=[]
    for key,label in [('dar','1+ observed DARs'),('publication','1+ PubMed publications'),('mesh','MeSH terms'),
                      ('nct','1+ NCT linkage'),('nih_ack','NIH funding acknowledgement'),('sstr','1+ SSTR reports')]:
        v=S[key].astype('boolean');pos=int(v.fillna(False).sum());known=int(v.notna().sum())
        cov.append({'key':key,'label':label,'positive':pos if known else None,'known':known,'unknown':N-known,
                    'percentage_all_roots':100*pos/N if known and N else None})
    RESULTS['overview']={'accessions':N,'publications':len(p_ok) if HAVE_LINKS and ('pubmed_fetch_note' in P or 'pubmed_fetch_note' in L) else None,
        'linked_pmid_candidates':int(L.pmid.nunique()) if HAVE_LINKS else None,
        'pubmed_retrieved':len(p_ok),'dars':len(D) if any(i['key']=='dar' for i in INPUTS) else None,
        'clinical_trials':int(T.nct_id.nunique()) if 'nct_id' in T else None,
        'root_nct_pairs':len(T[['root_accession','nct_id']].drop_duplicates()) if 'nct_id' in T else None,
        'coverage':cov,'dar_definition':'recorded accession-requester rows; not unique consent-group DARs',
        'funding_definition':'explicit NIH/NIH-IC mention in public study Funding Sources or Acknowledgement Statement',
        'sstr_definition':'successfully retrieved current SSTR summaries, at most one per root'}
    export_table('overview_coverage',pd.DataFrame(cov))
    if S.nih_ack.notna().sum()==0:gap('1','NIH study acknowledgement not collected; run A public-attribution block. RePORTER is not a substitute.')
    display(pd.DataFrame(cov))


## Slide 2: Growth (annual and cumulative, 2006-2026)


In [ ]:
# @title Slide 2 -- Growth: annual and cumulative, fixed 2006-2026 window
if not REUSE:
    f=TABLES['ftp'] if ACCESSION_DATE_BASIS=='ftp_artifact' else TABLES['ledger']
    field='first_preferred_public_versioned_ftp_timestamp' if ACCESSION_DATE_BASIS=='ftp_artifact' else 'first_observed_utc'
    caption=('Earliest surviving preferred public FTP artifact; not release or accession assignment.' if ACCESSION_DATE_BASIS=='ftp_artifact'
             else 'First observed by the local collector; prospective discovery, not historical release.')
    av=f.drop_duplicates('root_accession')[field] if field in f else pd.Series(dtype='object')
    pub_dates=value(P[P.pmid.isin(p_ok)].drop_duplicates('pmid'),'pubmed_pub_date')
    dar_dates=value(D,'initial_approval_date')
    RESULTS['growth']={'accessions':cumulative(av,'Accessions by earliest discovered artifact' if ACCESSION_DATE_BASIS=='ftp_artifact' else 'Accessions first observed locally'),
                      'publications':cumulative(pub_dates,'Currently linked unique PubMed papers'),
                      'dars':cumulative(dar_dates,'Recorded accession-requester rows'),
                      'accession_basis':ACCESSION_DATE_BASIS,'accession_note':caption}
    for k in ['accessions','publications','dars']:
        v=RESULTS['growth'][k];export_table('cumulative_'+k,pd.DataFrame(v['rows'],columns=['year','annual','cumulative']))
        print(k,':',v['n'],'dated observations')
        if v['n']==0:gap('2',f'No dates for {k}; the curve is not replaced with another clock.')


## Slide 3: Focused accession profile

Subjects, loaded samples and variables; named consent categories with study-local code mappings; actual version-by-participant-set matrix. Parent, child, pair and collection totals remain. Former report-count and label-count histograms are no longer displayed.


In [ ]:
# @title New slide 3 -- accession content, observed versions, and hierarchy
# Local views only: A's saved tables are never rewritten or recollected.
PROFILE_SPECS = [
 ('subjects','Subjects','Loaded subjects in the selected SSTR; not unique people across studies.'),
 ('samples','Samples (SSTR)','Loaded sample count from the selected SSTR; not SSTR report count, reads, or sequencing runs.'),
 ('variables','Variables','Explicit variable count from study metadata; not inferred traits or MeSH terms.')]



def _nonnegative_int(v):
    if isinstance(v,(bool,np.bool_)):return np.nan
    try:
        n=float(v)
        return n if np.isfinite(n) and n>=0 and n.is_integer() else np.nan
    except (TypeError,ValueError): return np.nan


def _root_rows(frame,rootframe):
    if frame.empty or 'root_accession' not in frame:return {}
    cur=current_root_rows(frame,rootframe)
    return {r['root_accession']:r for r in cur.to_dict('records')}


def _study_payload(row):
    raw=decoded(row.get('sstr_raw'))
    if not isinstance(raw,dict):return {},{}
    # Raw public telemetry, not individual-level subject/sample data.
    study=raw.get('study',{});stats=raw.get('study_stats',{})
    return study if isinstance(study,dict) else {}, stats if isinstance(stats,dict) else {}


def _tokens(v):
    if isinstance(v,dict):
        out=[]
        for item in v.values():out.extend(_tokens(item))
        return out
    if isinstance(v,(list,tuple,np.ndarray)):
        return [s for item in v for s in _tokens(item)]
    return [text(v)] if text(v) else []


def _accession_refs(v):
    return sorted({m.group(1).lower() for token in _tokens(v)
                   for m in re.finditer(r'(?<![A-Za-z0-9])(phs\d+)(?:\.v\d+)?(?:\.p\d+)?',token,re.I)})


def _list_value(v):
    d=decoded(v)
    return list(d) if isinstance(d,(list,tuple,np.ndarray)) else None


def make_profile(roots,tables,signals):
    cur_sstr=_root_rows(tables['sstr'],roots);cur_counts=_root_rows(tables.get('counts',pd.DataFrame()),roots)
    cur_auth=_root_rows(tables['auth'],roots);cur_page=_root_rows(tables['page'],roots)
    retained={};version_tokens={};universe=set(roots.root_accession)
    for row in tables['sstr'].to_dict('records'):
        r=row.get('root_accession')
        if r not in universe or status(row.get('sstr_fetch_note'))!='ok':continue
        study,_=_study_payload(row);ac=study.get('accver',{}) or {}
        token=text(row.get('accession')) or text(row.get('accession_sstr'))
        if not token and ac.get('accession'):
            token=f"{ac['accession']}.v{ac.get('version','?')}.p{ac.get('participant_set','?')}"
        if re.fullmatch(r'phs\d+\.v\d+\.p\d+',token,re.I):
            retained.setdefault(r,set()).add(token.lower());version_tokens.setdefault(r,[]).append(token)
    for row in tables.get('versions_raw',pd.DataFrame()).to_dict('records'):
        token=text(row.get('raw_accession'));r=rootid(token)
        if r in universe:version_tokens.setdefault(r,[]).append(token)
    metrics=[];lineage=[];edges=[];conflicts=[];sigs=signals.set_index('root_accession')
    for row in roots.to_dict('records'):
        r=row['root_accession'];ss=cur_sstr.get(r,{});aa=cur_auth.get(r,{});co=cur_counts.get(r,{});pg=cur_page.get(r,{})
        study,stats=_study_payload(ss or row)
        ss_ok=status(ss.get('sstr_fetch_note',row.get('sstr_fetch_note')))=='ok'
        aa_ok=status(aa.get('authorized_access_fetch_note'))=='ok'
        out={'root_accession':r}
        # Prefer current source tables; joined fields provide a compatibility fallback.
        def read_num(key,candidates):
            found=[]
            for origin,rec,field in candidates:
                val=_nonnegative_int(rec.get(field))
                if pd.notna(val):found.append((origin+'.'+field,val))
            if found:
                if len({v for _,v in found})>1:conflicts.append({'root_accession':r,'metric':key,'values':json.dumps(found)})
                out[key]=found[0][1];lineage.append({'root_accession':r,'metric':key,'source':found[0][0]})
            else:out[key]=np.nan
        read_num('subjects',[('sstr',ss,'n_subjects_total'),('sstr.study_stats.cnt_subjects',(stats.get('cnt_subjects',{}) or {}),'loaded'),('roots',row,'n_subjects_total')])
        if not ss_ok and status(row.get('sstr_fetch_note'))=='failed':out['subjects']=np.nan
        success=retained.get(r,set())
        out['sstr_reports']=len(success) if success else (1 if ss_ok else (0 if status(ss.get('sstr_fetch_note',row.get('sstr_fetch_note')))=='not_found' else np.nan))
        lineage.append({'root_accession':r,'metric':'sstr_reports','source':'retained successful version records; current-summary fallback' if not success else 'sstr accession key'})
        groups=study.get('consent_groups') if ss_ok else None
        if groups is None and ss_ok:groups=_list_value(ss.get('consent_groups',row.get('consent_groups')))
        out['consent_codes']=np.nan
        if isinstance(groups,list):
            codes=[text(g.get('code',g.get('consent_code'))) for g in groups if isinstance(g,dict)]
            if len(codes)==len(groups) and all(codes):out['consent_codes']=len(set(codes))
            elif not groups:out['consent_codes']=0
        # Native explicit count is permitted, but a generic n_consent_groups is not a code count.
        if pd.isna(out['consent_codes']):
            out['consent_codes']=_nonnegative_int(ss.get('n_distinct_consent_codes',row.get('n_distinct_consent_codes')))
        versions=set();participants=set();ctags=set();vp=_list_value(row.get('versions'))
        if vp is not None:
            for pair in vp:
                if isinstance(pair,(list,tuple,np.ndarray)) and len(pair)>=2:
                    v,p=_nonnegative_int(pair[0]),_nonnegative_int(pair[1])
                    if pd.notna(v) and v>0:versions.add(int(v))
                    if pd.notna(p) and p>0:participants.add(int(p))
        tokens=version_tokens.get(r,[])+_tokens(row.get('versions'))+[text(row.get('latest_accession'))]
        for token in tokens:
            for m in re.finditer(r'phs\d+\.v(\d+)(?:\.p(\d+))?',token,re.I):
                if m.group(1) and int(m.group(1))>0:versions.add(int(m.group(1)))
                if m.group(2) and int(m.group(2))>0:participants.add(int(m.group(2)))
        # .c labels must be explicit. Consent code number and group count are not substitutes.
        consent_tokens=_tokens(aa.get('consent_group_accessions')) if aa_ok else []
        consent_tokens+=_tokens(ss.get('consent_accessions')) if ss_ok else []
        for token in consent_tokens:
            for m in re.finditer(r'\.c(\d+)(?:\D|$)',token,re.I):ctags.add(int(m.group(1)))
        out['data_versions']=len(versions) if versions else np.nan
        out['participant_versions']=len(participants) if participants else np.nan
        out['consent_versions']=len(ctags) if ctags else np.nan
        out['v_labels']=json.dumps(sorted(versions));out['p_labels']=json.dumps(sorted(participants));out['c_labels']=json.dumps(sorted(ctags))
        for key,aliases in [('variables',['num_variables','n_variables']),('traits',['num_traits','n_traits']),('documents',['num_documents','n_documents']),('datasets',['num_datasets','n_datasets'])]:
            sources=[]
            if status(co.get('fetch_error'))!='failed':sources += [('counts',co,a) for a in aliases]
            if status(row.get('fetch_error'))!='failed':sources += [('roots',row,a) for a in aliases]
            read_num(key,sources)
        parents=set();children=set();parent_known=False;child_known=False
        if ss_ok:
            for field in ['parent_study_accver','parent_study','parent']:
                if field in study:parents.update(_accession_refs(study[field]));parent_known=True
            for field in ['substudies','child_studies','children']:
                if field in study:children.update(_accession_refs(study[field]));child_known=True
        for rec in [pg,row]:
            if status(rec.get('parent_fetch_note')) in {'ok','empty'}:
                parents.update(_accession_refs(decoded(rec.get('parent_raw'))));parent_known=True
            if status(rec.get('children_fetch_note')) in {'ok','empty'}:
                children.update(_accession_refs(decoded(rec.get('children_raw'))));child_known=True
        parents.discard(r);children.discard(r)
        for p in sorted(parents):edges.append({'parent_root':p,'child_root':r,'source':'explicit parent metadata'})
        for ch in sorted(children):edges.append({'parent_root':r,'child_root':ch,'source':'explicit child metadata'})
        out['parent_count']=len(parents) if parent_known else np.nan
        child_count=_nonnegative_int(co.get('num_substudies',row.get('num_substudies')))
        if pd.isna(child_count):child_count=_nonnegative_int(pg.get('n_children',row.get('n_children')))
        out['child_count']=len(children) if child_known else child_count
        if children and pd.notna(child_count) and len(children)!=child_count:conflicts.append({'root_accession':r,'metric':'child_count','values':json.dumps({'explicit_links':len(children),'count_field':child_count})})
        out['is_collection']=tri(aa.get('is_collection')) if aa_ok else pd.NA
        out['admin_ic']=text(ss.get('admin_ic',row.get('admin_ic'))) or None
        # Only direct metadata labels; no keyword or LLM disease inference.
        out['disease']=text(row.get('primary_disease')) or None
        metrics.append(out)
    profile=pd.DataFrame(metrics)
    ed=pd.DataFrame(edges,columns=['parent_root','child_root','source']).drop_duplicates(['parent_root','child_root'])
    # Complement reciprocal hierarchy information without changing input tables.
    if len(ed):
        pa=ed.groupby('child_root').parent_root.nunique();ch=ed.groupby('parent_root').child_root.nunique()
        for i,row in profile.iterrows():
            if row.root_accession in pa:profile.loc[i,'parent_count']=max(pa[row.root_accession],row.parent_count if pd.notna(row.parent_count) else 0)
            if row.root_accession in ch:profile.loc[i,'child_count']=max(ch[row.root_accession],row.child_count if pd.notna(row.child_count) else 0)
    return profile,ed,pd.DataFrame(lineage),pd.DataFrame(conflicts)


def profile_bins(vals):
    s=pd.to_numeric(pd.Series(vals),errors='coerce');good=s.notna()&np.isfinite(s)&s.ge(0)&s.eq(np.floor(s))
    x=s[good]
    if len(x) and x.max()>100:
        edges=[-.5,.5,9.5,99.5,999.5,9999.5,99999.5,np.inf];labels=['0','1-9','10-99','100-999','1k-9k','10k-99k','100k+']
    else:edges=[-.5,.5,1.5,2.5,4.5,9.5,24.5,np.inf];labels=['0','1','2','3-4','5-9','10-24','25+']
    counts,_=np.histogram(x,edges)
    return {'labels':labels,'counts':counts.tolist(),'n':int(good.sum()),'missing':int(s.isna().sum()),
            'invalid':int((s.notna()&~good).sum()),'max':float(x.max()) if len(x) else None,
            'median':float(x.median()) if len(x) else None,'zero':int(x.eq(0).sum())}

# These helpers read saved metadata only. No source file or checkpoint is changed.
VP_PATTERN = re.compile(r'(?<![A-Za-z0-9])(phs\d+)\.v(\d+)\.p(\d+)(?!\d)', re.I)


def version_pairs(value_, root, allow_numeric_pairs=False):
    """Parse actual co-observed (v,p) pairs; never form a Cartesian product."""
    found=set()
    obj=decoded(value_)
    if obj is None:obj=value_
    def walk(obj, allow_pairs=False):
        if isinstance(obj,dict):
            acc=text(obj.get('accession',obj.get('root_accession',root)))
            if rootid(acc)==root:
                v=_nonnegative_int(obj.get('version',obj.get('v')))
                p=_nonnegative_int(obj.get('participant_set',obj.get('p')))
                if pd.notna(v) and pd.notna(p) and v>0 and p>0:found.add((int(v),int(p)))
            for item in obj.values():walk(item,False)
        elif isinstance(obj,(list,tuple,np.ndarray)):
            if allow_pairs and len(obj)==2 and all(not isinstance(x,(list,dict,tuple,np.ndarray)) for x in obj):
                v,p=map(_nonnegative_int,obj)
                if pd.notna(v) and pd.notna(p) and v>0 and p>0:found.add((int(v),int(p)))
            for item in obj:walk(item,allow_pairs)
        else:
            for match in VP_PATTERN.finditer(text(obj)):
                if match.group(1).lower()==root and int(match.group(2))>0 and int(match.group(3))>0:
                    found.add((int(match.group(2)),int(match.group(3))))
    walk(obj,allow_numeric_pairs)
    return found


def collect_version_evidence(roots,tables):
    """Own-root identifiers only; child references never become the parent's version."""
    rows=[];universe=set(roots.root_accession)
    def record(r,origin,field,v,allow_pairs=False):
        for version,participant in sorted(version_pairs(v,r,allow_pairs)):
            rows.append({'root_accession':r,'data_version':version,'participant_set':participant,
                         'source':origin,'field':field,'accession':f'{r}.v{version}.p{participant}'})
    for row in roots.to_dict('records'):
        r=row['root_accession']
        for field in ['latest_accession','accession','queried_accession','accession_sstr']:
            record(r,'roots',field,row.get(field))
        record(r,'roots','versions',row.get('versions'),True)
        if pd.notna(_nonnegative_int(row.get('latest_v'))) and pd.notna(_nonnegative_int(row.get('latest_p'))):
            record(r,'roots','latest_v/latest_p',{'accession':r,'version':row['latest_v'],'participant_set':row['latest_p']})
    for key in ['versions_raw','sstr','auth','ftp']:
        for row in tables.get(key,pd.DataFrame()).to_dict('records'):
            r=row.get('root_accession')
            if r not in universe:continue
            if key=='sstr' and status(row.get('sstr_fetch_note'))!='ok':continue
            if key=='auth' and status(row.get('authorized_access_fetch_note'))!='ok':continue
            fields={'versions_raw':['raw_accession','identifiers_raw'],
                    'sstr':['accession','accession_sstr'],
                    'auth':['latest_accession_used'],
                    'ftp':['version_folders_seen','selected_version_folder']}[key]
            for field in fields:record(r,key,field,row.get(field))
            if key=='sstr':
                study,_=_study_payload(row)
                record(r,key,'study.accver',study.get('accver'))
                record(r,key,'sstr_version/sstr_participant_set',{'accession':r,'version':row.get('sstr_version'),'participant_set':row.get('sstr_participant_set')})
    columns=['root_accession','data_version','participant_set','source','field','accession']
    return pd.DataFrame(rows,columns=columns).drop_duplicates().sort_values(columns[:3]+['source','field']).reset_index(drop=True)


def version_summary(evidence,roots):
    pairs=evidence.drop_duplicates(['root_accession','data_version','participant_set'])
    highest=(pairs.sort_values(['data_version','participant_set']).drop_duplicates('root_accession',keep='last'))
    pair_rows=highest[['root_accession','data_version','participant_set','accession']].copy()
    counts=highest.groupby(['data_version','participant_set']).size().reset_index(name='accessions')
    all_counts=pairs.groupby(['data_version','participant_set']).size().reset_index(name='accessions')
    first=highest.data_version.eq(1)&highest.participant_set.eq(1)
    byroot=pairs.groupby('root_accession')
    retained=byroot.data_version.nunique();higher=highest.set_index('root_accession').data_version.gt(1)
    singleton=(retained.eq(1)&higher.reindex(retained.index).fillna(False))
    sent=evidence[evidence.root_accession.eq('phs001974')]
    result={'unit':'One highest numerically observed (v,p) pair per root',
        'pair_rule':'max(v,p) among co-observed pairs; not independent maxima and not a complete release history',
        'known_roots':len(highest),'unknown_roots':len(roots)-len(highest),
        'v1_p1':int(first.sum()),'other_pairs':int((~first).sum()),
        'v1_p1_percent':100*float(first.mean()) if len(first) else None,
        'max_v':int(highest.data_version.max()) if len(highest) else None,
        'max_p':int(highest.participant_set.max()) if len(highest) else None,
        'one_retained_v_label_but_suffix_gt1':int(singleton.sum()),
        'retained_root_vp_pairs':len(pairs),'matrix':counts.to_dict('records'),
        'all_observed_matrix':all_counts.to_dict('records'),
        'source_roots':evidence.groupby('source').root_accession.nunique().to_dict(),
        'sentinel':{'root':'phs001974','saved_pairs':sent[['accession','data_version','participant_set']].drop_duplicates().to_dict('records'),
            'v10_p1_seen':bool(((sent.data_version==10)&(sent.participant_set==1)).any()),
            'note':'This diagnostic inspects saved inputs; no value is inserted from the external example.'}}
    return result,pair_rows,pairs


def _consent_label(group):
    """Numeric consent codes have no global mapping. Preserve modifiers in labels."""
    short=text(group.get('short_name')) or text(group.get('consent_abbreviation')) or text(group.get('abbreviation'))
    full=text(group.get('name')) or text(group.get('consent_name')) or text(group.get('title'))
    if short:return short,full,'explicit abbreviation'
    if full:
        matches=re.findall(r'\(([A-Z][A-Z0-9_\- ,]+)\)',full)
        if matches:return matches[-1].strip(),full,'abbreviation in explicit name'
        return full,full,'explicit name'
    return '',full,'unmapped numeric code'


def collect_consent_labels(roots,tables):
    ss_by=_root_rows(tables.get('sstr',pd.DataFrame()),roots)
    aa_by=_root_rows(tables.get('auth',pd.DataFrame()),roots)
    rows=[];coverage=[]
    for rootrow in roots.to_dict('records'):
        r=rootrow['root_accession'];ss=ss_by.get(r,rootrow);aa=aa_by.get(r,{})
        groups=None;origin=None;ac=text(ss.get('accession')) or text(ss.get('accession_sstr')) or text(rootrow.get('latest_accession'))
        if status(ss.get('sstr_fetch_note'))=='ok':
            study,_=_study_payload(ss)
            groups=study.get('consent_groups')
            if not isinstance(groups,list):groups=_list_value(ss.get('consent_groups'))
            if isinstance(groups,list):origin='SSTR'
        if groups is None and status(aa.get('authorized_access_fetch_note'))=='ok':
            names=_list_value(aa.get('consent_group_names'))
            identifiers=_list_value(aa.get('consent_group_accessions'))
            if names is not None:
                groups=[];origin='Authorized Access names';ac=text(aa.get('latest_accession_used'))
                for i,name in enumerate(names):
                    code=None
                    if identifiers is not None and len(identifiers)==len(names):
                        match=re.search(r'\.c(\d+)(?!\d)',text(identifiers[i]))
                        if match:code=match.group(1)
                    groups.append({'name':text(name),'code':code})
        good=[];bad_codes=[]
        for group in groups or []:
            if not isinstance(group,dict):continue
            label,full,basis=_consent_label(group);code=group.get('code',group.get('consent_code'))
            numeric=_nonnegative_int(code);code=str(int(numeric)) if pd.notna(numeric) else text(code)
            if not label:
                bad_codes.append(code)
            else:good.append(label)
            rows.append({'root_accession':r,'source_accession':ac,'consent_code':code or None,
                         'label':label or None,'full_name':full or None,'label_source':basis,'source':origin,
                         'mapping_status':'labelled' if label else 'unmapped'})
        coverage.append({'root_accession':r,'metadata_available':groups is not None,
                         'labelled':bool(good),'n_labels':len(set(good)),
                         'unmapped_groups':len(bad_codes),'empty_groups':groups==[] if groups is not None else False})
    f=pd.DataFrame(rows,columns=['root_accession','source_accession','consent_code','label','full_name','label_source','source','mapping_status'])
    cover=pd.DataFrame(coverage)
    labelled=f.dropna(subset=['label']).drop_duplicates(['root_accession','label'])
    freq=labelled.groupby('label').root_accession.nunique().reset_index(name='accessions')
    freq=freq.sort_values(['accessions','label'],ascending=[False,True]).reset_index(drop=True)
    denominator=int(cover.labelled.sum())
    freq['percentage_labelled_roots']=100*freq.accessions/denominator if denominator else np.nan
    freq['percentage_all_roots']=100*freq.accessions/len(roots) if len(roots) else np.nan
    top=freq.head(7).to_dict('records')
    remainder=set(freq.label.iloc[7:])
    if remainder:
        other=labelled[labelled.label.isin(remainder)].root_accession.nunique()
        top.append({'label':f'Other labels ({len(remainder)})','accessions':int(other),
                    'percentage_labelled_roots':100*other/denominator,'percentage_all_roots':100*other/len(roots)})
    result={'rows':freq.to_dict('records'),'display_rows':top,'labelled_roots':denominator,
            'known_metadata_roots':int(cover.metadata_available.sum()),'unlabelled_roots':len(roots)-denominator,
            'roots_with_unmapped_codes':int(cover.unmapped_groups.gt(0).sum()),
            'empty_group_roots':int(cover.empty_groups.sum()),
            'note':'One accession can carry several named consents; bars and percentages can overlap. Numeric codes are study-local, not universal categories.'}
    return result,f,cover


def add_profile_details(profile,roots,tables):
    """Add samples and actual numeric version coordinates to the B-owned working view."""
    profile=profile.copy();ss_by=_root_rows(tables.get('sstr',pd.DataFrame()),roots)
    root_by=roots.set_index('root_accession').to_dict('index')
    sample_sources=[]
    for idx,row in profile.iterrows():
        r=row.root_accession;rootrow=root_by[r];ss=ss_by.get(r,rootrow)
        study,stats=_study_payload(ss)
        samples=np.nan;origin=None
        # Do not substitute assay/run totals or sum cross-consent/cross-version counts.
        if status(ss.get('sstr_fetch_note'))=='ok':
            candidates=[('sstr.n_samples_total',ss.get('n_samples_total')),
                        ('sstr.study_stats.cnt_samples.loaded',(stats.get('cnt_samples',{}) or {}).get('loaded'))]
            if ss is rootrow:candidates.append(('roots.n_samples_total',rootrow.get('n_samples_total')))
            for name,val in candidates:
                v=_nonnegative_int(val)
                if pd.notna(v):samples=v;origin=name;break
        profile.loc[idx,'samples']=samples
        if origin:sample_sources.append({'root_accession':r,'metric':'samples','source':origin})
    evidence=collect_version_evidence(roots,tables)
    audit,latest,pairs=version_summary(evidence,roots)
    vm=latest.set_index('root_accession');retained=pairs.groupby('root_accession')
    profile['max_data_version']=profile.root_accession.map(vm.data_version)
    profile['participant_set_for_max_version']=profile.root_accession.map(vm.participant_set)
    profile['highest_observed_accession']=profile.root_accession.map(vm.accession)
    # Keep old count columns in the analysis table for compatibility, not as suffix values.
    profile['data_versions']=profile.root_accession.map(retained.data_version.nunique())
    profile['participant_versions']=profile.root_accession.map(retained.participant_set.nunique())
    for field,label in [('v_labels','data_version'),('p_labels','participant_set')]:
        vals=retained[label].apply(lambda x:json.dumps(sorted(set(int(i) for i in x))))
        profile[field]=profile.root_accession.map(vals)
    consents,consent_rows,consent_coverage=collect_consent_labels(roots,tables)
    return profile,audit,evidence,latest,pairs,consents,consent_rows,consent_coverage,pd.DataFrame(sample_sources)


PROFILE,PROFILE_EDGES,PROFILE_LINEAGE,PROFILE_CONFLICTS=make_profile(R,TABLES,S)
(PROFILE,VERSION_AUDIT,VERSION_EVIDENCE,LATEST_PAIRS,ALL_VERSION_PAIRS,
 CONSENT_SUMMARY,CONSENT_ROWS,CONSENT_COVERAGE,SAMPLE_LINEAGE)=add_profile_details(PROFILE,R,TABLES)
PROFILE_LINEAGE=pd.concat([PROFILE_LINEAGE,SAMPLE_LINEAGE],ignore_index=True)
check('Profile keeps exactly one row per root',len(PROFILE)==N and PROFILE.root_accession.is_unique)
check('Version matrix conserves known roots',sum(x['accessions'] for x in VERSION_AUDIT['matrix'])==VERSION_AUDIT['known_roots'])
if not REUSE:
    hist={key:{**profile_bins(PROFILE[key]),'title':label,'definition':definition} for key,label,definition in PROFILE_SPECS}
    for key,data in hist.items():
        check('Histogram reconciles: '+key,sum(data['counts'])==data['n'])
        if not data['n']:gap('3',key+': no explicit usable saved field; no proxy substituted.')
    parents=PROFILE.parent_count;children=PROFILE.child_count;collections=PROFILE.is_collection.map(tri).astype('boolean')
    hierarchy={'parents':int(children.gt(0).sum()),'parent_known':int(children.notna().sum()),
               'children':int(parents.gt(0).sum()),'child_known':int(parents.notna().sum()),
               'pairs':len(PROFILE_EDGES),'pairs_both_in_universe':int((PROFILE_EDGES.parent_root.isin(rootset)&PROFILE_EDGES.child_root.isin(rootset)).sum()),
               'collections':int(collections.fillna(False).sum()) if collections.notna().any() else None,
               'collection_known':int(collections.notna().sum()),'unknown_collections':int(collections.isna().sum())}
    RESULTS['profile']={'histograms':hist,'hierarchy':hierarchy,'versions':VERSION_AUDIT,'consents':CONSENT_SUMMARY,
        'definitions':[{'key':k,'label':l,'definition':d} for k,l,d in PROFILE_SPECS],
        'conflicts':len(PROFILE_CONFLICTS),
        'notes':'Version matrix uses actual suffix values in a co-observed pair, one highest numeric pair per root. Named consent categories preserve modifiers and use local mappings. Subjects/samples may overlap across roots.'}
    for name,frame in [('accession_profile',PROFILE),('accession_profile_lineage',PROFILE_LINEAGE),
         ('accession_hierarchy_edges',PROFILE_EDGES),('accession_profile_conflicts',PROFILE_CONFLICTS),
         ('version_pair_evidence',VERSION_EVIDENCE),('highest_observed_version_pairs',LATEST_PAIRS),
         ('all_observed_version_pairs',ALL_VERSION_PAIRS),('consent_code_label_lookup',CONSENT_ROWS),
         ('consent_label_coverage',CONSENT_COVERAGE),('consent_label_frequencies',pd.DataFrame(CONSENT_SUMMARY['rows']))]:export_table(name,frame)
    export_table('version_participant_matrix',pd.DataFrame(VERSION_AUDIT['matrix'],columns=['data_version','participant_set','accessions']))
    write_json(OUT/'version_audit.json',VERSION_AUDIT);OUTPUTS['version_audit.json']=hashfile(OUT/'version_audit.json')
    display(pd.DataFrame([{'field':k,'known':v['n'],'missing':v['missing'],'median':v['median'],'max':v['max']} for k,v in hist.items()]))
    print('Version diagnostic:',{k:VERSION_AUDIT[k] for k in ['known_roots','max_v','max_p','v1_p1','one_retained_v_label_but_suffix_gt1']})
    print('phs001974 saved-field check:',VERSION_AUDIT['sentinel'])
    print('Named consent categories:',CONSENT_SUMMARY['labelled_roots'],'roots; unmapped codes:',CONSENT_SUMMARY['roots_with_unmapped_codes'])


## New slide 4: semantic profiles and data modalities

Native MeSH recovery, most-specific supported descriptor profiles, labelled hierarchy bridges, and SSTR/SRA assay vocabulary harmonization. RCDC is a conceptual reference, not an implemented or validated classification.


In [ ]:
# @title Semantic profiles and assay harmonization -- local, evidence-labelled analysis
# MeSH trees and OBI come only from A's pinned, content-hashed vocabulary snapshots.
# No term is inherited from a linked trial, parent accession or manuscript.
from collections import defaultdict

class MeshProfileIndex:
    def __init__(self,records):
        self.records=records;self.labels=defaultdict(set);self.strict=defaultdict(set);self.entries=defaultdict(set)
        self.tree_owner=defaultdict(set);self._closure={}
        for ident,r in records.items():
            self.labels[canonical_label(r['label'])].add(ident)
            for term in r.get('terms',[]):
                self.entries[canonical_label(term['label'])].add(ident)
                if term.get('preferred_concept'):self.strict[canonical_label(term['label'])].add(ident)
            for tree in r.get('tree_numbers',[]):self.tree_owner[tree].add(ident)
    def resolve(self,source_id,label):
        ident=text(source_id).upper();norm=canonical_label(label)
        if ident:
            if ident not in self.records:return None,'identifier_not_in_pinned_mesh'
            allowed=self.labels.get(norm,set())|self.entries.get(norm,set())
            return ident,'native_identifier' if not norm or ident in allowed else 'native_identifier_label_changed_or_conflicting'
        for lookup,status in [(self.labels,'exact_descriptor_label'),(self.strict,'strict_synonym_in_preferred_concept')]:
            ids=lookup.get(norm,set())
            if len(ids)==1:return next(iter(ids)),status
            if len(ids)>1:return None,'ambiguous_exact_label'
        if self.entries.get(norm):return None,'nonpreferred_entry_requires_review'
        return None,'label_unresolved'
    def ancestors(self,ident):
        if ident in self._closure:return self._closure[ident]
        result={ident:0} if ident in self.records else {}
        for t in self.records.get(ident,{}).get('tree_numbers',[]):
            parts=t.split('.')
            for k in range(1,len(parts)):
                prefix='.'.join(parts[:k]);hops=len(parts)-k
                for ancestor in self.tree_owner.get(prefix,[]):
                    result[ancestor]=min(hops,result.get(ancestor,10**9))
        self._closure[ident]=result;return result
    def paths_to(self,child,parent):
        paths=[]
        for tree in self.records.get(child,{}).get('tree_numbers',[]):
            for prefix in self.records.get(parent,{}).get('tree_numbers',[]):
                if tree==prefix or tree.startswith(prefix+'.'):
                    parts=tree.split('.');k=len(prefix.split('.'))
                    paths.append(['.'.join(parts[:j]) for j in range(len(parts),k-1,-1)])
        return sorted(paths)
    def narrower(self,child,parent):return child!=parent and parent in self.ancestors(child)
    def most_specific(self,ids):
        ids=set(ids)&set(self.records)
        return sorted(i for i in ids if not any(self.narrower(j,i) for j in ids if i!=j))
    def depth(self,ident):
        return max([len(t.split('.')) for t in self.records.get(ident,{}).get('tree_numbers',[])]+[0])
    def branch(self,ident):
        trees=self.records.get(ident,{}).get('tree_numbers',[])
        groups=set()
        for t in trees:
            groups.add('Disease / mental disorder' if t.startswith('C') or t.startswith('F03') else
                'Method / equipment' if t.startswith('E') else 'Biological process / trait' if t.startswith(('G','F')) else 'Other MeSH domain')
        return sorted(groups or {'No tree position'})
    def label(self,ident):return self.records.get(ident,{}).get('label',ident)
    def bridge(self,left,right,max_hops=6,min_depth=1):
        """Most-specific supported shared ancestors; direct agreement remains a separate result."""
        left=set(left)&set(self.records);right=set(right)&set(self.records)
        if not left or not right:return []
        candidates={}
        for a in left:
            ac=self.ancestors(a)
            for b in right:
                bc=self.ancestors(b)
                for c in ac.keys()&bc.keys():
                    hops=ac[c]+bc[c]
                    if ac[c]>max_hops or bc[c]>max_hops or self.depth(c)<min_depth:continue
                    exact=c==a==b
                    # Hierarchy fallback is restricted to disease/mental-disorder paths.
                    # Exact non-disease agreement remains explicit; generic ancestors do not bridge.
                    if not exact and not any(t.startswith('C') or t.startswith('F03') for t in self.records[c].get('tree_numbers',[])):continue
                    if self.label(c).casefold() in {'humans','disease','diseases','pathologic processes'}:continue
                    row={'bridge_mesh_id':c,'bridge_label':self.label(c),'dataset_mesh_id':a,'trial_mesh_id':b,
                         'dataset_hops':ac[c],'trial_hops':bc[c],'tree_depth':self.depth(c),
                         'dataset_tree_paths':self.paths_to(a,c),'trial_tree_paths':self.paths_to(b,c),
                         'match_type':'exact' if exact else 'broader_narrower' if c in (a,b) else 'common_ancestor',
                         'broad_fallback':self.depth(c)<3 or self.label(c)=='Neoplasms'}
                    rank=(hops,0 if exact else 1,a,b)
                    if c not in candidates or rank<candidates[c][0]:candidates[c]=(rank,row)
        # An antichain retains independently supported specific topics instead of counting their parents again.
        selected=self.most_specific(candidates)
        return [candidates[i][1] for i in selected]


def sem_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()] if Path(path).exists() else []

def sem_output(name,rows,columns=None):
    frame=pd.DataFrame(rows)
    if frame.empty and columns:frame=pd.DataFrame(columns=columns)
    # Nested objects remain JSON, not Python repr, in CSV cells.
    for k in frame.columns:
        frame[k]=frame[k].map(lambda x:json.dumps(safe(x),ensure_ascii=False,sort_keys=True) if isinstance(x,(list,dict,tuple)) else x)
    export_table(name,frame)


def sem_candidate_id(entity_type,entity_id,ident,source_hash,field,quote):
    return hashlib.sha256(json.dumps([entity_type,entity_id,ident,source_hash,field,quote],ensure_ascii=False).encode()).hexdigest()[:24]


def semantic_text_candidates(texts,index):
    """Dictionary matches create review candidates, never native or accepted annotations."""
    lookup=defaultdict(set)
    for ident,r in index.records.items():
        for label in [r['label']]+[t['label'] for t in r.get('terms',[])]:
            norm=canonical_label(label)
            # Require useful multi-character words, not single-letter/gene-code coincidences.
            if len(norm)>=5 and len(norm.split())<=12:lookup[norm].add(ident)
    byfirst=defaultdict(list)
    for phrase in lookup:byfirst[phrase.split()[0]].append(phrase)
    out=[];seen=set()
    for r in texts:
        raw=text(r.get('text'));norm=canonical_label(raw)
        if not norm:continue
        # Original offsets are not assumed after Unicode/space normalization; retain source text and exact matched phrase.
        tokens=set(re.findall(r'[\w-]+',norm));matches=[]
        for token in tokens:
            for phrase in byfirst.get(token,[]):
                for m in re.finditer(r'(?<!\w)'+re.escape(phrase)+r'(?!\w)',norm):
                    matches.append((m.start(),m.end(),phrase))
        selected=[]
        for start,end,phrase in sorted(matches,key=lambda x:(-(x[1]-x[0]),x[0])):
            if any(start>=a and end<=b for a,b,_ in selected):continue
            selected.append((start,end,phrase))
        for start,end,phrase in selected:
            for ident in sorted(lookup[phrase]):
                cid=sem_candidate_id(r['entity_type'],r['entity_id'],ident,r.get('source_sha256'),r['source_field'],phrase)
                if cid in seen:continue
                seen.add(cid);context=norm[max(0,start-75):min(len(norm),end+75)]
                flag=bool(re.search(r'\b(no|not|without|exclude|excluding|family history)\b',norm[max(0,start-35):start]))
                out.append({'candidate_id':cid,'entity_type':r['entity_type'],'entity_id':r['entity_id'],
                    'mesh_id':ident,'mesh_label':index.label(ident),'matched_phrase':phrase,
                    'source_field':r['source_field'],'source':r.get('source'),'source_sha256':r.get('source_sha256'),
                    'source_text':raw,'normalized_context':context,'negation_or_scope_warning':flag,
                    'ambiguous_entry':len(lookup[phrase])>1,'decision':'pending','reviewer':'',
                    'mapping_scope':'own source text; not a native MeSH annotation'})
    return out

# Transparent analysis crosswalk. Targets are checked against the retained OBI ontology.
# Broad display groups are local analytical categories, not asserted ontology equivalences.
ASSAY_RULES=[
 {'aliases':['WGS','whole genome sequencing','whole-genome sequencing'], 'label':'Whole-genome sequencing','family':'DNA sequencing','sra':'WGS','analyte':'DNA','resolution':'not specified','product':'sequence assay','obi':['whole genome sequencing assay','whole genome sequencing']},
 {'aliases':['WXS','WES','exome sequencing','whole exome sequencing'], 'label':'Exome sequencing','family':'DNA sequencing','sra':'WXS','analyte':'DNA','resolution':'not specified','product':'sequence assay','obi':['exome sequencing assay','whole exome sequencing assay']},
 {'aliases':['DNAseq','DNA-seq','DNA seq','DNA sequencing'], 'label':'DNA sequencing, unspecified','family':'DNA sequencing','sra':None,'analyte':'DNA','resolution':'not specified','product':'sequence assay','obi':['DNA sequencing assay']},
 {'aliases':['RNA-Seq','RNAseq','RNA seq','RNA sequencing'], 'label':'RNA sequencing','family':'RNA sequencing','sra':'RNA-Seq','analyte':'RNA','resolution':'not specified','product':'sequence assay','obi':['RNA-seq assay','RNA sequencing assay']},
 {'aliases':['scRNA-seq','single cell RNA sequencing','single-cell RNA-seq'], 'label':'Single-cell RNA sequencing','family':'RNA sequencing','sra':'RNA-Seq','analyte':'RNA','resolution':'single cell','product':'sequence assay','obi':['single cell RNA sequencing assay','single-cell RNA-seq assay','single cell RNA-seq assay']},
 {'aliases':['snRNA-seq','single nucleus RNA sequencing','single-nucleus RNA-seq'], 'label':'Single-nucleus RNA sequencing','family':'RNA sequencing','sra':'RNA-Seq','analyte':'RNA','resolution':'single nucleus','product':'sequence assay','obi':['single nucleus RNA sequencing assay']},
 {'aliases':['miRNA-Seq','microRNA sequencing'], 'label':'MicroRNA sequencing','family':'RNA sequencing','sra':'miRNA-Seq','analyte':'RNA','resolution':'not specified','product':'sequence assay','obi':['microRNA profiling by high throughput sequencing assay','microRNA sequencing assay']},
 {'aliases':['ssRNA-seq','strand-specific RNA sequencing'], 'label':'Strand-specific RNA sequencing','family':'RNA sequencing','sra':'ssRNA-seq','analyte':'RNA','resolution':'not specified','product':'sequence assay','obi':['RNA-seq assay']},
 {'aliases':['ncRNA-Seq'], 'label':'Noncoding RNA sequencing','family':'RNA sequencing','sra':'ncRNA-Seq','analyte':'RNA','resolution':'not specified','product':'sequence assay','obi':['RNA-seq assay']},
 {'aliases':['ATAC-seq','ATACseq'], 'label':'ATAC-seq','family':'Chromatin / epigenomic sequencing','sra':'ATAC-seq','analyte':'chromatin-associated DNA','resolution':'not specified','product':'sequence assay','obi':['assay for transposase-accessible chromatin using sequencing','ATAC-seq assay']},
 {'aliases':['ChIP-Seq','ChIPseq'], 'label':'ChIP-seq','family':'Chromatin / epigenomic sequencing','sra':'ChIP-Seq','analyte':'chromatin-associated DNA','resolution':'not specified','product':'sequence assay','obi':['chromatin immunoprecipitation sequencing assay','ChIP-seq assay']},
 {'aliases':['Bisulfite-Seq','WGBS','bisulfite sequencing'], 'label':'Bisulfite sequencing','family':'Chromatin / epigenomic sequencing','sra':'Bisulfite-Seq','analyte':'DNA','resolution':'not specified','product':'sequence assay','obi':['bisulfite sequencing assay','whole genome bisulfite sequencing assay']},
 {'aliases':['Hi-C','HiC'], 'label':'Hi-C','family':'Chromatin / epigenomic sequencing','sra':'Hi-C','analyte':'DNA','resolution':'not specified','product':'sequence assay','obi':['Hi-C assay','Hi-C']},
 {'aliases':['Targeted-Capture','targeted capture'], 'label':'Targeted capture, target unspecified','family':'Other targeted sequencing','sra':'Targeted-Capture','analyte':'not specified','resolution':'not specified','product':'sequence assay','obi':[]},
 {'aliases':['AMPLICON','amplicon sequencing'], 'label':'Amplicon sequencing','family':'Other targeted sequencing','sra':'AMPLICON','analyte':'not specified','resolution':'not specified','product':'sequence assay','obi':['amplicon sequencing assay']},
 {'aliases':['WGA','whole genome amplification'], 'label':'Whole-genome amplification','family':'DNA amplification / preparation','sra':'WGA','analyte':'DNA','resolution':'not specified','product':'preparation / reported strategy','obi':[]},
 {'aliases':['Seq_RNA_Expression'], 'label':'RNA expression data, sequencing-derived','family':'RNA expression-derived data','sra':None,'analyte':'RNA','resolution':'not specified','product':'derived data','obi':[]},
 {'aliases':['Seq_DNA_SNP_CNV','Seq_DNA_SNP_MAF_Ind','Seq_DNA_SV','Seq_DNA_SNP'], 'label':'DNA variant data, sequencing-derived','family':'DNA variant-derived data','sra':None,'analyte':'DNA','resolution':'not specified','product':'derived data','obi':[]},
 {'aliases':['Array_SNP','Array_SNP_CNV','SNP array','genotyping array'], 'label':'Genotyping array','family':'Arrays','sra':None,'analyte':'DNA','resolution':'not specified','product':'array assay / data','obi':['SNP microarray assay','SNP genotyping by array assay']},
 {'aliases':['Array_DNA_Methylation','DNA methylation array'], 'label':'DNA methylation array','family':'Arrays','sra':None,'analyte':'DNA','resolution':'not specified','product':'array assay / data','obi':['DNA methylation profiling by array assay']},
 {'aliases':['Array_RNA_Expression','gene expression array'], 'label':'Gene-expression array','family':'Arrays','sra':None,'analyte':'RNA','resolution':'not specified','product':'array assay / data','obi':['transcription profiling by array assay']},
 {'aliases':['Imputation_SNP','imputed genotypes'], 'label':'Imputed genotype data','family':'Imputed genotype data','sra':None,'analyte':'DNA-derived genotypes','resolution':'not specified','product':'derived data','obi':[]},
 {'aliases':['proteomics','mass spectrometry','LC-MS/MS'], 'label':'Proteomic / mass-spectrometry data','family':'Other molecular data','sra':None,'analyte':'protein or mass-spectrometry analyte','resolution':'not specified','product':'reported assay / data','obi':[]},
 {'aliases':['spatial transcriptomics'], 'label':'Spatial transcriptomics','family':'Spatial molecular profiling','sra':None,'analyte':'RNA','resolution':'spatial','product':'reported assay / data','obi':['spatial transcriptomics assay']},
]

def normalize_assay(label,obi_records):
    norm=canonical_label(label)
    rule=next((r for r in ASSAY_RULES if norm in {canonical_label(a) for a in r['aliases']}),None)
    if rule is None:return {'harmonized_label':text(label),'family':'Unmapped / unspecified','mapping_status':'unmapped','sra_strategy':None,
        'analyte':None,'resolution':None,'product':None,'obi_uri':None,'obi_label':None,'obi_relation':'not_mapped'}
    labels=defaultdict(set)
    for uri,t in obi_records.items():
        for s in [t['label']]+t.get('exact_synonyms',[]):labels[canonical_label(s)].add(uri)
    target=None
    for label_ in rule['obi']:
        matches=labels.get(canonical_label(label_),set())
        if len(matches)==1:target=next(iter(matches));break
    return {'harmonized_label':rule['label'],'family':rule['family'],'mapping_status':'explicit_local_crosswalk',
        'sra_strategy':rule['sra'],'analyte':rule['analyte'],'resolution':rule['resolution'],'product':rule['product'],
        'obi_uri':target,'obi_label':obi_records[target]['label'] if target else None,
        'obi_relation':'assay_class_context_not_data_equivalence' if target else 'target_not_resolved_in_snapshot'}


def current_assay_rows(rows,roots):
    current={r['root_accession']:text(r.get('latest_accession')) for r in roots};selected=[];excluded=[]
    for r in rows:
        root=r.get('root_accession');host=r.get('reported_under_root')
        if root not in current:excluded.append({**r,'exclusion':'owner_outside_root_universe'});continue
        accession=text(r.get('source_accession'))
        expected=current.get(host,'')
        if expected and '.v' in expected and accession and accession!=expected:
            excluded.append({**r,'exclusion':'noncurrent_host_version'});continue
        ver=r.get('data_version');part=r.get('participant_set')
        own_expected=current.get(root,'')
        m=re.search(r'\.v(\d+)\.p(\d+)',own_expected)
        if m and ver is not None and part is not None and (int(ver),int(part))!=(int(m.group(1)),int(m.group(2))):
            excluded.append({**r,'exclusion':'noncurrent_owned_version'});continue
        selected.append({**r,'version_scope_status':'explicit_version_observed' if re.search(r'\.v\d+\.p\d+',accession) else 'version_not_reported'})
    return selected,excluded



def semantic_recorded_inventory(native,root_records):
    """Source labels/IDs only. This inventory does not validate or expand MeSH."""
    allowed={r['root_accession'] for r in root_records};by_entity=defaultdict(dict);rows=[]
    def add(kind,entity,ident,label,source,field,evidence=None):
        if kind=='dataset' and entity not in allowed:return
        ident=text(ident).strip();label=text(label).strip()
        if not ident and not label:return
        # Do not label an unverified source ID as an ontology match.
        key=('id:'+ident.upper()) if ident else 'label:'+canonical_label(label)
        if key not in by_entity[(kind,entity)]:
            by_entity[(kind,entity)][key]={'term_key':key,'source_id':ident or None,'label':label or ident,'entity_type':kind,'entity_id':entity}
        rows.append({'entity_type':kind,'entity_id':entity,'term_key':key,'source_id':ident or None,
            'source_label':label,'source':source,'source_field':field,'evidence_id':evidence,
            'status':'source_recorded_not_vocabulary_validated'})
    for r in native:
        if r.get('role') not in {'dataset','condition'}:continue
        add(r.get('entity_type'),r.get('entity_id'),r.get('source_id'),r.get('source_label'),r.get('source'),r.get('source_field'),r.get('evidence_id'))
    def atoms(raw):
        val=decoded(raw)
        if val is None:val=raw
        if isinstance(val,dict):
            code=val.get('mesh_id',val.get('code',val.get('id')))
            label=val.get('label',val.get('display',val.get('name')))
            if code or label:yield code,label
            else:
                for k in ('coding','condition','mesh_terms','terms'):
                    if k in val:yield from atoms(val[k])
        elif isinstance(val,(list,tuple,np.ndarray)):
            for v in val:yield from atoms(v)
        elif isinstance(val,str) and text(val).lower() not in {'','[]','null','none','nan'}:
            if re.fullmatch(r'D(?:\d{6}|\d{9})',val,re.I):yield val,None
            else:yield None,val
    # Only dedicated MeSH fields, and only roots lacking native-sidecar annotations.
    # Never use trial terms, article keywords or a free-text disease as dataset evidence.
    for r in root_records:
        root=r['root_accession']
        if by_entity.get(('dataset',root)):continue
        for ident,label in atoms(r.get('mesh_terms')):
            add('dataset',root,ident,label,'canonical_root_table','mesh_terms')
    counts=Counter();labels={};ids={}
    for (kind,entity),terms in by_entity.items():
        if kind!='dataset':continue
        for key,t in terms.items():counts[key]+=1;labels.setdefault(key,t['label']);ids.setdefault(key,t['source_id'])
    termrows=[{'term_key':k,'source_id':ids[k],'label':labels[k],'roots':v,'percent_all_roots':100*v/len(allowed) if allowed else None}
        for k,v in sorted(counts.items(),key=lambda kv:(-kv[1],labels[kv[0]]))]
    return by_entity,termrows,rows

def analyze_semantics(root_records,nct_frame,sem_inputs,review_rows=None):
    native=sem_inputs.get('native',[]);coverage=sem_inputs.get('coverage',[])
    index=MeshProfileIndex(sem_inputs.get('mesh',{}));obi=sem_inputs.get('obi',{})
    rootset={r['root_accession'] for r in root_records};eligible=defaultdict(set);fieldonly=defaultdict(set);resolved=[]
    recorded,recorded_terms,recorded_rows=semantic_recorded_inventory(native,root_records)
    entities={('dataset',r) for r in rootset}
    for r in native:
        entity=(r['entity_type'],r['entity_id'])
        if entity[0]=='dataset' and entity[1] not in rootset:continue
        entities.add(entity);ident,state=index.resolve(r.get('source_id'),r.get('source_label'))
        if not index.records:state='vocabulary_unavailable_not_evaluated'
        # Preserve ancestor and intervention annotation lanes; never silently promote to native condition.
        allowed=r.get('role') in {'dataset','condition'} and ident is not None
        row={**r,'mesh_id':ident,'mesh_label':index.label(ident) if ident else None,'resolution':state,
            'eligible_native_profile':allowed,'tree_numbers':index.records.get(ident,{}).get('tree_numbers',[])}
        resolved.append(row)
        if allowed:
            eligible[entity].add(ident)
            if 'Diseases/Traits' not in r.get('source_field',''):fieldonly[entity].add(ident)
    # Existing B linkage table is the only universe of dataset-trial pairs.
    pairset={(r['root_accession'],text(r['nct_id']).upper()) for r in nct_frame.to_dict('records')
             if r.get('root_accession') in rootset and re.fullmatch(r'NCT\d{8}',text(r.get('nct_id')),re.I)} if 'nct_id' in nct_frame else set()
    entities.update(('trial',nct) for _,nct in pairset)
    candidates=semantic_text_candidates(sem_inputs.get('texts',[]),index)
    c_lookup={r['candidate_id']:r for r in candidates};accepted=defaultdict(set);review_audit=[]
    for review in review_rows or []:
        if text(review.get('decision')).lower()!='accepted':continue
        cid=text(review.get('candidate_id'));candidate=c_lookup.get(cid)
        valid=(candidate is not None and text(review.get('reviewer')) and text(review.get('mesh_id'))==candidate['mesh_id']
               and text(review.get('source_sha256'))==text(candidate.get('source_sha256')))
        if valid:
            accepted[(candidate['entity_type'],candidate['entity_id'])].add(candidate['mesh_id'])
        review_audit.append({**review,'accepted_into_reviewed_profile':bool(valid),'reason':'candidate/source/vocabulary/reviewer checked' if valid else 'unmatched candidate, hash, ID or reviewer'})
    profiles=[];profilelong=[];refinements=[];allclosure={};specific_sets={}
    for kind,ident in sorted(entities):
        nativeids=eligible[(kind,ident)];reviewedids=nativeids|accepted[(kind,ident)]
        asserted=reviewedids if SEMANTIC_INCLUDE_REVIEWED else nativeids
        specific=index.most_specific(asserted);specific_sets[(kind,ident)]=set(specific)
        closure={}
        for mid in specific:
            for anc,hops in index.ancestors(mid).items():closure[anc]=min(hops,closure.get(anc,10**9))
        allclosure[(kind,ident)]=closure
        for parent in sorted(asserted-set(specific)):
            children=[m for m in specific if index.narrower(m,parent)]
            refinements.append({'entity_type':kind,'entity_id':ident,'retained_broad_mesh_id':parent,'broad_label':index.label(parent),
                'supported_specific_mesh_ids':children,'specific_labels':[index.label(m) for m in children],
                'rule':'broad source annotation retained; only redundant display parent suppressed'})
        profiles.append({'entity_type':kind,'entity_id':ident,'native_mesh_ids':sorted(nativeids),'native_mesh_labels':[index.label(m) for m in sorted(nativeids)],
            'reviewed_additions':sorted(accepted[(kind,ident)]),'specific_mesh_ids':specific,'specific_mesh_labels':[index.label(m) for m in specific],
            'annotation_count':len(nativeids),'profile_state':'native_terms_resolved' if nativeids else 'reviewed_only' if asserted else 'unavailable_or_unresolved',
            'max_observed_depth':max([index.depth(m) for m in asserted]+[0]),
            'unresolved_terms':sum(r['entity_type']==kind and r['entity_id']==ident and r['mesh_id'] is None for r in resolved)})
    df=Counter(m for (kind,_),closure in allclosure.items() if kind=='dataset' for m in closure)
    weights={}
    for (kind,ident),closure in allclosure.items():
        vector={mid:(SEMANTIC_ANCESTOR_DECAY**dist)*(math.log((len(rootset)+1)/(df[mid]+1))+1) for mid,dist in closure.items()}
        weights[(kind,ident)]=vector
        for mid,w in sorted(vector.items()):profilelong.append({'entity_type':kind,'entity_id':ident,'mesh_id':mid,'mesh_label':index.label(mid),
            'hierarchy_distance':closure[mid],'direct_profile_term':mid in specific_sets[(kind,ident)],'weight':w,'domain':index.branch(mid),
            'weight_definition':'decay^distance * (log((dataset_roots+1)/(dataset_roots_with_term+1))+1); descriptive, not calibrated'})
    def cosine(a,b):
        den=math.sqrt(sum(x*x for x in a.values())*sum(x*x for x in b.values()))
        return sum(a[k]*b.get(k,0) for k in a)/den if den else None
    bridges=[];pairrows=[]
    evidence=defaultdict(set)
    for r in sem_inputs.get('pair_evidence',[]):evidence[(r['root_accession'],r['nct_id'])].add(r.get('relationship_basis','recorded_reference_unadjudicated'))
    for root,nct in sorted(pairset):
        ds=eligible[('dataset',root)];ts=eligible[('trial',nct)]
        left=ds|accepted[('dataset',root)] if SEMANTIC_INCLUDE_REVIEWED else ds
        right=ts|accepted[('trial',nct)] if SEMANTIC_INCLUDE_REVIEWED else ts
        found=index.bridge(left,right,SEMANTIC_MAX_ANCESTOR_HOPS,SEMANTIC_MIN_BRIDGE_DEPTH)
        status_=('dataset_and_trial_annotations_unavailable' if not left and not right else
            'dataset_annotations_unavailable' if not left else 'trial_condition_annotations_unavailable' if not right else
            'no_supported_shared_ancestor_within_limits' if not found else
            'exact' if all(r['match_type']=='exact' for r in found) else 'hierarchical_context')
        if not index.records:status_='not_evaluated_vocabulary_unavailable'
        elif not left or not right:
            status_='annotations_unresolved' if recorded.get(('dataset',root)) and recorded.get(('trial',nct)) else status_
        p={'root_accession':root,'nct_id':nct,'native_exact_shared':sorted(ds&ts),
            'field_only_exact_shared':sorted(fieldonly[('dataset',root)]&ts),
            'match_status':status_,'bridge_count':len(found),'bridge_labels':[r['bridge_label'] for r in found],
            'has_broad_fallback':any(r['broad_fallback'] for r in found),
            'native_jaccard':len(ds&ts)/len(ds|ts) if ds|ts else None,
            'profile_cosine':cosine(weights.get(('dataset',root),{}),weights.get(('trial',nct),{})),
            'relationship_basis':sorted(evidence.get((root,nct),{'recorded_reference_unadjudicated'})),
            'meaning':'semantic context along a recorded reference; not verified trial origin or reuse'}
        pairrows.append(p)
        for f in found:bridges.append({**p,**f,'pair_weight':1/len(found),
            'dataset_source_evidence':[r['evidence_id'] for r in resolved if r['entity_type']=='dataset' and r['entity_id']==root and r['mesh_id']==f['dataset_mesh_id']],
            'trial_source_evidence':[r['evidence_id'] for r in resolved if r['entity_type']=='trial' and r['entity_id']==nct and r['mesh_id']==f['trial_mesh_id']]})
    selected,excluded=current_assay_rows(sem_inputs.get('assays',[]),root_records)
    normalized=[];canonical={};_assay_mapping_cache={};assay_profiles={r:{'families':set(),'strategies':set(),'labels':set(),'raw_labels':set(),'unmapped':set(),'derived':set()} for r in rootset}
    for r in selected:
        _label_key=canonical_label(r['raw_label'])
        if _label_key not in _assay_mapping_cache:_assay_mapping_cache[_label_key]=normalize_assay(r['raw_label'],obi)
        mapping=_assay_mapping_cache[_label_key];root=r['root_accession']
        x={**r,**mapping,'sample_counts_additive':False,'harmonization_version':'explicit-assay-crosswalk-1'}
        # All details retained. Distinct observations keyed by actual owner and exact scope, not hosting parent.
        key=(root,r.get('data_version'),r.get('participant_set'),text(r.get('consent_code')),canonical_label(r['raw_label']),text(r.get('experiment_accession')))
        previous=canonical.get(key)
        if previous is not None and any(previous.get(k)!=x.get(k) for k in ('sample_count','subject_count','is_sequence')):x['conflicting_duplicate_counts']=True
        x['sequence_flag_disagreement']=(mapping['product']=='sequence assay' and r.get('is_sequence') is False)
        normalized.append(x)
        if previous is None or (r.get('scope')=='self' and previous.get('scope')!='self'):canonical[key]=x
        prof=assay_profiles[root];prof['raw_labels'].add(r['raw_label'])
        if mapping['mapping_status']=='unmapped':prof['unmapped'].add(r['raw_label'])
        else:
            prof['families'].add(mapping['family']);prof['labels'].add(mapping['harmonized_label'])
            if mapping['sra_strategy']:prof['strategies'].add(mapping['sra_strategy'])
            if mapping['product']=='derived data':prof['derived'].add(mapping['harmonized_label'])
    assay_roots=[]
    for root,profile in sorted(assay_profiles.items()):
        assay_roots.append({'root_accession':root,**{k:sorted(v) for k,v in profile.items()},
            'has_mapped_assay':bool(profile['families']),'has_assay_evidence':bool(profile['raw_labels']),
            'n_distinct_assay_labels':len(profile['labels']),
            'missingness':'not reported is not a negative assay result; parent aggregates do not imply this root has files'})
    assay_source_comparison=[]
    for root in sorted(rootset):
        evidence=[x for x in normalized if x['root_accession']==root]
        sr=[x for x in evidence if x.get('source_field')=='library_strategy']
        ss=[x for x in evidence if x.get('source_field')!='library_strategy']
        sf={x['family'] for x in ss if x['mapping_status']!='unmapped'}
        rf={x['family'] for x in sr if x['mapping_status']!='unmapped'}
        assay_source_comparison.append({'root_accession':root,'sstr_evidence_rows':len(ss),'sra_evidence_rows':len(sr),
            'sstr_families':sorted(sf),'sra_families':sorted(rf),'shared_families':sorted(sf&rf),
            'sstr_only_families':sorted(sf-rf),'sra_only_families':sorted(rf-sf),
            'comparison_status':'both_sources_observed' if sr and ss else 'one_or_both_sources_unobserved',
            'interpretation':'Source-only labels are review signals, not contradictions or absence of assays; coverage/version/consent may differ.'})
    family_counts=Counter(f for p in assay_roots for f in p['families']);unmapped=Counter(l for p in assay_roots for l in p['unmapped'])
    # Readability profile: disease headings two levels down, one root per category, multi-label.
    themes=defaultdict(set)
    for (kind,root),specific in specific_sets.items():
        if kind!='dataset':continue
        for mid in specific:
            for t in index.records[mid].get('tree_numbers',[]):
                if not (t.startswith('C') or t.startswith('F03')):continue
                parts=t.split('.');prefix='.'.join(parts[:SEMANTIC_THEME_DEPTH])
                owners=index.tree_owner.get(prefix,set()) or {mid}
                for cat in owners:themes[cat].add(root)
    theme_rows=[{'mesh_id':m,'label':index.label(m),'roots':len(rs),'percent_all_roots':100*len(rs)/len(rootset)} for m,rs in themes.items()]
    theme_rows.sort(key=lambda r:(-r['roots'],r['label']))
    cross=[]
    for theme in theme_rows[:10]:
        members=themes[theme['mesh_id']]
        for family,_ in family_counts.most_common(8):
            count=sum(root in members and family in assay_profiles[root]['families'] for root in rootset)
            cross.append({'mesh_id':theme['mesh_id'],'theme':theme['label'],'assay_family':family,'roots':count})
    specific_counts=Counter(m for (kind,_),ids in specific_sets.items() if kind=='dataset' for m in ids
        if any(t.startswith('C') or t.startswith('F03') for t in index.records[m].get('tree_numbers',[])))
    specific_rows=[{'mesh_id':m,'label':index.label(m),'roots':v,'percent_all_roots':100*v/len(rootset)}
        for m,v in sorted(specific_counts.items(),key=lambda kv:(-kv[1],index.label(kv[0])))]
    dataset_profiles=[p for p in profiles if p['entity_type']=='dataset']
    result={'ontology_status':sem_inputs.get('ontology_status',{}),'n_roots':len(rootset),
        'mesh_vocabulary_available':bool(index.records),'obi_vocabulary_available':bool(obi),
        'roots_with_recorded_mesh':sum(bool(v) for (kind,_),v in recorded.items() if kind=='dataset'),
        'recorded_descriptor_counts':recorded_terms,
        'descriptor_display_basis':'most_specific_resolved' if index.records else 'source_recorded_no_hierarchy',
        'pairs_with_resolved_both_sides':sum(bool(eligible[('dataset',r)]) and bool(eligible[('trial',t)]) for r,t in pairset),
        'roots_with_native_mesh':sum(bool(p['native_mesh_ids']) for p in dataset_profiles),
        'roots_with_specific_terms':sum(bool(p['specific_mesh_ids']) for p in dataset_profiles),
        'roots_with_redundant_broad_terms_refined':len({r['entity_id'] for r in refinements if r['entity_type']=='dataset'}),
        'neoplasms_only_roots':sum(p['specific_mesh_labels']==['Neoplasms'] for p in dataset_profiles),
        'roots_with_assay_evidence':sum(p['has_assay_evidence'] for p in assay_roots),
        'roots_with_mapped_assay':sum(p['has_mapped_assay'] for p in assay_roots),
        'pair_counts':dict(Counter(r['match_status'] for r in pairrows)),
        'pair_total':len(pairrows),'native_exact_pairs':sum(bool(p['native_exact_shared']) for p in pairrows),
        'field_only_exact_pairs':sum(bool(p['field_only_exact_shared']) for p in pairrows),
        'hierarchy_additional_pairs':sum(not p['native_exact_shared'] and p['bridge_count']>0 for p in pairrows),
        'broad_fallback_pairs':sum(p['has_broad_fallback'] for p in pairrows),
        'theme_counts':theme_rows,'specific_descriptor_counts':specific_rows,'assay_counts':[{'family':f,'roots':v,'percent_all_roots':100*v/len(rootset)} for f,v in family_counts.most_common()],
        'cross':cross,'unmapped_assay_labels':[{'label':l,'roots':v} for l,v in unmapped.most_common()],
        'candidates_pending':sum(c['mesh_id'] not in accepted[(c['entity_type'],c['entity_id'])] for c in candidates),'reviewed_additions':sum(len(v) for v in accepted.values()),
        'note':'MeSH concept profiles and broad assay display categories are exploratory; not RCDC categories, a validated classifier, or evidence of data harmonization at participant level.'}
    outputs={'semantic_recorded_annotations':recorded_rows,'semantic_recorded_descriptor_counts':recorded_terms,'semantic_annotations_resolved':resolved,'semantic_profiles':profiles,'semantic_profile_weights':profilelong,
        'semantic_refinement_audit':refinements,'semantic_text_review_candidates':candidates,'semantic_review_decisions_audit':review_audit,
        'semantic_trial_pair_comparison':pairrows,'semantic_trial_bridges':bridges,'semantic_assay_observations':normalized,
        'semantic_assay_observations_deduplicated':list(canonical.values()),'semantic_assay_exclusions':excluded,
        'semantic_assay_discrepancies':[x for x in normalized if x.get('sequence_flag_disagreement') or x.get('conflicting_duplicate_counts')],
        'semantic_assay_source_comparison':assay_source_comparison,'semantic_assay_profiles':assay_roots,'semantic_specific_descriptor_counts':specific_rows,'semantic_theme_counts':theme_rows,'semantic_theme_assay_matrix':cross,
        'semantic_assay_crosswalk':ASSAY_RULES,'semantic_source_coverage':coverage}
    return result,outputs


def load_semantic_inputs():
    folder=BASE/'semantic_sources';mf=folder/'manifest.json'
    result={'native':[],'texts':[],'coverage':[],'assays':[],'mesh':{},'obi':{},'pair_evidence':[],'ontology_status':{}}
    if mf.exists():
        manifest=json.loads(mf.read_text())
        for name,sha in manifest.get('outputs',{}).items():
            p=(folder/name).resolve()
            if not p.is_relative_to(folder.resolve()):raise ValueError('Unsafe semantic output path')
            if not p.exists() or hashfile(p)!=sha:raise ValueError('Semantic source hash mismatch: '+str(p))
        for key,name in [('native','native_annotations'),('texts','entity_text'),('coverage','source_coverage'),('assays','assay_observations'),('pair_evidence','dataset_trial_evidence')]:
            result[key]=sem_jsonl(folder/(name+'.jsonl'))
        for v in manifest.get('vocabularies',[]):
            p=Path(v['parsed_path']);raw=Path(v['raw_path'])
            if not p.is_file() or hashfile(p)!=v['parsed_sha256'] or not raw.is_file() or hashfile(raw)!=v['sha256']:raise ValueError('Vocabulary snapshot missing/changed; rerun A: '+v['key'])
            data=json.loads(p.read_text())
            if v['key'].startswith('mesh_'):result['mesh']=data['descriptors']
            elif v['key']=='obi':result['obi']=data['terms']
        result['ontology_status']=manifest.get('vocabulary_status',{})
    else:
        gap('semantics','Run A semantic collection: no native-recovery or pinned ontology sidecars found. No live lookup in B.')
    return result

if not REUSE:
    _semantic_inputs=load_semantic_inputs()
    _semantic_reviews=[]
    if (BASE/'semantic_mesh_adjudications.csv').exists():
        _semantic_reviews=pd.read_csv(BASE/'semantic_mesh_adjudications.csv',keep_default_na=False).to_dict('records')
    _semantic_summary,_semantic_tables=analyze_semantics(R.to_dict('records'),T,_semantic_inputs,_semantic_reviews)
    RESULTS['semantics']=_semantic_summary
    for _sem_name,_sem_rows in _semantic_tables.items():sem_output(_sem_name,_sem_rows,['status'])
    print('Semantic profiles:',{k:v for k,v in _semantic_summary.items() if isinstance(v,(int,str))})
    check('Semantic profiles preserve accession universe',_semantic_summary['n_roots']==N)
    _pair_sums=defaultdict(float)
    for _b in _semantic_tables['semantic_trial_bridges']:_pair_sums[(_b['root_accession'],_b['nct_id'])]+=_b['pair_weight']
    check('Semantic bridge weights sum to one per bridged pair',all(abs(v-1)<1e-9 for v in _pair_sums.values()))
    check('Native source rows retain original evidence IDs',all('evidence_id' in r for r in _semantic_tables['semantic_annotations_resolved']))
    check('No trial-derived dataset annotation',all(r['entity_type']!='dataset' or not str(r.get('source','')).startswith('https://clinicaltrials.gov') for r in _semantic_tables['semantic_annotations_resolved']))

    _native_counts=Counter(r['entity_id'] for r in _semantic_inputs['native'] if r['entity_type']=='dataset')
    _cov_by_root={r['entity_id']:r for r in _semantic_inputs['coverage'] if r['entity_type']=='dataset'}
    for _i,_r in S.iterrows():
        _root=_r.root_accession
        if _native_counts[_root]>0:S.loc[_i,'mesh']=True
        elif _cov_by_root.get(_root,{}).get('page_status') in ('native_mesh_section_empty','native_mesh_section_absent'):
            if _cov_by_root[_root].get('native_field_terms',0)==0:S.loc[_i,'mesh']=False
    for _cov in RESULTS['overview']['coverage']:
        if _cov['key']=='mesh':
            _v=S.mesh.astype('boolean');_cov.update(label='Native MeSH terms (expanded sources)',positive=int(_v.fillna(False).sum()) if _v.notna().any() else None,
                known=int(_v.notna().sum()),unknown=int(_v.isna().sum()),percentage_all_roots=100*int(_v.fillna(False).sum())/N if _v.notna().any() else None)
    export_table('overview_coverage',pd.DataFrame(RESULTS['overview']['coverage']))
    export_table('root_signals',S)
    # Most-specific observed descriptors for the later descriptive group comparison.
    _SEMANTIC_GROUP_MESH={}
    _ix=MeshProfileIndex(_semantic_inputs['mesh'])
    for _p in _semantic_tables['semantic_profiles']:
        if _p['entity_type']=='dataset' and _p['native_mesh_ids']:
            _SEMANTIC_GROUP_MESH[_p['entity_id']]=[{'mesh_id':m,'label':_ix.label(m),'term_key':m} for m in _ix.most_specific(_p['native_mesh_ids'])]


## Slide 5: 2x2 and intensity

In [ ]:
# @title Slide 4 -- 2x2 intersection and accession-level intensity
if not REUSE:
    known=S.dar.notna()&S.publication.notna();z=S.loc[known]
    rows=[]
    for a,b,label in [(1,1,'DAR + publication'),(0,1,'Publication, no DAR'),(1,0,'DAR, no publication'),(0,0,'Neither')]:
        n=int((z.dar.eq(bool(a))&z.publication.eq(bool(b))).sum())
        rows.append({'dar':a,'publication':b,'label':label,'count':n,'percent_known':100*n/len(z) if len(z) else None})
    check('2x2 counts sum to known denominator',sum(r['count'] for r in rows)==len(z))
    RESULTS['matrix']={'rows':rows,'known':len(z),'unknown':N-len(z)}
    RESULTS['paper_hist']=histogram_counts(S.paper_count)
    RESULTS['dar_hist']=histogram_counts(S.dar_count)
    export_table('dar_publication_2x2',pd.DataFrame(rows))
    display(pd.DataFrame(rows))


## Slide 6: Both recorded signals versus neither

Descriptive group sizes, subjects/samples/variables/consent-count summaries, and top five MeSH terms per group. The unestimated-model figure is removed. Existing model code is retained as an optional diagnostic (OFF) and never substitutes an earlier fit or changes the descriptive cohort.


In [ ]:
# @title New slide 5 -- multivariable comparison: DAR + publication vs neither
# Exploratory cross-sectional contrast, NOT a model of independent reuse or causation.
# No DAR count, paper count, RCR, or publication-derived funding variable enters X.
MODEL_FEATURES = {
 'subjects':('Subjects','log2p1'), 'variables':('Variables','log2p1'),
 'consent_codes':('Consent codes','log2p1'), 'data_versions':('Observed v# labels','log2p1'),
 'BDC':('BDC listed','binary'), 'SRA':('SRA available','binary'),
 'is_collection':('Collection','binary'), 'discovery_year':('Discovered artifact year','decade'),
 'admin_ic':('Administering IC','category')}


def comparison_population(signals):
    z=signals.copy();known=z.dar.notna()&z.publication.notna()
    both=known&z.dar.fillna(False)&z.publication.fillna(False)
    neither=known&~z.dar.fillna(False)&~z.publication.fillna(False)
    selected=z.loc[both|neither,['root_accession','BDC','SRA']].copy()
    selected['both_vs_neither']=both.loc[selected.index].astype(int)
    info={'both':int(both.sum()),'neither':int(neither.sum()),'mixed_excluded':int((known&~both&~neither).sum()),
          'unknown_excluded':int((~known).sum()),'eligible':len(selected)}
    return selected,info


def structural_families(root_ids,edges,family_input):
    ids=set(root_ids);parent={r:r for r in ids}
    def find(x):
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b:parent[max(a,b)]=min(a,b)
    usable=edges[edges.parent_root.isin(ids)&edges.child_root.isin(ids)]
    for row in usable.itertuples():union(row.parent_root,row.child_root)
    complete_map=False
    if len(family_input) and {'root_accession','family_id'}.issubset(family_input):
        f=family_input.copy()
        if 'verified' in f:f=f[f.verified.map(tri).fillna(False)]
        f=f[f.root_accession.isin(ids)].dropna(subset=['family_id'])
        if f.root_accession.duplicated().any():raise ValueError('Conflicting documented family IDs; fix source map before inference.')
        complete_map=ids.issubset(set(f.root_accession))
        for _,group in f.groupby('family_id'):
            members=group.root_accession.tolist()
            for r in members[1:]:union(members[0],r)
    groups={r:find(r) for r in ids}
    return groups,{'documented_map_complete':complete_map,'structural_edges_used':len(usable),
                    'family_count':len(set(groups.values())),
                    'limitation':'Unlinked roots are singleton working clusters; absence of a recorded family tie is not proof of independence.'}


def fit_both_neither(signals,profile,edges,tables):
    population,info=comparison_population(signals)
    f=population.merge(profile,on='root_accession',how='left',validate='one_to_one')
    if ACCESSION_DATE_BASIS=='ftp_artifact':
        date_table=tables['ftp'];date_field='first_preferred_public_versioned_ftp_timestamp'
    else:date_table=tables['ledger'];date_field='first_observed_utc'
    if 'root_accession' in date_table and date_field in date_table:
        years=date_table.drop_duplicates('root_accession').set_index('root_accession')[date_field].map(year)
        f['discovery_year']=f.root_accession.map(years).where(lambda x:x.between(1800,GROWTH_END_YEAR))
    else:f['discovery_year']=np.nan
    result={'status':'not_fitted','population':info,'specification':list(MODEL_COVARIATES),'included':[],'omitted':[],
            'terms':[],'warnings':[],'coefficient_rows':[],'model_type':'Binomial GLM with logit link',
            'interpretation':'Odds of both recorded signals versus neither, conditional on the two selected groups; not odds of reuse in all dbGaP.',
            'date_basis':ACCESSION_DATE_BASIS,'summary':[]}
    if not MODEL_ENABLED:result['reason']='Disabled in settings';return result,{},f
    unknown_specs=set(MODEL_COVARIATES)-set(MODEL_FEATURES)
    if unknown_specs:raise ValueError('Unsupported covariates (outcome-derived variables are not allowed): '+str(sorted(unknown_specs)))
    feature_data=pd.DataFrame(index=f.index);display_names={};summary=[]
    n=len(f)
    for key in MODEL_COVARIATES:
        label,kind=MODEL_FEATURES[key];raw=f[key] if key in f else pd.Series(np.nan,index=f.index)
        if kind=='category':
            raw=raw.map(lambda v:text(v) or None);valid=raw.notna();unique=raw.dropna().nunique()
        elif kind=='binary':
            raw=raw.map(tri).astype('Float64').astype(float);valid=raw.notna();unique=raw.dropna().nunique()
        else:
            raw=pd.to_numeric(raw,errors='coerce').replace([np.inf,-np.inf],np.nan)
            if kind=='log2p1':raw=raw.where(raw.ge(0))
            valid=raw.notna();unique=raw.dropna().nunique()
        for target,labelgroup in [(1,'DAR + publication'),(0,'Neither')]:
            vals=raw[f.both_vs_neither.eq(target)];nn=int(vals.notna().sum())
            summary.append({'feature':key,'label':label,'group':labelgroup,'eligible':len(vals),'known':nn,
                            'missing':int(vals.isna().sum()),'median':float(vals.median()) if nn and kind!='category' else None,
                            'p25':float(vals.quantile(.25)) if nn and kind!='category' else None,
                            'p75':float(vals.quantile(.75)) if nn and kind!='category' else None})
        if not n or valid.sum()<MODEL_MIN_OBSERVED or valid.mean()<MODEL_MIN_COVERAGE:
            result['omitted'].append({'feature':key,'reason':'Insufficient observed coverage','known':int(valid.sum())});continue
        if unique<2:result['omitted'].append({'feature':key,'reason':'No variation'});continue
        if kind=='log2p1':feature_data[key]=np.log2(1+raw);display_names[key]=label+' (2x [1+n])'
        elif kind=='decade':feature_data[key]=(raw-2016)/10;display_names[key]=label+' (+10 y)'
        elif kind=='binary':feature_data[key]=raw;display_names[key]=label+' (yes/no)'
        else:
            counts=raw.value_counts()
            keep=sorted(counts[counts.ge(MODEL_MIN_CATEGORY)].head(MODEL_MAX_CATEGORIES).index)
            pooled=raw.where(raw.isin(keep),'Other');pooled=pooled.where(raw.notna())
            levels=sorted(pooled.dropna().unique())
            if len(levels)<2:result['omitted'].append({'feature':key,'reason':'No variation after sparse-category grouping'});continue
            ref=levels[0];result['category_reference']={key:ref}
            for lev in levels[1:]:
                name=key+'='+str(lev);feature_data[name]=pooled.eq(lev).astype(float).where(pooled.notna())
                display_names[name]=label+': '+str(lev)+' vs '+str(ref)
        result['included'].append(key)
    result['summary']=summary
    if feature_data.shape[1]<2:result['reason']='Fewer than two nonconstant, sufficiently observed covariates';return result,{},f
    complete=feature_data.notna().all(axis=1)
    f['model_complete_case']=complete
    complete_cases=f.loc[complete].copy();y=complete_cases.both_vs_neither.astype(float)
    X=feature_data.loc[complete].astype(float)
    result['population'].update({'complete':len(y),'complete_both':int(y.sum()),'complete_neither':int(len(y)-y.sum()),
                                 'missing_excluded':int((~complete).sum())})
    if len(y)<MODEL_MIN_N or y.nunique()<2:
        result['reason']='Too few complete cases or one outcome group absent';return result,{},f
    # Detect exact collinearity instead of silently fitting a rank-deficient model.
    constant=[c for c in X if X[c].nunique()<2]
    for c in constant:result['omitted'].append({'feature':c,'reason':'Constant in complete-case sample'})
    X=X.drop(columns=constant)
    X.insert(0,'Intercept',1.)
    result['n_parameters']=X.shape[1]
    if X.shape[1]<3:result['reason']='Fewer than two varying predictors in complete-case sample';return result,{},f
    if np.linalg.matrix_rank(X.to_numpy())<X.shape[1]:
        result['reason']='Rank-deficient design; inspect correlated version/count predictors and adjust settings';return result,{},f
    if min(int(y.sum()),int(len(y)-y.sum()))<MODEL_MIN_CLASS_PER_PARAMETER*X.shape[1]:
        result['reason']='Sparse smaller outcome group relative to parameter count (configured safeguard)';return result,{},f
    family_map,family_info=structural_families(f.root_accession,edges,tables.get('families',pd.DataFrame()))
    groups=complete_cases.root_accession.map(family_map)
    nclusters=int(groups.nunique());use_clusters=nclusters<len(y)
    if use_clusters and nclusters<MODEL_MIN_CLUSTERS:
        result['reason']='Too few structural family clusters for requested cluster-robust uncertainty';return result,{},f
    result['family']=family_info;result['family']['fitted_clusters']=nclusters
    covariance='cluster' if use_clusters else 'HC1'
    result['covariance']=covariance
    if not family_info['documented_map_complete']:result['warnings'].append(family_info['limitation'])
    if 'discovery_year' in result['included']:
        result['warnings'].append('Discovery-year adjustment is a retained-artifact/first-observation proxy, not verified study age.')
    else:result['warnings'].append('No date adjustment is fitted; study age and time at risk remain uncontrolled.')
    try:
        import statsmodels.api as sm
        import warnings
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            model=sm.GLM(y.to_numpy(),X,family=sm.families.Binomial())
            fit=model.fit(maxiter=200,cov_type=covariance,
                          cov_kwds={'groups':groups.to_numpy(),'use_correction':True} if use_clusters else {})
        warning_text=[str(w.message) for w in caught];result['warnings']+=warning_text
        if not fit.converged or any('separation' in t.lower() for t in warning_text):
            result['reason']='Nonconvergence or separation; no inferential coefficients released';return result,{},f
        ci=fit.conf_int();param=fit.params
        if not np.isfinite(param).all() or not np.isfinite(ci.to_numpy()).all() or np.abs(param).max()>25:
            result['reason']='Nonfinite or extreme estimates; no inferential coefficients released';return result,{},f
        rows=[]
        for key in X.columns:
            if key=='Intercept':continue
            lo,hi=ci.loc[key]
            if max(abs(lo),abs(hi))>700:raise ValueError('Confidence interval cannot be exponentiated safely')
            rows.append({'term':key,'label':display_names[key],'log_odds':float(param[key]),'odds_ratio':float(np.exp(param[key])),
                         'ci_low':float(np.exp(lo)),'ci_high':float(np.exp(hi)),'p_value':float(fit.pvalues[key])})
        result.update({'status':'fitted','reason':'Exploratory simultaneous adjustment; no p-value-based selection',
                       'coefficient_rows':rows,'terms':list(X.columns),'model_n':len(y),
                       'formula':'both_vs_neither ~ '+' + '.join(X.columns[1:]),'converged':bool(fit.converged)})
        # Predicted probability is conditional on inclusion in these two groups.
        preds=complete_cases[['root_accession','both_vs_neither']].copy();preds['fitted_probability_both']=fit.predict(X).to_numpy()
        design=X.copy();design.insert(0,'root_accession',complete_cases.root_accession.to_numpy())
        return result,{'comparison_coefficients':pd.DataFrame(rows),'comparison_predictions':preds,'comparison_design':design},f
    except (ImportError,ValueError,np.linalg.LinAlgError,RuntimeError) as exc:
        result['reason']=f'Model unavailable: {type(exc).__name__}: {exc}';return result,{},f
    except Exception as exc:
        # Keep the rest of the informational deck usable, with the failure explicitly recorded.
        result['reason']=f'Model failed: {type(exc).__name__}: {exc}';return result,{},f

if not REUSE:
    comparison,model_tables,model_cohort=fit_both_neither(S,PROFILE,PROFILE_EDGES,TABLES)
    RESULTS['comparison']=comparison
    export_table('both_neither_cohort',model_cohort)
    export_table('both_neither_covariate_summary',pd.DataFrame(comparison['summary']))
    # Explicit empty schema prevents stale coefficients from a prior successful run being reused.
    coefficients=pd.DataFrame(comparison.get('coefficient_rows',[]),columns=['term','label','log_odds','odds_ratio','ci_low','ci_high','p_value'])
    export_table('comparison_coefficients',coefficients)
    for name,frame in model_tables.items():export_table(name,frame)
    if comparison['status']!='fitted':
        export_table('comparison_predictions',pd.DataFrame(columns=['root_accession','both_vs_neither','fitted_probability_both']))
        export_table('comparison_design',pd.DataFrame(columns=['root_accession']))
    if MODEL_ENABLED and comparison['status']!='fitted':gap('optional_model',comparison['reason'])
    print('Both versus neither:',comparison['status'],comparison['population'])
    if comparison['status']=='fitted':display(coefficients)
# Descriptive group comparison is independent of whether an optional model can fit.
def mesh_labels(raw):
    """Read supplied MeSH descriptors only; no title inference or label expansion."""
    result={}
    obj=decoded(raw)
    if obj is None:return []
    def add(code,label):
        code=text(code);label=text(label)
        if not label:return
        # Preserve source descriptor identifiers where supplied.
        key=code if code else 'label:'+canonical_label(label)
        if key not in result:result[key]={'mesh_id':code or None,'label':label,'term_key':key}
    def walk(x):
        if isinstance(x,dict):
            if 'coding' in x:
                for coding in x.get('coding') or []:
                    if isinstance(coding,dict) and coding.get('system') in {'urn:oid:2.16.840.1.113883.6.177','http://id.nlm.nih.gov/mesh','https://id.nlm.nih.gov/mesh'}:
                        add(coding.get('code'),coding.get('display'))
            elif any(k in x for k in ['mesh_id','label','display']):
                add(x.get('mesh_id',x.get('code',x.get('id'))),x.get('label',x.get('display')))
            else:
                for key,val in x.items():
                    if re.fullmatch(r'D\d+',str(key)) and isinstance(val,str):add(key,val)
        elif isinstance(x,(list,tuple,np.ndarray)):
            for y in x:walk(y)
        elif isinstance(x,str):
            # A string in the dedicated MeSH field remains an exact source label.
            if not re.fullmatch(r'D\d+',x):add(None,x)
    walk(obj)
    return list(result.values())


def group_details(signals,profile,roots):
    cohort,pop=comparison_population(signals)
    cohort=cohort.merge(profile,on='root_accession',how='left',validate='one_to_one')
    summary=[];mesh_rows=[];coverage=[]
    rootmap=roots.set_index('root_accession').to_dict('index')
    for _,row in cohort.iterrows():
        raw=rootmap[row.root_accession].get('mesh_terms')
        terms=globals().get("_SEMANTIC_GROUP_MESH",{}).get(row.root_accession,mesh_labels(raw))
        for term in terms:mesh_rows.append({'root_accession':row.root_accession,'group':int(row.both_vs_neither),**term})
        coverage.append({'root_accession':row.root_accession,'group':int(row.both_vs_neither),'has_named_mesh':bool(terms)})
    descriptors=pd.DataFrame(mesh_rows,columns=['root_accession','group','mesh_id','label','term_key'])
    cov=pd.DataFrame(coverage,columns=['root_accession','group','has_named_mesh'])
    groups=[]
    for target,label in [(1,'DAR + publication'),(0,'Neither')]:
        sub=cohort[cohort.both_vs_neither.eq(target)]
        for feature,ftitle in [('subjects','Subjects'),('samples','Samples'),('variables','Variables'),('consent_codes','Consent codes per accession')]:
            vals=pd.to_numeric(sub[feature],errors='coerce').dropna()
            summary.append({'feature':feature,'label':ftitle,'group':target,'group_label':label,'eligible':len(sub),
                            'known':len(vals),'missing':len(sub)-len(vals),
                            'median':float(vals.median()) if len(vals) else None,
                            'p25':float(vals.quantile(.25)) if len(vals) else None,
                            'p75':float(vals.quantile(.75)) if len(vals) else None})
        d=descriptors[descriptors.group.eq(target)]
        known=int(cov.loc[cov.group.eq(target),'has_named_mesh'].sum()) if len(cov) else 0
        freqs=d.groupby('term_key').root_accession.nunique().rename('accessions').reset_index()
        labels=d.drop_duplicates('term_key')[['term_key','mesh_id','label']]
        freqs=freqs.merge(labels,on='term_key',how='left').sort_values(['accessions','label','term_key'],ascending=[False,True,True])
        freqs['percentage_group']=100*freqs.accessions/len(sub) if len(sub) else np.nan
        freqs['percentage_named_mesh']=100*freqs.accessions/known if known else np.nan
        groups.append({'group':target,'label':label,'n':len(sub),'with_named_mesh':known,'without_named_mesh':len(sub)-known,
                       'top5':freqs.head(5).to_dict('records'),'all_terms':freqs.to_dict('records')})
    result={'population':pop,'summary':summary,'mesh_groups':groups,
            'note':'Descriptive, unadjusted comparison. MeSH frequencies count each root once per descriptor. Terms can overlap; percentages do not sum to 100. No causal or independent-reuse interpretation.',
            'mesh_basis':'Most-specific independently observed native descriptors from semantic sidecars when resolved; otherwise exact saved root terms. No unreviewed title-inferred diseases or trial-to-dataset copying.'}
    return result,cohort,descriptors,cov

if not REUSE:
    group_result,group_cohort,group_terms,group_coverage=group_details(S,PROFILE,R)
    RESULTS['group_comparison']=group_result
    export_table('both_neither_descriptive_summary',pd.DataFrame(group_result['summary']))
    export_table('both_neither_mesh_links',group_terms)
    export_table('both_neither_mesh_coverage',group_coverage)
    all_terms=[]
    for g in group_result['mesh_groups']:
        for entry in g['all_terms']:all_terms.append({'group':g['label'],'group_roots':g['n'],'with_named_mesh':g['with_named_mesh'],**entry})
    export_table('both_neither_mesh_terms',pd.DataFrame(all_terms,columns=['group','group_roots','with_named_mesh','term_key','accessions','mesh_id','label','percentage_group','percentage_named_mesh']))
    print('Descriptive comparison:',group_result['population'])
    for g in group_result['mesh_groups']:
        print(f"  {g['label']}: {g['with_named_mesh']:,}/{g['n']:,} roots have named MeSH terms")
        if not g['top5']:gap('5','No supplied MeSH labels for '+g['label']+'; no title inference used.')


## Slide 7: data contributors versus data accessors

Two columns retain annual PI/group and institution counts. The bottom row ranks contributed accessions per PI/group (left) and observed DAR root-WLID pairs per PI/group (right). Contributor attribution is not relabeled as verified upload activity.


In [ ]:
# @title Slide 7 -- contributor and accessor participation, PI volume and identity audit
if not REUSE:
    # Actual submission records require explicit supporting provenance and validated role.
    entities=TABLES['submission_entities'].copy()
    mode='verified_submission_records'
    if not entities.empty:
        required={'root_accession','entity_id','entity_name','institution_name','source_url','role'}
        check('Submission entity contract',required.issubset(entities),sorted(required-set(entities)))
        entities=entities[entities.role.astype(str).str.lower().isin(['data_submitter','submitting_pi','submitting_group']) & entities.source_url.map(text).ne('')].copy()
        if 'validated' in entities:entities=entities[entities.validated.map(tri).fillna(False)]
        entities['person_key']=entities.entity_id.map(text);entities['person_label']=entities.entity_name
        entities['institution_label']=entities.institution_name
    else:
        mode='public_study_pi_attribution_proxy'
        rows=[]
        if SUBMITTER_MODE != 'verified_only' and 'attributions_json' in attr:
            for row in attr.to_dict('records'):
                for entry in decoded(row.get('attributions_json')) or []:
                    role=text(entry.get('role'))
                    if not re.search(r'principal investigator|submitter|submitting|data coordinating',role,re.I):continue
                    if re.search(r'funding|project officer',role,re.I):continue
                    label=text(entry.get('person_or_group_label')) or text(entry.get('text'))
                    rows.append({'root_accession':row['root_accession'],'person_key':canonical_label(label),
                        'person_label':label,'institution_label':text(entry.get('organization_label')),
                        'role':role,'source_url':row.get('source_url')})
        entities=pd.DataFrame(rows,columns=['root_accession','person_key','person_label','institution_label','role','source_url'])
    entities['institution_key']=value(entities,'institution_label').map(canonical_label)
    subpeople=rank_records(entities,'person_key','person_label','distinct root / submitting entity' if mode.startswith('verified') else 'distinct root / attributed PI label (proxy)')
    suborg=rank_records(entities,'institution_key','institution_label','distinct roots per institution label')
    # Requester identity is explicit, conservative and label-based. Never use `name` (project title) as a PI name.
    req=D.copy()
    if 'root_accession' not in req:req['root_accession']=pd.Series(dtype='string')
    req['requester_id']=value(req,'wlid').map(text)
    req['institution_label']=value(req,'org_name').map(text)
    req['institution_key']=req.institution_label.map(canonical_label)
    req['person_label']=[' '.join(filter(None,[text(row.get('fname')),text(row.get('mname')),text(row.get('lname'))])) for row in req.to_dict('records')]
    req['person_key']=[canonical_label(n)+' | '+o if n else 'wlid:'+i for n,o,i in zip(req.person_label,req.institution_key,req.requester_id)]
    req.loc[req.person_label.eq(''),'person_label']='WLID '+req.loc[req.person_label.eq(''),'requester_id']
    req=req[req.requester_id.ne('')].copy()
    # Institutions ranked by unique root-WLID relationships, not raw duplicate rows.
    ir=req.drop_duplicates(['root_accession','requester_id','institution_key'])
    ir=ir[ir.institution_key.ne('')]
    orgcounts=ir.groupby('institution_key').size().sort_values(ascending=False)
    orglabels=ir.groupby('institution_key').institution_label.first()
    reqorg={'rows':[{'entity_key':k,'label':orglabels[k],'accessions':int(v)} for k,v in orgcounts.items()],
            'unique':len(orgcounts) if len(req) else None,'gini':gini(orgcounts.values),'unit':'distinct root-WLID pairs',
            'covered_roots':int(ir.root_accession.nunique())}
    req['person_display']=[n+' | '+o if o else n for n,o in zip(req.person_label,req.institution_label)]
    reqpeople=rank_records(req,'person_key','person_display','distinct roots per PI-name / organization label; ID fallback')
    wl=req.groupby('requester_id').size()
    rootpairs=req.drop_duplicates(['root_accession','requester_id']).groupby('requester_id').size()
    RESULTS['entities']={'submission_mode':mode,'submitter_people':subpeople,'submitter_institutions':suborg,
        'requester_people':reqpeople,'requester_institutions':reqorg,
        'unique_requester_ids':int(req.requester_id.nunique()) if any(i['key']=='dar' for i in INPUTS) else None,
        'requester_gini_rows':gini(wl.values),'requester_gini_root_pairs':gini(rootpairs.values),
        'requester_raw_rows_with_id':len(req),'submission_covered_roots':int(entities.root_accession.nunique()) if len(entities) else 0,
        'identity_note':'Exact displayed labels; no fuzzy person/ROR resolution. Study attribution is not proof of data upload.'}
    for key in ['submitter_people','submitter_institutions','requester_people','requester_institutions']:
        export_table(key,pd.DataFrame(RESULTS['entities'][key]['rows'],columns=['entity_key','label','accessions']))
    if not len(entities):gap('6','No submitter/attributed PI records. Run A public-attribution collection or provide validated submission_entities.')
    print('Requester IDs:',RESULTS['entities']['unique_requester_ids'],'; raw-row Gini:',RESULTS['entities']['requester_gini_rows'])


# @title Annual participation and request-volume distributions (replaces top-ten panels)
# Individual source rows are read-only. Annual counts are unique WITHIN year, not new entrants.
def ext_date(values):
    try:return pd.to_datetime(values, errors='coerce', utc=True, format='mixed')
    except TypeError:return pd.to_datetime(values, errors='coerce', utc=True)

def ext_bool(x):return text(x).lower() in {'true','1','1.0','yes'}

def ext_annual(frame,key,date_col,label,basis):
    """One entity per year; blank years when there is no usable input at all."""
    f=frame.copy()
    if key not in f:f[key]=pd.Series(dtype='string')
    if date_col not in f:f[date_col]=pd.Series(dtype='string')
    f['_date']=ext_date(f[date_col]);valid=f[key].map(text).ne('')&f['_date'].notna()
    if DATA_CUTOFF:valid &= f['_date'].le(pd.to_datetime(DATA_CUTOFF,utc=True))
    use=f.loc[valid].copy();use['_year']=use['_date'].dt.year
    counts=use.drop_duplicates([key,'_year']).groupby('_year').size()
    last=pd.Timestamp(DATA_CUTOFF).year if DATA_CUTOFF else GROWTH_END_YEAR
    rows=[{'year':y,'unique_entities':int(counts.get(y,0)) if len(use) and y<=last else None}
          for y in range(GROWTH_START_YEAR,GROWTH_END_YEAR+1)]
    first=use.groupby(key)._year.min() if len(use) else pd.Series(dtype=float)
    return {'label':label,'basis':basis,'rows':rows,'n_entities':int(use[key].nunique()),
            'n_eligible_records':len(use),'n_input_records':len(f),'excluded_missing_id_or_date':int((~valid).sum()),
            'before_window':int((use._year<GROWTH_START_YEAR).sum()),
            'new_entities':[{'year':y,'new_entities':int(first.eq(y).sum())} for y in range(GROWTH_START_YEAR,GROWTH_END_YEAR+1)],
            'note':'Counts unique entities within each year. A returning entity may appear in several years; no interpolation across first/last approval.'}

def ext_volume(frame,key,label):
    use=frame.loc[frame[key].map(text).ne('')].copy() if key in frame else pd.DataFrame()
    counts=use.groupby(key).size() if len(use) else pd.Series(dtype='int64')
    edges=[.5,1.5,4.5,9.5,24.5,49.5,99.5,249.5,499.5,999.5,np.inf]
    labs=['1','2-4','5-9','10-24','25-49','50-99','100-249','250-499','500-999','1000+']
    h,_=np.histogram(counts,edges)
    ranked=counts.sort_values(ascending=False,kind='stable')
    return {'label':label,'counts':h.tolist(),'labels':labs,'n':len(counts),'total_records':int(counts.sum()),
            'gini':gini(counts),'max':int(counts.max()) if len(counts) else None,
            'rows':[{'entity_key':k,'dar_root_wlid_pairs':int(v),'entity_rank':i+1} for i,(k,v) in enumerate(ranked.items())],
            'unit':'Distinct root-WLID relations per entity; neither administrative renewal counts nor completed uses.'}


def contributor_volume(frame,basis):
    """Rank contributed roots per PI/group, not files, versions or repeated assertions."""
    use=frame.copy()
    for key in ('root_accession','person_key'):
        if key not in use:use[key]=pd.Series(dtype='string')
    use=use.loc[use.root_accession.notna()&use.person_key.map(text).ne('')].copy()
    use=use.drop_duplicates(['root_accession','person_key'])
    counts=use.groupby('person_key').size() if len(use) else pd.Series(dtype='int64')
    pairs=sorted(counts.items(),key=lambda kv:(-int(kv[1]),str(kv[0])))
    display=value(use,'person_label',None)
    if display.isna().all():display=value(use,'entity_name',None)
    labels=dict(zip(use.person_key,display))
    rows=[{'entity_key':str(k),'label':text(labels.get(k)) or str(k),'contributed_accessions':int(v),'entity_rank':i+1} for i,(k,v) in enumerate(pairs)]
    proxy=basis=='current_attribution_by_discovery_cohort_proxy'
    return {'rows':rows,'n':len(rows),'total_records':int(counts.sum()),'max':int(counts.max()) if len(counts) else None,
        'gini':gini(counts),'value_field':'contributed_accessions','label':'Contributing PI / group labels',
        'basis':basis,'attribution_proxy':proxy,'covered_roots':int(use.root_accession.nunique()),
        'unit':'Distinct root-PI/group attribution pairs (not verified submissions)' if proxy else 'Distinct root-submitting PI/group pairs (not upload events)',
        'note':'One root counted once per PI/group. Co-attributed roots contribute to each listed PI. No historical participation inferred from current attribution.'}

if not REUSE:
    # Take authoritative submission-event dates only from an explicit, independently supported event table.
    se=TABLES['submission_events'].copy()
    if se.empty and 'submitted_at' in TABLES['submission_entities']:
        se=TABLES['submission_entities'].copy()
    needed={'root_accession','entity_id','entity_name','institution_name','submitted_at','role','source_url','validated'}
    if not se.empty and needed.issubset(se):
        se=se.loc[se.validated.map(ext_bool)&se.source_url.map(text).ne('') &
          se.role.astype(str).str.lower().isin(['data_submitter','submitting_pi','submitting_group'])].copy()
        se['person_key']=se.entity_id.map(text)
        se['institution_key']=[text(i) or canonical_label(n) for i,n in zip(value(se,'institution_id'),se.institution_name)]
        basis='verified_submission_events'
        temporal=se
        plabel='Data-submitting PIs / groups';olabel='Data-submitting institutions';date='submitted_at'
    else:
        temporal=entities.copy();date='_discovery_date'
        ff=TABLES['ftp'] if ACCESSION_DATE_BASIS=='ftp_artifact' else TABLES['ledger']
        field='first_preferred_public_versioned_ftp_timestamp' if ACCESSION_DATE_BASIS=='ftp_artifact' else 'first_observed_utc'
        dates=ff.drop_duplicates('root_accession').set_index('root_accession')[field] if field in ff and 'root_accession' in ff else pd.Series(dtype='object')
        temporal[date]=value(temporal,'root_accession').map(dates)
        basis='current_attribution_by_discovery_cohort_proxy'
        plabel='Attributed PIs / groups by cohort';olabel='Attributed institutions by cohort'
        gap('7','No validated dated submission events: upper annual panels show CURRENT attribution by root discovery cohort, not who actually submitted in that year.')
    # req was created in the existing entity block using fname/mname/lname, never project-title field name.
    rr=req.copy()
    rr=rr[rr.root_accession.notna() & rr.requester_id.map(text).ne('')].copy()
    rr['_event_date']=ext_date(value(rr,'initial_approval_date'))
    # Collapse repeated current snapshots of the same root-WLID relation. Select earliest dated evidence.
    rr=rr.sort_values('_event_date',na_position='last').drop_duplicates(['root_accession','requester_id'])
    rr['person_key']=[('nameorg:'+canonical_label(n)+'|'+o) if text(n) and not text(n).startswith('WLID ') else 'wlid:'+i
                      for n,o,i in zip(rr.person_label,rr.institution_key,rr.requester_id)]
    rr['person_key']=rr['person_key'].astype('string')
    a={
      'submitter_people':ext_annual(temporal,'person_key',date,plabel,basis),
      'submitter_institutions':ext_annual(temporal,'institution_key',date,olabel,basis),
      'requester_people':ext_annual(rr,'person_key','_event_date','DAR-requesting PIs / groups','initial_approval_date; name + organization proxy, WLID fallback'),
      'requester_institutions':ext_annual(rr,'institution_key','_event_date','DAR-requesting institutions','initial_approval_date; normalized exact organization labels')}
    volumes={'institutions':ext_volume(rr,'institution_key','Institutions'),
             'people':ext_volume(rr,'person_key','PI / group proxies')}
    contribution_source=se if basis=='verified_submission_events' else entities
    contribution=contributor_volume(contribution_source,basis)
    export_table('contributor_volume_per_pi',pd.DataFrame(contribution['rows'],columns=['entity_key','label','contributed_accessions','entity_rank']))
    RESULTS['participation']={'annual':a,'volumes':volumes,'contributor_volume':contribution,'submission_basis':basis,
        'n_unique_wlids':int(rr.requester_id.nunique()),'n_named_person_proxies':int(rr.loc[~rr.person_key.str.startswith('wlid:'),'person_key'].nunique()),
        'n_wlid_fallbacks':int(rr.loc[rr.person_key.str.startswith('wlid:'),'person_key'].nunique()),
        'n_missing_organization_records':int(rr.institution_key.eq('').sum()),
        'notes':'Annual requester counts are among observed approvals, not all application submissions. Current organization labels need not be historical affiliations. Distinct PI/group proxies are not adjudicated people; institutional aliases are not fuzzy-merged.'}
    for key,data in a.items():export_table('annual_'+key,pd.DataFrame(data['rows']))
    for key,data in volumes.items():export_table('dar_volume_per_'+key,pd.DataFrame(data['rows'],columns=['entity_key','dar_root_wlid_pairs','entity_rank']))
    export_table('requester_identity_audit',rr[['root_accession','requester_id','person_key','institution_key','_event_date']].rename(columns={'_event_date':'observed_initial_approval_date'}))
    print('Annual participation:',{k:v['n_entities'] for k,v in a.items()})


## New slide 8: data types and evolution
Current type counts, modality breadth, annual counts and within-cohort proportions. True submission dates take precedence; current labels stratified by discovery date remain an explicit proxy.

In [ ]:
# @title Data types over time: inventory by discovery cohort versus dated submissions
# One root may contribute to several assay categories. No pie chart implies they are disjoint.
def ext_type_analysis():
    obs=pd.DataFrame(_semantic_tables.get('semantic_assay_observations_deduplicated',[]))
    profiles=pd.DataFrame(_semantic_tables.get('semantic_assay_profiles',[]))
    pairs=[]
    for row in profiles.to_dict('records'):
        for fam in row.get('families',[]):pairs.append({'root_accession':row['root_accession'],'data_type':fam})
    rp=pd.DataFrame(pairs,columns=['root_accession','data_type']).drop_duplicates()
    snapshot_basis='semantic assay-source evidence'
    if rp.empty:
        # Conservative fallback from explicit saved root labels, not study titles.
        for row in R.to_dict('records'):
            raw=decoded(row.get('data_types'))
            if not isinstance(raw,(list,tuple,np.ndarray)):continue
            for item in raw:
                if isinstance(item,str):
                    norm=normalize_assay(item,{})
                    if norm.get('mapping_status')!='unmapped':pairs.append({'root_accession':row['root_accession'],'data_type':norm['family']})
        rp=pd.DataFrame(pairs,columns=['root_accession','data_type']).drop_duplicates()
        snapshot_basis='explicit saved data_types labels; no raw or timing validation'
    totals=rp.groupby('data_type').root_accession.nunique().sort_values(ascending=False)
    te=TABLES['assay_submission_events'].copy()
    needed={'root_accession','data_type','submitted_at','validated','source_url'}
    if not te.empty and needed.issubset(te):
        te=te.loc[te.validated.map(ext_bool)&te.source_url.map(text).ne('')].copy()
        te['data_type']=te.data_type.map(lambda x:normalize_assay(x,{}).get('family') or text(x))
        te['_date']=ext_date(te.submitted_at);basis='verified dated assay submission events'
        event=te
    else:
        ff=TABLES['ftp'] if ACCESSION_DATE_BASIS=='ftp_artifact' else TABLES['ledger']
        field='first_preferred_public_versioned_ftp_timestamp' if ACCESSION_DATE_BASIS=='ftp_artifact' else 'first_observed_utc'
        dates=ff.drop_duplicates('root_accession').set_index('root_accession')[field] if field in ff and 'root_accession' in ff else pd.Series(dtype='object')
        event=rp.copy();event['_date']=ext_date(event.root_accession.map(dates));basis='current data-type labels by accession discovery cohort (NOT submission chronology)'
    if DATA_CUTOFF:event=event[event['_date'].isna()|event['_date'].le(pd.to_datetime(DATA_CUTOFF,utc=True))]
    event=event.loc[event['_date'].notna()].copy();event['year']=event['_date'].dt.year
    # First observed root-type event; later re-submissions do not reintroduce that root-type pair.
    event=event.sort_values('_date').drop_duplicates(['root_accession','data_type'])
    annual=event.groupby(['year','data_type']).size()
    cats=list(totals.index)
    for v in sorted(set(value(event,'data_type').dropna())-set(cats)):cats.append(v)
    years=list(range(GROWTH_START_YEAR,GROWTH_END_YEAR+1));rows=[]
    for cat in cats:
        cum=int(((event['year']<GROWTH_START_YEAR)&event.data_type.eq(cat)).sum())
        for y in years:
            n=int(annual.get((y,cat),0));cum+=n
            known=event.loc[event.year.eq(y),'root_accession'].nunique()
            rows.append({'year':y,'data_type':cat,'annual_roots':n if (not DATA_CUTOFF or y<=pd.Timestamp(DATA_CUTOFF).year) else None,'cumulative_root_type_pairs':cum if (not DATA_CUTOFF or y<=pd.Timestamp(DATA_CUTOFF).year) else None,
                         'roots_with_any_dated_type_in_cohort':int(known),
                         'percent_of_annotated_cohort':100*n/known if known and (not DATA_CUTOFF or y<=pd.Timestamp(DATA_CUTOFF).year) else None,'date_basis':basis})
    breadth=rp.groupby('root_accession').size();_hi=int(breadth.max()) if len(breadth) else 0
    distributions={'labels':[str(i) for i in range(1,_hi+1)],'counts':[int(breadth.eq(i).sum()) for i in range(1,_hi+1)],'n':len(breadth)}
    # Snapshot specimen totals intentionally remain non-additive: same sample may receive several assays.
    sample=[]
    if not obs.empty:
        # Use one retained version/participant-set pair per owning root, not a sum over version history.
        if {'root_accession','data_version','participant_set'}.issubset(obs):
            obs['_v']=pd.to_numeric(obs.data_version,errors='coerce');obs['_p']=pd.to_numeric(obs.participant_set,errors='coerce')
            chosen=obs.sort_values(['_v','_p'],na_position='first').drop_duplicates('root_accession',keep='last')[['root_accession','data_version','participant_set']]
            obs=obs.merge(chosen,on=['root_accession','data_version','participant_set'],how='inner',validate='many_to_one')
        for cat,g in obs.groupby('family',dropna=False) if 'family' in obs else []:
            v=pd.to_numeric(value(g,'sample_count'),errors='coerce')
            sample.append({'data_type':text(cat),'reported_sample_assay_total':float(v.dropna().sum()) if v.notna().any() else None,
                           'records_with_sample_count':int(v.notna().sum()),'scope':'selected retained version sample-assay counts; NOT unique biological samples; not summed across historical versions'})
    out={'basis':basis,'snapshot_basis':snapshot_basis,'categories':cats,'rows':rows,
         'type_counts':[{'data_type':c,'roots':int(n),'percent_all_roots':100*n/N} for c,n in totals.items()],
         'annotated_roots':int(rp.root_accession.nunique()),'dated_roots':int(event.root_accession.nunique()),
         'undated_root_types':max(0,len(rp)-len(event)),'types_per_root':distributions,
         'sample_assay_counts':sample,'note':'Current types can have been added after root discovery. Discovery-cohort plots describe present composition of old/new roots, not the first availability of each assay. Categories overlap; absence of a label is not absence of the data type.'}
    return out,rp

if not REUSE:
    RESULTS['data_type_evolution'],_type_pairs=ext_type_analysis()
    export_table('data_type_annual',pd.DataFrame(RESULTS['data_type_evolution']['rows']))
    export_table('data_type_current_counts',pd.DataFrame(RESULTS['data_type_evolution']['type_counts']))
    export_table('data_type_sample_assay_counts',pd.DataFrame(RESULTS['data_type_evolution']['sample_assay_counts']))
    export_table('data_type_root_pairs',_type_pairs)
    print('Data-type time basis:',RESULTS['data_type_evolution']['basis'])


## Published-study replication workbench
Reimplements the documented main-figure analyses when the required instruments are supplied. The default report separates method-matched new-cohort calculations, descriptive proxies, reference values, and missing instruments. It is not a reproduction of the original investigators' numerical results.

In [ ]:
# @title Published-study analysis recipes: observed proxies versus method-matched reanalysis
# Definitions follow Paltoo et al. 2014 and Coady et al. 2017 main-text figures/captions.
# Historical series are NOT digitized or invented. Missing administrative instruments remain explicit gaps.
PAPER_REFS={
 'paltoo':{'doi':'10.1038/ng.3062','pmid':'25162809','url':'https://pmc.ncbi.nlm.nih.gov/articles/PMC4182942/',
           'reference_end':'2013-12-01','reference_units':'studies, consent-group DARs, investigators, DAC-reported secondary-use papers'},
 'coady':{'doi':'10.1056/NEJMsa1603542','pmid':'28402243','url':'https://pmc.ncbi.nlm.nih.gov/articles/PMC5665376/',
          'reference_end':'2016-05-31','reference_units':'repository studies, fulfilled requests, request-linked publications'},
 'scope':'Main-text methods/captions inspected; supplementary figure layouts and original machine-readable series not available in this build. Original author-level data are not bundled.'}

# Original Table 1 numbers; percentages are WITHIN profit-status blocks, not across all institutions.
PALTOO_TABLE1_REFERENCE=[
 ['Not for profit','Academic',340,64.0,1473,70.5],['Not for profit','Government',37,7.0,151,7.2],
 ['Not for profit','Hospital',54,10.2,186,8.9],['Not for profit','Research institute',100,18.8,279,13.4],
 ['For profit','Biotechnology',68,57.1,86,51.2],['For profit','Genealogy',1,.8,1,.6],
 ['For profit','Hospital',2,1.7,3,1.8],['For profit','Information technology',12,10.1,16,9.5],
 ['For profit','Pharma',33,27.7,59,35.1],['For profit','Research',3,2.5,3,1.8]]

REPLICATION_CONTRACTS={
 'submission_events':['submission_id','root_accession','entity_id','entity_name','institution_id','institution_name','submitted_at','role','source_url','validated'],
 'assay_submission_events':['root_accession','data_type','submitted_at','source_url','validated'],
 'rep_lifecycle':['study_id','root_accession','study_type','release_date','followup_end','release_verified','request_history_complete','publication_history_complete','n_participants','completion_year','purpose','research_area','source_url'],
 'rep_access':['request_id','study_id','requester_id','institution_id','consent_group_id','request_date','decision_date','approved','fulfilled_date','download_date','download_observation_complete','request_unit','purpose_category','purpose_validated','external_funding','external_funding_known','source_url','validated'],
 'rep_publications':['study_id','request_id','pmid','publication_date','secondary_use_verified','request_link_verified','source_url','validated'],
 'rep_institutions':['institution_id','institution_label','profit_status','institution_type','country','continent','latitude','longitude','source_url','validated'],
 'rep_samples':['study_id','root_accession','sample_id','broad_phenotype','technology','snp_array_size','snapshot_date','source_url','validated'],
 'rep_topics':['pmid','topic','source_url','validated'],
 'rep_metrics':['pmid','journal','citation_count','citation_percentile','percentile_direction','metric_system','metric_as_of','publication_date','cohort','source_url','validated']}


def rep_flag(x):
    v=text(x).casefold()
    if v in {'true','1','1.0','yes'}:return True
    if v in {'false','0','0.0','no'}:return False
    return None


def rep_valid(frame,columns):
    f=frame.copy();missing=sorted(set(columns)-set(f.columns))
    if missing:return pd.DataFrame(columns=columns),'Missing fields: '+', '.join(missing)
    if 'validated' in f:f=f[f.validated.map(ext_bool)].copy()
    if 'source_url' in f:f=f[f.source_url.map(text).ne('')].copy()
    return f,('No validated source-supported rows' if f.empty else '')


def rep_period(y):
    if pd.isna(y):return 'Unknown'
    y=int(y)
    if 2000<=y<=2003:return '2000-2003'
    if 2004<=y<=2007:return '2004-2007'
    if 2008<=y<=2011:return '2008-2011'
    if 2012<=y<=2016:return '2012-2016'
    if y>=2017:return f'{2017+4*((y-2017)//4)}-{2020+4*((y-2017)//4)}'
    return 'Before 2000'


def rep_km(durations,events):
    """Product-limit survival, log-log Greenwood intervals, explicit risk sets.
    Curves terminate at the last observed follow-up. Ties: events occur before censoring.
    """
    d=np.asarray(durations,float);e=np.asarray(events,bool)
    keep=np.isfinite(d)&(d>=0);d=d[keep];e=e[keep]
    if len(d)==0:return []
    out=[{'years':0.0,'survival':1.0,'survival_low':1.0,'survival_high':1.0,'at_risk':len(d),'events':0,'censored':0}]
    surv=1.;g=0.
    for t in np.unique(d):
        risk=int((d>=t).sum());n=int(((d==t)&e).sum());c=int(((d==t)&~e).sum())
        if n:
            surv*=1-n/risk
            if risk>n:g+=n/(risk*(risk-n))
        if 0<surv<1:
            theta=math.log(-math.log(surv));se=math.sqrt(g)/abs(math.log(surv))
            lo=math.exp(-math.exp(theta+1.95996398454*se));hi=math.exp(-math.exp(theta-1.95996398454*se))
        else:lo=hi=surv
        out.append({'years':float(t),'survival':surv,'survival_low':lo,'survival_high':hi,'at_risk':risk,'events':n,'censored':c})
    return out


def rep_logrank_two_groups(frame):
    """Two-group log-rank test; tied events precede censoring, one degree of freedom."""
    groups=sorted(frame.study_type.dropna().unique()) if len(frame) else []
    if len(groups)!=2:return {'chi_square':None,'p_value':None,'reason':'Need exactly two study types'}
    a=frame[frame.study_type.eq(groups[0])];b=frame[frame.study_type.eq(groups[1])];oe=0.;variance=0.
    for t in sorted(frame.loc[frame.event,'years'].unique()):
        n1=int(a.years.ge(t).sum());n2=int(b.years.ge(t).sum());n=n1+n2
        d1=int((a.years.eq(t)&a.event).sum());d2=int((b.years.eq(t)&b.event).sum());d=d1+d2
        if n:oe+=d1-d*n1/n
        if n>1:variance+=n1*n2*d*(n-d)/(n*n*(n-1))
    if variance<=0:return {'chi_square':None,'p_value':None,'reason':'No informative event contrast'}
    chi=oe*oe/variance
    return {'chi_square':chi,'p_value':math.erfc(math.sqrt(chi/2)),'df':1,'groups':groups,'reason':'Unadjusted exploratory log-rank comparison'}


def build_literature_analyses():
    report={'references':PAPER_REFS,'items':[],'tables':{},'plots':{},'exclusions':[]}
    def add(key,paper,location,title,analysis_status,detail,table=None,plot=None):
        row={'id':key,'paper':paper,'location':location,'title':title,'status':analysis_status,'detail':detail,
             'source_url':PAPER_REFS[paper]['url']}
        report['items'].append(row)
        if table is not None:
            report['tables'][key]=safe(table if isinstance(table,list) else table.to_dict('records'))
            export_table('rep_'+key,pd.DataFrame(report['tables'][key]))
        if plot is not None:report['plots'][key]=safe(plot)
        return row
    # ---- Paltoo: phenotype x genomic technology; samples not accessions ----
    sa,msg=rep_valid(TABLES['rep_samples'],['sample_id','study_id','broad_phenotype','technology','snp_array_size','source_url','validated'])
    if len(sa):
        for c in ['sample_id','study_id','broad_phenotype','technology']:sa[c]=sa[c].map(text)
        sa=sa[sa.sample_id.ne('')&sa.study_id.ne('')].copy()
        for key,cols,filter_mask in [
             ('paltoo_f1a',['broad_phenotype','snp_array_size'],sa.technology.str.lower().isin(['snp array','genotyping array','gwas'])),
             ('paltoo_f1b',['technology'],sa.technology.str.lower().isin(['whole genome','whole-genome sequencing','whole exome','exome sequencing','targeted genome','rna-seq','rna sequencing'])),
             ('paltoo_f1c',['technology'],pd.Series(True,index=sa.index))]:
            q=sa.loc[filter_mask].drop_duplicates(['study_id','sample_id']+cols)
            counts=q.groupby(cols,dropna=False).size().reset_index(name='sample_assay_observations')
            add(key,'paltoo','Figure 1'+key[-1],'Samples by phenotype / technology','method_matched_new_cohort' if len(q) else 'no_eligible_rows',
                'Distinct study-sample identities within category. Cross-category reuse remains possible; all units and category labels are exported. No participant-level equivalence is asserted.',counts,
                {'kind':'category','records':counts.to_dict('records'),'categories':cols,'value':'sample_assay_observations'})
    else:
        for key,loc in [('paltoo_f1a','Figure 1a'),('paltoo_f1b','Figure 1b'),('paltoo_f1c','Figure 1c')]:
            add(key,'paltoo',loc,'Sample composition','not_estimable',msg or 'Need dated sample identities, technology and submitted broad phenotype; array-size bins are not present in SSTR totals.')
    # A useful calculated analogue is provided without calling accession counts sample counts.
    co=[];themes={r['entity_id']:r.get('specific_mesh_labels',[]) for r in _semantic_tables.get('semantic_profiles',[]) if r.get('entity_type')=='dataset'}
    for t in _type_pairs.to_dict('records'):
        for disease in themes.get(t['root_accession'],[]) or ['Unclassified native MeSH']:
            co.append({'root_accession':t['root_accession'],'disease':disease,'data_type':t['data_type']})
    if co:
        cc=pd.DataFrame(co).drop_duplicates().groupby(['disease','data_type']).root_accession.nunique().reset_index(name='roots')
        add('paltoo_f1_root_analogue','paltoo','Figure 1 analogue','Current accession composition','descriptive_proxy',
            'Root counts by native MeSH descriptor and data family; NOT Paltoo sample totals. A root can occupy several categories.',cc,
            {'kind':'category','records':cc.to_dict('records'),'categories':['disease','data_type'],'value':'roots'})
    # ---- Strict administrative event inputs ----
    access,amsg=rep_valid(TABLES['rep_access'],['request_id','study_id','requester_id','institution_id','request_date','approved','source_url','validated'])
    life,lmsg=rep_valid(TABLES['rep_lifecycle'],['study_id','study_type','release_date','followup_end','release_verified','request_history_complete','publication_history_complete','source_url'])
    if len(life):
        life=life.loc[life.release_verified.map(ext_bool)].copy()
        life['study_id']=life.study_id.map(text);life=life[life.study_id.ne('')].copy()
        if life.study_id.duplicated().any():raise ValueError('rep_lifecycle must have one explicit repository study per study_id. Do not duplicate parent/child roots.')
        life['_release']=ext_date(life.release_date);life['_end']=ext_date(life.followup_end)
        if DATA_CUTOFF:
            _cut=pd.to_datetime(DATA_CUTOFF,utc=True)
            life['_end']=life['_end'].where(life['_end'].isna()|life['_end'].le(_cut),_cut)
        eligible=life['_release'].notna()&life['_end'].notna()&life['_end'].ge(life['_release'])&life.study_type.isin(['clinical_trial','observational'])
        report['exclusions'].extend([{'module':'lifecycle','study_id':r.study_id,'reason':'Missing/invalid verified release, follow-up, or study_type'} for r in life.loc[~eligible].itertuples()])
        life=life[eligible].copy()
    if len(access):
        # Consent-group applications can share project IDs; explicit unique request_id required.
        for _c in ['request_id','study_id','requester_id','institution_id']:access[_c]=access[_c].map(text)
        access=access.loc[access.request_id.ne('')&access.study_id.ne('')&access.requester_id.ne('')].copy()
        if access.request_id.duplicated().any():raise ValueError('rep_access request_id is not unique; provide actual consent-group request keys, not project IDs.')
        for c in ['request_date','decision_date','fulfilled_date','download_date']:
            access['_'+c]=ext_date(value(access,c))
            if DATA_CUTOFF and c!='request_date':access.loc[access['_'+c].gt(pd.to_datetime(DATA_CUTOFF,utc=True)),'_'+c]=pd.NaT
        access['_approved']=access.approved.map(rep_flag)
        access['_download_known']=value(access,'download_observation_complete').map(ext_bool)|access['_download_date'].notna()
        # Impossible negative dates are excluded, never repaired by clipping.
        for _c in ['decision_date','fulfilled_date','download_date']:
            _bad=access['_'+_c].lt(access['_request_date'])
            report['exclusions'].extend([{'module':'access','study_id':r.study_id,'request_id':r.request_id,'reason':_c+' predates request; excluded from endpoint'} for r in access.loc[_bad].itertuples()])
            access.loc[_bad,'_'+_c]=pd.NaT
            if _c=='download_date':access.loc[_bad,'_download_known']=False
        if DATA_CUTOFF:access=access[access['_request_date'].le(pd.to_datetime(DATA_CUTOFF,utc=True))].copy()
    # Paltoo F2: separate calendrical counts and cumulative counts, not inferred from snapshot status.
    consent=access[value(access,'request_unit').astype(str).eq('consent_group')] if len(access) else access
    if len(consent):
        dd=consent.copy();dd['_year']=dd['_request_date'].dt.year
        years=range(int(dd._year.min()),int(dd._year.max())+1) if dd._year.notna().any() else []
        rows=[]
        for y in years:
            g=dd[dd._year.eq(y)];days=(g['_decision_date']-g['_request_date']).dt.total_seconds()/86400;days=days[days>=0]
            ap_known=g['_approved'].notna();dl_known=g['_download_known']
            ap_positive=int(g['_approved'].eq(True).sum());dl_positive=int(g['_download_date'].notna().sum())
            rows.append({'year':y,'submitted_requests':len(g),
                         'approved_requests':ap_positive if ap_known.all() else None,
                         'observed_approvals_lower_bound':ap_positive,'approval_status_known':int(ap_known.sum()),
                         'downloaded_request_units':dl_positive if dl_known.all() else None,
                         'observed_downloads_lower_bound':dl_positive,'download_status_known':int(dl_known.sum()),
                         'mean_processing_days':float(days.mean()) if len(days) else None,'n_processing_dates':len(days)})
        table=pd.DataFrame(rows)
        if len(table):
            for c in ['submitted_requests','approved_requests','downloaded_request_units']:table['cumulative_'+c]=table[c].cumsum(skipna=False) if table[c].notna().any() else None
        add('paltoo_f2','paltoo','Figure 2','Access application, approval, download and processing','method_aligned_adaptation',
             'Request-cohort calendar grouping. Processing uses request-to-decision dates; download count requires a recorded download for that request. Missing processing/download fields are not zero. Request-unit download counts need not equal unique physical datasets downloaded in the paper. Unknown status withholds a total and retains a lower bound. Precise original series grouping cannot be inferred from the article alone.',table,
             {'kind':'access','records':table.to_dict('records')})
    else:
        add('paltoo_f2','paltoo','Figure 2','Access operations','not_estimable',amsg or 'Current approved-project rows lack consent-group application IDs, submission/decision timestamps and download events; approvals cannot reproduce the other series.')
    # ---- Table 1: profit-specific denominators and distinct investigator-institution links ----
    ins,imsg=rep_valid(TABLES['rep_institutions'],['institution_id','profit_status','institution_type','source_url','validated'])
    ref=pd.DataFrame(PALTOO_TABLE1_REFERENCE,columns=['profit_status','institution_type','institutions','published_percent_institutions','investigators','published_percent_investigators'])
    add('paltoo_t1_published','paltoo','Table 1','Published institution table','published_reference','Transcribed numeric reference values; never merged into the current dataset. Percentages are within profit-status stratum.',ref)
    if len(ins) and len(access):
        if ins.institution_id.duplicated().any():raise ValueError('rep_institutions needs one reviewed record per institution_id.')
        a=access[access['_approved'].eq(True)].merge(ins,on='institution_id',how='left',validate='many_to_one',suffixes=('','_institution'))
        rows=[]
        for profit,g in a.dropna(subset=['profit_status','institution_type']).groupby('profit_status'):
            ni=g.institution_id.nunique();nv=len(g[['requester_id','institution_id']].drop_duplicates())
            for cat,q in g.groupby('institution_type'):
                nr=q.institution_id.nunique();nvq=len(q[['requester_id','institution_id']].drop_duplicates())
                rows.append({'profit_status':profit,'institution_type':cat,'institutions':nr,'percent_institutions':100*nr/ni,
                             'investigator_institution_links':nvq,'percent_investigator_links':100*nvq/nv})
        add('paltoo_t1_current','paltoo','Table 1 analogue','Current institution classification','method_matched_new_cohort',
             'Approved requesters only. Explicit reviewed taxonomy; unknown organizations excluded and reported. Investigators affiliated with multiple institutions contribute to multiple links.',rows)
    else:add('paltoo_t1_current','paltoo','Table 1 analogue','Institution-type composition','not_estimable',imsg or 'Need curated institution types/profit status and approval-linked investigator identities; no name-based guesses.')
    # ---- Publications: request-linked/verified versus all page-linked ----
    pu,pmsg=rep_valid(TABLES['rep_publications'],['study_id','request_id','pmid','publication_date','secondary_use_verified','request_link_verified','source_url','validated'])
    if len(pu):
        pu['pmid']=pu.pmid.map(pmidid);pu['_pubdate']=ext_date(pu.publication_date)
        pu=pu[pu.pmid.notna()&pu._pubdate.notna()&pu.secondary_use_verified.map(ext_bool)].copy()
        if pu.groupby('pmid')._pubdate.nunique().gt(1).any():raise ValueError('Conflicting publication dates for the same PMID in replication inputs; reconcile before scoring.')
        pu=pu.sort_values('_pubdate')
        if DATA_CUTOFF:pu=pu[pu._pubdate.le(pd.to_datetime(DATA_CUTOFF,utc=True))]
    if len(pu):
        p=pu.drop_duplicates('pmid');g=cumulative(p._pubdate,'Verified secondary publications')
        add('paltoo_f3','paltoo','Figure 3','Verified secondary-publication accumulation','method_matched_new_cohort',
            'Only verified secondary-use records; use DAC annual-report provenance for the original ascertainment mechanism. Endpoints and selected corpus differ from the historical paper.',g['rows'],{'kind':'time','records':g['rows'],'series':['cumulative']})
    else:add('paltoo_f3','paltoo','Figure 3','Secondary-publication accumulation','not_estimable',pmsg or 'Linked PubMed records are not DAC-reported or independently verified secondary-use publications.')
    add('paltoo_f3_linked_analogue','paltoo','Figure 3 analogue','All linked publication accumulation','descriptive_proxy',
        'Uses the existing unique linked-PMID chronology. Not an estimate of secondary reuse.',RESULTS['growth']['publications']['rows'],
        {'kind':'time','records':RESULTS['growth']['publications']['rows'],'series':['cumulative']})
    # ---- Coady Figure 1: deduplicated fulfilled study x investigator delivery ----
    curves=[];usable_access=pd.DataFrame();invalid_access_studies=set()
    if len(life) and len(access):
        usable_access=access.merge(life[['study_id','study_type','_release','_end']],on='study_id',how='inner',validate='many_to_one')
        bad=usable_access._request_date.lt(usable_access._release)|usable_access._request_date.gt(usable_access._end)
        report['exclusions'].extend([{'module':'access','study_id':r.study_id,'request_id':r.request_id,'reason':'Request outside verified release/follow-up interval'} for r in usable_access.loc[bad].itertuples()])
        invalid_access_studies=set(usable_access.loc[bad|usable_access._request_date.isna(),'study_id'])
        usable_access=usable_access[~bad & usable_access._request_date.notna()].copy()
        delivered=usable_access[usable_access['_fulfilled_date'].notna() & usable_access._fulfilled_date.ge(usable_access._release)&usable_access._fulfilled_date.le(usable_access._end)].sort_values('_fulfilled_date').drop_duplicates(['study_id','requester_id'])
        for typ in ['clinical_trial','observational']:
            ls=life[life.study_type.eq(typ)];a=delivered[delivered.study_type.eq(typ)]
            if ls.empty:continue
            start=int(ls._release.dt.year.min());end=int(ls._end.dt.year.max())
            for y in range(start,end+1):curves.append({'year':y,'study_type':typ,'available_studies':int((ls._release.dt.year<=y).sum()),'cumulative_study_requester_deliveries':int((a._fulfilled_date.dt.year<=y).sum())})
        add('coady_f1','coady','Figure 1A/B','Availability and fulfilled requests','method_matched_new_cohort' if len(delivered) else 'partial',
            'Separate clinical-trial and observational-study panels. Repeated deliveries of the same study to the same requester are collapsed. Without fulfilled_date only the availability series is supported.',curves,
            {'kind':'coady_availability','records':curves,'deliveries_measured':bool(len(delivered))})
        for _typ,_suffix in [('clinical_trial','a'),('observational','b')]:
            _panel=[r for r in curves if r['study_type']==_typ]
            if _panel:add('coady_f1'+_suffix,'coady','Figure 1'+_suffix.upper(),_typ.replace('_',' ').title()+': inventory and delivery','method_matched_new_cohort','One fulfilled study-requester pair, repeated deliveries removed. Annual cumulative series; not a pixel-copy of the source.',_panel,{'kind':'coady_availability','records':_panel,'deliveries_measured':bool(len(delivered))})
    else:add('coady_f1','coady','Figure 1A/B','Availability and fulfilled requests','not_estimable',lmsg or amsg or 'Need verified study release dates, explicit study types and fulfilled delivery records.')
    # ---- Coady Figure 2 / Paltoo S1: institution-level coordinates from supplied crosswalk ----
    if len(usable_access) and len(ins) and {'latitude','longitude'}.issubset(ins):
        loc=ins.copy();loc['latitude']=pd.to_numeric(loc.latitude,errors='coerce');loc['longitude']=pd.to_numeric(loc.longitude,errors='coerce')
        loc=loc[loc.latitude.between(-90,90)&loc.longitude.between(-180,180)]
        act=usable_access[usable_access['_fulfilled_date'].notna()&usable_access._fulfilled_date.ge(usable_access._release)&usable_access._fulfilled_date.le(usable_access._end)].copy()
        trialusers=set(act.loc[act.study_type.eq('clinical_trial'),'requester_id'])
        act['cohort']=act.requester_id.map(lambda x:'At least one clinical trial' if x in trialusers else 'Observational only')
        act=act.sort_values('_fulfilled_date').drop_duplicates('requester_id')
        q=act[['requester_id','institution_id','cohort']].merge(loc,on='institution_id',how='left',validate='many_to_one')
        loccounts=q.dropna(subset=['latitude','longitude']).groupby(['longitude','latitude','cohort']).requester_id.nunique().reset_index(name='investigators')
        add('coady_f2','coady','Figure 2','Requester geography','method_matched_new_cohort' if len(loccounts) else 'not_estimable',
            f'{int(q.latitude.notna().sum())}/{len(q)} fulfilled-data requesters located using curated institution coordinates at first recorded delivery. No geocoding or country inference.',loccounts,
            {'kind':'geo','records':loccounts.to_dict('records')})
        if 'country' in q:
            country=q.groupby(['country','cohort'],dropna=False).requester_id.nunique().reset_index(name='requesters')
            add('paltoo_s1','paltoo','Supplementary Figure 1 analogue','Country distribution','descriptive_proxy','Supplied curated country labels; same explicit located requester subset, not all historical approved collaborators.',country)
    else:
        add('coady_f2','coady','Figure 2','Requester geography','not_estimable','Need fulfilled-study requesters plus reviewed institution IDs and coordinates. Organization names alone do not establish location.')
        add('paltoo_s1','paltoo','Supplementary Figure 1 analogue','Requester country distribution','not_estimable','Need source-supported institution/country crosswalk and investigator identities.')
    # ---- Coady Figure 3 and S2: risk sets include event-free studies; no positive-only filtering ----
    for endpoint,flag,key,location in [('publication','publication_history_complete','coady_f3','Figure 3'),('request','request_history_complete','coady_s2','Supplementary Figure 2 method')]:
        evs=pd.DataFrame();invalid_endpoint_studies=set(invalid_access_studies)
        if endpoint=='request' and len(usable_access):evs=usable_access[['study_id','_request_date']].rename(columns={'_request_date':'_event'}).copy()
        elif endpoint=='publication' and len(pu) and len(usable_access):
            validated=pu[pu.request_link_verified.map(ext_bool)].copy()
            # Join by both study and request: a PMID attached to some other study does not qualify.
            aa=usable_access[['request_id','study_id','_request_date']]
            _test=validated.merge(aa,on=['request_id','study_id'],how='left',validate='many_to_one',indicator=True)
            _bad=_test._merge.eq('left_only')|_test._pubdate.lt(_test._request_date)
            invalid_endpoint_studies.update(_test.loc[_bad,'study_id'])
            report['exclusions'].extend([{'module':'publication','study_id':r.study_id,'request_id':r.request_id,'reason':'Publication request link not found or publication predates request; study withheld from survival endpoint'} for r in _test.loc[_bad].itertuples()])
            evs=validated.merge(aa,on=['request_id','study_id'],how='inner',validate='many_to_one')
            evs=evs[evs._pubdate.ge(evs._request_date)][['study_id','_pubdate']].rename(columns={'_pubdate':'_event'})
        risk=[];km=[]
        if len(life):
            eligible_life=life[life[flag].map(ext_bool)&~life.study_id.isin(invalid_endpoint_studies)].copy()
            for s in eligible_life.to_dict('records'):
                dates=evs.loc[evs.study_id.eq(s['study_id']),'_event'] if len(evs) else pd.Series(dtype='datetime64[ns, UTC]')
                dates=dates[dates.ge(s['_release'])&dates.le(s['_end'])]
                observed=len(dates)>0
                end=dates.min() if observed else s['_end']
                risk.append({'study_id':s['study_id'],'study_type':s['study_type'],'years':(end-s['_release']).total_seconds()/(86400*365.25),'event':observed,
                             'release':s['_release'].isoformat(),'end':s['_end'].isoformat()})
            rf=pd.DataFrame(risk)
            if len(rf) and (endpoint!='publication' or len(pu)) and (endpoint!='request' or len(access)):
                for group,g in rf.groupby('study_type'):
                    for rr in rep_km(g.years,g.event):km.append({**rr,'study_type':group,'endpoint':endpoint})
                add(key,'coady',location,'Time from release to first '+endpoint,'method_matched_new_cohort',
                    'Verified availability dates and source-certified complete histories; event-free studies right-censored. Noninformative censoring is an assumption. Registry trial start is NOT data release. Publication endpoint requires an explicit, verified request-publication link.',km,
                    {'kind':'survival','records':km,'endpoint':endpoint,'risk_rows':risk})
                export_table('rep_'+key+'_risk_set',rf)
                horizon_rows=[]
                for group,g in rf.groupby('study_type'):
                    curve=rep_km(g.years,g.event);median=next((r['years'] for r in curve if r['survival']<=.5),None)
                    hrow={'study_type':group,'studies':len(g),'events':int(g.event.sum()),'median_years':median,'last_followup_years':float(g.years.max())}
                    for horizon in [1,3,5]:
                        previous=[r for r in curve if r['years']<=horizon]
                        hrow[f'event_probability_at_{horizon}_years']=(1-previous[-1]['survival']) if previous and g.years.max()>=horizon else None
                        hrow[f'at_risk_at_{horizon}_years']=int(g.years.ge(horizon).sum())
                    horizon_rows.append(hrow)
                report['tables'][key+'_summary']=safe(horizon_rows)
                export_table('rep_'+key+'_summary',pd.DataFrame(horizon_rows))
                report['logrank_'+endpoint]=rep_logrank_two_groups(rf)
            else:add(key,'coady',location,'Release-based '+endpoint+' analysis','not_estimable','No eligible complete-history population or required event source; do not interpret absent input as zero events.')
        else:add(key,'coady',location,'Release-based '+endpoint+' analysis','not_estimable',lmsg or 'No verified release/follow-up cohort.')
    # ---- Descriptive trial table and request-purpose table, aligned to original groupings ----
    if len(life):
        x=life[life.study_type.eq('clinical_trial')].copy();x['release_period']=x._release.dt.year.map(rep_period)
        counts=x.groupby('release_period').study_id.nunique().reset_index(name='trials')
        if 'n_participants' in x:
            x['_participants']=pd.to_numeric(x.n_participants,errors='coerce')
            stats=x.groupby('release_period')._participants.agg(['count','median','mean','std']).reset_index()
            counts=counts.merge(stats,on='release_period')
        add('coady_t2_current','coady','Table 2 companion','Trial portfolio by availability period','partial',
            'Clinical-trial studies, not NCT-linked roots. Participant count coverage shown; complete intervention/purpose distributions require supplied fields.',counts)
        if len(usable_access):
            q=usable_access[usable_access.study_type.eq('clinical_trial')].copy();q['first_request_period']=q.groupby('requester_id')._request_date.transform('min').dt.year.map(rep_period)
            rows=[]
            for period,g in q.groupby('first_request_period'):
                rows.append({'first_request_period':period,'investigators':g.requester_id.nunique(),'requests':g.request_id.nunique(),'institutions':g.institution_id.nunique()})
            add('coady_t1_current','coady','Table 1 companion','Investigators and requests','partial','Counts by first clinical-trial request period. Other original categories are not inferred from project titles.',rows)
            purposes=q[value(q,'purpose_validated').map(ext_bool)].groupby(['first_request_period','purpose_category']).request_id.nunique().reset_index(name='requests') if 'purpose_category' in q else pd.DataFrame()
            export_table('rep_coady_request_purposes',purposes)
    # ---- Supplemental topic and journal distributions: keep bibliometric sources distinct ----
    topics,tmsg=rep_valid(TABLES['rep_topics'],['pmid','topic','source_url','validated'])
    if len(topics) and len(pu):
        topics['pmid']=topics.pmid.map(pmidid);topics=topics[topics.pmid.isin(pu.pmid)].drop_duplicates(['pmid','topic'])
        tc=topics.groupby('topic').pmid.nunique().reset_index(name='papers');tc['percent_verified_corpus']=100*tc.papers/pu.pmid.nunique()
        add('paltoo_s2','paltoo','Supplementary Figure 2 analogue','Publication topics','method_matched_new_cohort','Only supplied reviewed publication topics in the verified secondary-use corpus; no assignment from dataset MeSH.',tc,
            {'kind':'category','records':tc.to_dict('records'),'categories':['topic'],'value':'papers'})
    else:add('paltoo_s2','paltoo','Supplementary Figure 2 analogue','Publication topics','not_estimable','Needs reviewed paper-level topics and verified secondary-use publications, not dataset categories.')
    jj=P[P.pmid.isin(p_ok)].copy() if 'pmid' in P else pd.DataFrame()
    if len(jj) and 'pubmed_journal' in jj:
        if 'citation_count' in I:jj=jj.merge(I[['pmid','citation_count']],on='pmid',how='left',validate='one_to_one')
        jc=jj.groupby('pubmed_journal').agg(papers=('pmid','nunique')).reset_index()
        if 'citation_count' in jj:
            cs=jj.groupby('pubmed_journal').citation_count.sum(min_count=1).rename('icite_citations').reset_index();jc=jc.merge(cs,on='pubmed_journal')
        add('paltoo_st1_proxy','paltoo','Supplementary Table 1 analogue','Journals and citations','descriptive_proxy',
            'Current linked-paper corpus and iCite citations; not the DAC-reported corpus or the historical Scopus snapshot. Journal impact factors are not substituted.',jc.sort_values('papers',ascending=False))
    metrics,mmsg=rep_valid(TABLES['rep_metrics'],['pmid','citation_percentile','percentile_direction','metric_system','metric_as_of','publication_date','cohort','source_url','validated'])
    if len(metrics):
        metrics['pmid']=metrics.pmid.map(pmidid);metrics=metrics[metrics.pmid.notna()].copy()
        metrics['_pubdate']=ext_date(metrics.publication_date);metrics['_metricdate']=ext_date(metrics.metric_as_of)
        age=(metrics._metricdate-metrics._pubdate).dt.total_seconds()/86400
        mm=metrics[metrics.metric_system.astype(str).str.casefold().eq('incites')&metrics.percentile_direction.eq('lower_is_more_cited')&age.ge(365.25)].copy()
        mm['citation_percentile']=pd.to_numeric(mm.citation_percentile,errors='coerce');mm=mm[mm.citation_percentile.between(0,100)]
        mm=mm.drop_duplicates(['pmid','cohort'])
        add('coady_s3','coady','Supplementary Figure 3 method','Citation percentile distributions','method_matched_new_cohort' if len(mm) else 'not_estimable',
            'InCites percentiles with at least one year citation exposure. Cohort labels explicitly supplied; NHLBI comparison sample must be independently sampled. RCR and NIH-percentile are not substituted.',mm,
            {'kind':'percentiles','records':mm[['pmid','cohort','citation_percentile']].to_dict('records')})
    else:add('coady_s3','coady','Supplementary Figure 3 method','Citation percentiles','not_estimable',mmsg or 'Need the specified citation-percentile system and reference corpus; iCite RCR is not InCites percentile.')
    # Original supplemental regression is not silently substituted with a different exposure denominator.
    add('coady_s1','coady','Supplementary Table 1 / Figure 1','Request-rate regression','not_implemented_pending_protocol',
        'Main text identifies a Poisson analysis, but the supplementary exposure/covariate specification was not available for inspection. A new count model would be an adaptation, not a faithful replication.')
    report['scope']='Executable new-cohort methods and expressly labelled proxies. Not a numerical reproduction of the original authors\' analyses. All current calculations require local records; original administrative microdata are not provided.'
    export_table('replication_crosswalk',pd.DataFrame(report['items']))
    export_table('replication_exclusions',pd.DataFrame(report['exclusions'],columns=['module','study_id','request_id','reason']))
    return report

if not REUSE:
    RESULTS['literature_replication']=build_literature_analyses()
    print('Published-study recipes:',dict(Counter(x['status'] for x in RESULTS['literature_replication']['items'])))


## Slide 8: Paper RCR and the top accessions

Two separate top-five lists: DAR records and linked PubMed papers. Each row shows both counts, the accession, and the study title; full details and links remain in CSV and speaker notes.


In [ ]:
# @title Slide 7 -- publication portfolios and unique-paper RCR
if not REUSE:
    scores=number(I,'rcr');good=scores.notna()&np.isfinite(scores)&scores.ge(0)
    # Require an explicit successful record when the source status is present.
    if 'icite_fetch_note' in I:good &= I.icite_fetch_note.map(status).eq('ok')
    scored=I.loc[good].copy();scored['rcr']=scores[good]
    x=scored.rcr
    edges=[0,0.5,1,2,5,10,20,50,100,np.inf]
    h,_=np.histogram(x,edges)
    RESULTS['rcr']={'n':len(x),'missing':len(set(L.pmid))-len(x),'median':float(x.median()) if len(x) else None,
        'q25':float(x.quantile(.25)) if len(x) else None,'q75':float(x.quantile(.75)) if len(x) else None,
        'zero':int(x.eq(0).sum()),'labels':['0-<0.5','0.5-<1','1-<2','2-<5','5-<10','10-<20','20-<50','50-<100','100+'],
        'counts':h.tolist(),'top10_weighted_share':float(x.nlargest(max(1,math.ceil(.1*len(x)))).sum()/x.sum()) if len(x) and x.sum()>0 else None}
    # Requested repeated papers-per-accession panel uses the identical distribution as slide 3.
    export_table('rcr_histogram',pd.DataFrame({'rcr_bin':RESULTS['rcr']['labels'],'papers':h}))
    check('RCR bins conserve scored papers',int(h.sum())==len(x))
# Top accessions by recorded requests and linked PubMed papers: observed counts, not impact ranks.
def study_title(row):
    for key in ['study_name','study_title','title']:
        t=text(row.get(key))
        if t:return t
    titles=_list_value(row.get('titles_seen'))
    if titles:
        labels=[text(v) for v in titles if text(v)]
        if labels:return ' / '.join(sorted(set(labels)))
    return 'Title unavailable'


def top_accessions(roots,signals,links,icite,trials):
    f=signals.copy();rmap=roots.set_index('root_accession').to_dict('index')
    f['study_title']=f.root_accession.map(lambda r:study_title(rmap[r]))
    f['latest_accession']=f.root_accession.map(lambda r:text(rmap[r].get('latest_accession')) or r)
    counts=links.groupby('root_accession').pmid.nunique() if len(links) else pd.Series(dtype='float64')
    f['pubmed_papers']=f.root_accession.map(counts)
    f.loc[f.publication.eq(False).fillna(False),'pubmed_papers']=0
    f['pubmed_count_complete']=f.paper_count.notna()
    f['pubmed_count_lower_bound']=f.pubmed_papers.notna()&~f.pubmed_count_complete
    f['dar_records']=f.dar_count
    if 'nct_id' in trials:
        nct=trials.groupby('root_accession').nct_id.nunique();f['linked_trials']=f.root_accession.map(nct)
        f.loc[f.nct.eq(False).fillna(False),'linked_trials']=0
    else:f['linked_trials']=np.nan
    if {'pmid','rcr'}.issubset(icite) and len(links):
        x=icite.copy();x['rcr']=pd.to_numeric(x.rcr,errors='coerce')
        good=x.rcr.notna()&np.isfinite(x.rcr)&x.rcr.ge(0)
        if 'icite_fetch_note' in x:good &= x.icite_fetch_note.map(status).eq('ok')
        p=links[['root_accession','pmid']].drop_duplicates().merge(x.loc[good,['pmid','rcr']],on='pmid',how='left',validate='many_to_one')
        stats=p.groupby('root_accession').rcr.agg(['median','count'])
        f['median_rcr']=f.root_accession.map(stats['median']);f['scored_pmids']=f.root_accession.map(stats['count'])
    else:f['median_rcr']=np.nan;f['scored_pmids']=np.nan
    f['study_url']=f.latest_accession.map(lambda a:'https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id='+a)
    cols=['root_accession','latest_accession','study_title','dar_records','pubmed_papers','pubmed_count_complete','pubmed_count_lower_bound',
          'median_rcr','scored_pmids','linked_trials','BDC','study_url']
    f=f[cols]
    result={'groups':[],'note':'Top five observed accession counts in each ranking. Related accessions and shared papers can recur. Counts are not family-deduplicated; no manual top-study selection.',
            'pubmed_definition':'Distinct successfully retrieved linked PMIDs. A lower-bound marker means unresolved linked PMIDs remain.',
            'dar_definition':'Recorded accession-requester rows, consistent with the overview; not verified distinct scientific projects.'}
    for metric,title in [('dar_records','Top five by DAR records'),('pubmed_papers','Top five by linked PubMed papers')]:
        ranked=f[f[metric].notna() & f[metric].gt(0)].sort_values([metric,'root_accession'],ascending=[False,True])
        top=ranked.head(5)
        tie_n=int(ranked[metric].eq(top[metric].iloc[-1]).sum()) if len(top)==5 else 0
        result['groups'].append({'metric':metric,'title':title,'rows':top.to_dict('records'),
            'eligible_positive_roots':len(ranked),'roots_tied_at_fifth':tie_n,
            'tie_rule':'Ascending root accession within equal counts; all ranks exported.'})
    return result,f

if not REUSE:
    leaders,leader_rows=top_accessions(R,S,PL,I,T)
    RESULTS['top_accessions']=leaders
    export_table('accession_activity_details',leader_rows)
    for group in leaders['groups']:
        export_table('top5_'+group['metric'],pd.DataFrame(group['rows'],columns=list(leader_rows)))
    print('Top-accession details prepared for slide 7; full untruncated labels saved to CSV.')


## Slide 9: same-root Sankey

In [ ]:
# @title Slide 8 -- same-root alluvial counts, not a sequence of scientific conversion
if not REUSE:
    stages=[('accession','All accessions'),('sstr','SSTR'),('dar','DAR'),('publication','PubMed'),('nct','NCT link')]
    nodes=[]; links=[]; pathcounts=Counter()
    state=lambda x:'Unknown' if pd.isna(x) else ('Yes' if bool(x) else 'No')
    for row in S.to_dict('records'):
        path=('All',)+tuple(state(row[k]) for k,_ in stages[1:]);pathcounts[path]+=1
    totals={}
    for j,(key,title) in enumerate(stages):
        for st in (['All'] if j==0 else ['Yes','No','Unknown']):
            n=sum(v for path,v in pathcounts.items() if path[j]==st)
            node=f'{j}:{st}';nodes.append({'id':node,'stage':j,'stage_label':title,'state':st,'n':n});totals[node]=n
    for j in range(len(stages)-1):
        pairs=Counter()
        for path,n in pathcounts.items():pairs[(path[j],path[j+1])]+=n
        for (a,b),n in pairs.items():links.append({'source':f'{j}:{a}','target':f'{j+1}:{b}','n':n})
    for j in range(1,5):check('Sankey stage '+str(j)+' conserves roots',sum(v['n'] for v in nodes if v['stage']==j)==N)
    for node in nodes:
        if node['stage']>0:check('Sankey inflow '+node['id'],sum(l['n'] for l in links if l['target']==node['id'])==node['n'])
        if node['stage']<4:check('Sankey outflow '+node['id'],sum(l['n'] for l in links if l['source']==node['id'])==node['n'])
    RESULTS['sankey']={'nodes':nodes,'links':links,'n':N,'unit':'root accessions at every stage; not SSTR/DAR/publication/trial event totals'}
    export_table('sankey_nodes',pd.DataFrame(nodes));export_table('sankey_links',pd.DataFrame(links))
    export_table('sankey_full_paths',pd.DataFrame([dict(zip([x[0] for x in stages],path),roots=n) for path,n in pathcounts.items()]))


## Slide 10: Venn intersections

In [ ]:
# @title Slide 9 -- exact SRA / BDC / AnVIL intersections inside dbGaP
if not REUSE:
    keys=['SRA','BDC','AnVIL'];known=S[keys].notna().all(axis=1);counts={''.join(map(str,b)):0 for b in itertools.product([0,1],repeat=3)}
    for row in S.loc[known,keys].itertuples(index=False,name=None):counts[''.join(str(int(x)) for x in row)]+=1
    margins={k:{'positive':int(S[k].fillna(False).sum()),'known':int(S[k].notna().sum()),'unknown':int(S[k].isna().sum())} for k in keys}
    RESULTS['venn']={'names':keys,'counts':counts,'known':int(known.sum()),'unknown':int((~known).sum()),'universe':N,'marginals':margins}
    check('Venn regions sum to known membership',sum(counts.values())==int(known.sum()))
    export_table('repository_regions',pd.DataFrame([{'SRA':int(k[0]),'BDC':int(k[1]),'AnVIL':int(k[2]),'roots':v} for k,v in counts.items()]))


## Slide 11: BDC comparison

In [ ]:
# @title Slide 10 -- BDC-listed vs not listed; not a complete-copy or platform-effect claim
if not REUSE:
    rows=[]
    for key,label in [('dar','1+ observed DARs'),('publication','1+ PubMed publications'),('nct','1+ NCT linkage'),
                      ('sstr','1+ SSTR reports'),('mesh','MeSH terms'),('nih_ack','NIH acknowledgement'),('SRA','SRA availability')]:
        known=S.BDC.notna()&S[key].notna();z=S.loc[known]
        a=z.loc[z.BDC.astype(bool),key];b=z.loc[~z.BDC.astype(bool),key]
        p1=float(a.mean()) if len(a) else None;p0=float(b.mean()) if len(b) else None
        rows.append({'key':key,'label':label,'bdc_positive':int(a.sum()),'bdc_n':len(a),
            'other_positive':int(b.sum()),'other_n':len(b),'bdc_pct':100*p1 if p1 is not None else None,
            'other_pct':100*p0 if p0 is not None else None,'difference_pp':100*(p1-p0) if p1 is not None and p0 is not None else None})
    RESULTS['bdc']={'bdc_n':int(S.BDC.fillna(False).sum()),'other_n':int((~S.BDC.dropna()).sum()),'unknown':int(S.BDC.isna().sum()),
        'rows':rows,'label':'explicit BDC listing','caveat':'Listing can cover selected versions, consent groups or files; comparisons are descriptive.'}
    export_table('bdc_comparison',pd.DataFrame(rows));display(pd.DataFrame(rows))


## Slides 12-13: methods, sources, discussion and validation


In [ ]:
# @title Slides 11-12 -- sources, methods, consistent figures and export checks
REFERENCES=[
 {'key':'paltoo','label':'Paltoo et al. (2014), Nature Genetics','doi':'10.1038/ng.3062',
  'url':'https://pmc.ncbi.nlm.nih.gov/articles/PMC4182942/',
  'finding':'2013 snapshot: 304 studies; 2,221 investigators; 924 DAC-reported secondary-use papers.',
  'boundary':'DAC-reported reuse is not the same as every dbGaP-linked paper.','reviewed':'2026-10-07'},
 {'key':'cabili','label':'Cabili et al. (2021), Cell Genomics','doi':'10.1016/j.xgen.2021.100031',
  'url':'https://pmc.ncbi.nlm.nih.gov/articles/PMC9903839/',
  'finding':'2006-2019 analysis: 1,250 unique released datasets; 1,745 including updated versions/rereleases.',
  'boundary':'Requesting PIs, renewals, roots and releases are different units.','reviewed':'2026-10-07'},
 {'key':'coady','label':'Coady et al. (2017), NEJM','doi':'10.1056/NEJMsa1603542',
  'url':'https://www.nejm.org/doi/full/10.1056/NEJMsa1603542',
  'finding':'NHLBI repository: median 235 days to first request; requests and subsequent publications linked explicitly.',
  'boundary':'A median cannot estimate demand after three silent years.','reviewed':'2026-10-07'},
 {'key':'sstr','label':'NCBI dbGaP Submission Guide','url':'https://www.ncbi.nlm.nih.gov/gap/docs/submissionguide/',
  'finding':'SSTR describes subjects, samples, consent and processing; public study attribution differs from submission-system identities.',
  'boundary':'A successful summary is not an access event or proof of complete deposition.','reviewed':'2026-10-07'},
 {'key':'icite','label':'NIH iCite: RCR','url':'https://support.icite.nih.gov/hc/en-us/articles/9105587565851-Relative-Citation-Ratio-RCR',
  'finding':'RCR is a normalized citation-rate measure; 1 is the NIH reference median.',
  'boundary':'Not a measure of patient benefit or the effect of sharing.','reviewed':'2026-10-07'},
 {'key':'bdc','label':'NHLBI BDC: access requirements','url':'https://bdcatalyst.gitbook.io/biodata-catalyst-documentation/written-documentation/data-access/understanding-access-requirements-for-biodata-catalyst',
  'finding':'Controlled BDC-hosted data require dbGaP authorization.',
  'boundary':'Cloud location does not itself explain missing approved-requester records.','reviewed':'2026-10-07'}]

REFERENCES.extend([
 {'key':'version_definitions','label':'NCBI dbGaP study versions','url':'https://www.ncbi.nlm.nih.gov/gap/docs/submissionguide/',
  'finding':'Data version v# and participant set p# have distinct update rules.',
  'boundary':'Observed labels are not a complete version history; p# is not the number of subjects.','reviewed':'2026-10-07'},
 {'key':'sstr_schema','label':'NCBI SSTR public JSON schema','url':'https://www.ncbi.nlm.nih.gov/gap/sstr/schema/dbgap_study.v1.schema.json',
  'finding':'Consent groups carry code, name and short_name; subject/sample loaded counts and hierarchy are explicit fields.',
  'boundary':'Numeric consent codes are study-local. Sample counts are not SSTR report counts.','reviewed':'2026-10-07'},
 {'key':'glm','label':'statsmodels Binomial GLM fit','url':'https://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLM.fit.html',
  'finding':'Logistic generalized linear model with selectable covariance estimator.',
  'boundary':'Model-based intervals do not eliminate confounding or complete-case selection.','reviewed':'2026-10-07'}])

def draw_hist(data,path,title,xlabel,color):
    fig,ax=plt.subplots(figsize=(7.2,3.6))
    if data.get('n',0):
        counts=data['counts'];ax.bar(range(len(counts)),counts,color=color)
        ax.set_xticks(range(len(counts)));ax.set_xticklabels(data['labels'],rotation=30,ha='right')
        for i,n in enumerate(counts):
            if n:ax.text(i,n+max(counts)*.018,f'{n:,}',ha='center',fontsize=10)
        ax.set_ylim(0,max(counts)*1.20 if max(counts)>0 else 1)
    else:
        ax.text(.5,.5,'Not measured from the available inputs',ha='center',va='center',transform=ax.transAxes,wrap=True);ax.set_xticks([])
    ax.set_title(title,loc='left',weight='bold');ax.set_xlabel(xlabel);ax.set_ylabel('Papers' if path.startswith('rcr') else 'Accessions')
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'));fig.tight_layout()
    return savefig(fig,path)
def savefig(fig,name):
    files={}
    for ext in ['png','svg']:
        p=OUT/(name+'.'+ext);fig.savefig(p,bbox_inches='tight',facecolor='white');OUTPUTS[p.name]=hashfile(p);files[ext]=p.name
    if SHOW_FIGURES:display(fig)
    plt.close(fig);return files

def draw_growth(data,name,kind):
    fig,ax=plt.subplots(figsize=(5.0,2.6))
    f=pd.DataFrame(data['rows'])
    if data.get('n',0):
        heights=pd.to_numeric(f[kind],errors='coerce')
        colors=[PALETTE['grey'] if y==GROWTH_END_YEAR else PALETTE['blue' if kind=='annual' else 'teal'] for y in f.year]
        bars=ax.bar(f.year,heights,color=colors,width=.82)
        if len(bars):bars[-1].set_hatch('//')
        ax.set_ylim(bottom=0)
    else:ax.text(.5,.5,'No usable dates',transform=ax.transAxes,ha='center',va='center',fontsize=12)
    ax.set_xlim(GROWTH_START_YEAR-.6,GROWTH_END_YEAR+.6)
    ax.set_xticks([2006,2010,2014,2018,2022,2026]);ax.tick_params(axis='x',labelsize=10)
    ax.set_ylabel('Annual count' if kind=='annual' else 'Cumulative count',fontsize=11)
    ax.set_xlabel('Calendar year',fontsize=10);ax.yaxis.set_major_locator(MaxNLocator(nbins=4,integer=True))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True);fig.tight_layout(pad=.7)
    return savefig(fig,'growth_'+name+'_'+kind)


def draw_profile_hist(data,key):
    # Separate figure for each requested distribution, reused in HTML and PPTX.
    fig,ax=plt.subplots(figsize=(3.8,2.2))
    if data['n']:
        vals=data['counts'];ax.bar(range(len(vals)),vals,width=.8,color=PALETTE['teal'])
        ax.set_xticks(range(len(vals)));ax.set_xticklabels(data['labels'],rotation=33,ha='right',fontsize=9)
        ax.set_ylim(bottom=0)
    else:
        msg='No explicit trait count' if key=='traits' else ('No explicit c# labels' if key=='consent_versions' else 'No usable saved field')
        ax.text(.5,.5,msg,ha='center',va='center',transform=ax.transAxes,fontsize=11)
        ax.set_xticks([])
    ax.set_title(data['title'],loc='left',fontsize=13,fontweight='bold',pad=4)
    ax.set_ylabel('Accessions',fontsize=9);ax.yaxis.set_major_locator(MaxNLocator(nbins=3,integer=True))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'));ax.tick_params(axis='y',labelsize=9)
    fig.text(.01,.01,f"Known {data['n']:,} | missing {data['missing']:,}"+(f" | invalid {data['invalid']:,}" if data['invalid'] else ''),fontsize=9)
    fig.tight_layout(rect=(0,.07,1,1),pad=.5)
    return savefig(fig,'profile_'+key)


def draw_consent_labels(data):
    fig,ax=plt.subplots(figsize=(7.2,3.1))
    rows=data['display_rows']
    if rows:
        vals=[r['accessions'] for r in rows]
        ax.barh(range(len(rows)),vals,color=PALETTE['teal'])
        labels=[r['label'] if len(r['label'])<=38 else r['label'][:35]+'...' for r in rows]
        ax.set_yticks(range(len(rows)));ax.set_yticklabels(labels,fontsize=10)
        ax.invert_yaxis();ax.set_xlim(0,max(vals)*1.20)
        for j,r in enumerate(rows):ax.text(r['accessions']+max(vals)*.02,j,f"{r['accessions']:,}",va='center',fontsize=10)
    else:
        ax.text(.5,.5,'No named consent categories in saved metadata',ha='center',va='center',transform=ax.transAxes,fontsize=11)
        ax.set_yticks([])
    ax.set_title('Named consent categories',loc='left',fontsize=13,fontweight='bold')
    ax.set_xlabel('Accessions with this label (categories may overlap)',fontsize=10)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4,integer=True));ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    fig.text(.01,.012,f"Labelled {data['labelled_roots']:,} roots | unlabelled {data['unlabelled_roots']:,}; no global numeric-code mapping",fontsize=9)
    fig.tight_layout(rect=(0,.065,1,1),pad=.5)
    return savefig(fig,'profile_consent_labels')


def draw_version_matrix(data):
    from matplotlib.colors import LogNorm
    fig,ax=plt.subplots(figsize=(7.4,3.1))
    rows=data['matrix']
    if rows:
        xs=sorted({r['data_version'] for r in rows});ys=sorted({r['participant_set'] for r in rows})
        a=np.zeros((len(ys),len(xs)),dtype=int)
        for row in rows:a[ys.index(row['participant_set']),xs.index(row['data_version'])]=row['accessions']
        img=ax.imshow(np.ma.masked_equal(a,0),origin='lower',aspect='auto',cmap='Blues',norm=LogNorm(vmin=1,vmax=max(2,int(a.max()))))
        ax.set_facecolor('#F1F4F6')
        ax.set_xticks(range(len(xs)));ax.set_xticklabels(['v'+str(x) for x in xs],rotation=45 if len(xs)>15 else 0,fontsize=max(6,min(10,190/len(xs))))
        ax.set_yticks(range(len(ys)));ax.set_yticklabels(['p'+str(y) for y in ys],fontsize=max(7,min(10,80/len(ys))))
        for y,x in zip(*np.nonzero(a)):
            ax.text(x,y,f'{a[y,x]:,}',ha='center',va='center',fontsize=max(5,min(9,160/len(xs),80/len(ys))),
                    color='white' if img.norm(a[y,x])>.60 else PALETTE['text'])
        cb=fig.colorbar(img,ax=ax,pad=.02,fraction=.04);cb.set_label('Roots (log color)',fontsize=9);cb.ax.tick_params(labelsize=8)
        ax.set_xlabel('Highest observed data-version suffix',fontsize=10);ax.set_ylabel('Associated participant set',fontsize=10)
    else:
        ax.text(.5,.5,'No usable v/p pairs in saved fields',ha='center',va='center',transform=ax.transAxes,fontsize=11)
        ax.set_xticks([]);ax.set_yticks([])
    ax.set_title('Data version x participant set',loc='left',fontsize=13,fontweight='bold')
    share=f"{data['v1_p1_percent']:.1f}%" if data.get('v1_p1_percent') is not None else 'not measured'
    fig.text(.01,.012,f"Known {data['known_roots']:,} | v1.p1 {share} | other pairs {data['other_pairs']:,}; one pair per root",fontsize=9)
    fig.tight_layout(rect=(0,.065,1,1),pad=.5)
    return savefig(fig,'profile_version_matrix')

def draw_comparison(model):
    fig,ax=plt.subplots(figsize=(8.0,4.4))
    if model['status']=='fitted':
        # Keep substantive terms in configured order, NOT p-value order. Full model in CSV.
        rows=[r for r in model['coefficient_rows'] if not r['term'].startswith('admin_ic=')][:8]
        if not rows:rows=model['coefficient_rows'][:8]
        for y,r in enumerate(rows):
            ax.errorbar(r['odds_ratio'],y,xerr=[[r['odds_ratio']-r['ci_low']],[r['ci_high']-r['odds_ratio']]],fmt='o',capsize=3,color=PALETTE['blue'])
        ax.set_yticks(range(len(rows)));ax.set_yticklabels([r['label'] for r in rows],fontsize=11)
        ax.invert_yaxis();ax.set_xscale('log');ax.axvline(1,color=PALETTE['grey'],ls='--',lw=1)
        ax.set_xlabel('Adjusted odds ratio (95% CI; log scale)',fontsize=11)
        ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)
    else:
        ax.set_axis_off();ax.text(.03,.80,'Model not estimated',fontsize=17,weight='bold',transform=ax.transAxes)
        ax.text(.03,.66,'\n'.join(textwrap.wrap(model.get('reason','Inputs unavailable'),65)),fontsize=13,va='top',transform=ax.transAxes)
        ax.text(.03,.26,'The descriptive comparison and exclusions are still exported.\nNo estimates are copied from an earlier model.',fontsize=11,va='top',transform=ax.transAxes)
    fig.tight_layout(pad=1)
    return savefig(fig,'both_neither_adjusted')

def draw_semantic_bars(rows,labelkey,title,filename):
    fig,ax=plt.subplots(figsize=(7.1,3.25));show=rows[:7]
    if show:
        y=np.arange(len(show));vals=[r['roots'] for r in show]
        ax.barh(y,vals,color=PALETTE['teal']);ax.set_yticks(y)
        ax.set_yticklabels(['\n'.join(textwrap.wrap(r[labelkey],31)) for r in show],fontsize=10)
        ax.invert_yaxis();ax.set_xlim(0,max(vals)*1.22)
        for k,v in enumerate(vals):ax.text(v+max(vals)*.015,k,f'{v:,}',va='center',fontsize=10)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True,nbins=4));ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
        ax.set_xlabel('Distinct root accessions (categories overlap)',fontsize=10)
    else:
        ax.text(.5,.5,'Not measured: run A semantic collection',ha='center',va='center',transform=ax.transAxes,fontsize=12);ax.set_axis_off()
    ax.set_title('\n'.join(textwrap.wrap(title,54)),loc='left',weight='bold',fontsize=12)
    fig.tight_layout(pad=.6)
    return savefig(fig,filename)

def draw_semantic_matrix(data):
    rows=data['cross'];themes=list(dict.fromkeys(r['theme'] for r in rows));families=list(dict.fromkeys(r['assay_family'] for r in rows))
    fig,ax=plt.subplots(figsize=(9.4,4.0))
    if rows:
        a=np.zeros((len(themes),len(families)),dtype=int)
        for row in rows:a[themes.index(row['theme']),families.index(row['assay_family'])]=row['roots']
        im=ax.imshow(a,aspect='auto',cmap='Blues')
        ax.set_yticks(range(len(themes)));ax.set_yticklabels(['\n'.join(textwrap.wrap(x,28)) for x in themes],fontsize=9)
        ax.set_xticks(range(len(families)));ax.set_xticklabels(['\n'.join(textwrap.wrap(x,18)) for x in families],fontsize=8,rotation=30,ha='right')
        for y,x in zip(*np.nonzero(a)):ax.text(x,y,str(a[y,x]),ha='center',va='center',fontsize=9,color='white' if a[y,x]>a.max()*.55 else PALETTE['text'])
        fig.colorbar(im,ax=ax,label='Distinct roots',fraction=.035,pad=.02)
    else:ax.set_axis_off();ax.text(.5,.5,'No joint resolved disease and assay evidence',ha='center',transform=ax.transAxes)
    ax.set_title('Recorded disease themes x assay families',loc='left',fontsize=14,weight='bold')
    fig.tight_layout(pad=.6);return savefig(fig,'semantic_theme_assay_matrix')

# Consistent small-multiple figures; each chart is separately exported for inspection.
def ext_save(fig,name):
    fig.tight_layout(pad=.6)
    return savefig(fig,name)

def ext_annual_plot(data,key):
    fig,ax=plt.subplots(figsize=(7.0,1.95));rows=data['rows']
    measured=[r for r in rows if r['unique_entities'] is not None]
    if measured:
        ax.bar([r['year'] for r in measured],[r['unique_entities'] for r in measured],width=.78,color=PALETTE['teal'] if key.startswith('submit') else PALETTE['blue'])
        ax.set_xlim(2005.4,2026.6);ax.set_xticks([2006,2010,2014,2018,2022,2026]);ax.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=4));ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
        ax.set_ylabel('Unique entities / year',fontsize=10);ax.set_xlabel('Discovery-cohort year (proxy)' if data['basis']=='current_attribution_by_discovery_cohort_proxy' else 'Submission year' if data['basis']=='verified_submission_events' else 'Observed initial-approval year',fontsize=10)
    else:
        ax.text(.5,.5,'No usable identities and dates\nMissing is not zero',ha='center',va='center',transform=ax.transAxes,fontsize=12);ax.set_axis_off()
    ax.set_title('Unique PIs / groups by year' if key.endswith('people') else 'Unique institutions by year',loc='left',weight='bold',fontsize=12)
    ax.tick_params(labelsize=9)
    return ext_save(fig,'annual_'+key)


def participant_rank_plot(data,role):
    """All observed PIs/groups, descending volume. Not a top-ten plot."""
    contributor=role=='contributors';key='contributed_accessions' if contributor else 'dar_root_wlid_pairs'
    vals=np.asarray([r[key] for r in data.get('rows',[])],dtype=float)
    fig,ax=plt.subplots(figsize=(7.0,1.95))
    color=PALETTE['teal' if contributor else 'blue']
    title='Contributed accessions per PI / group' if contributor else 'DAR requests per PI / group'
    if len(vals):
        if (vals<=0).any() or (np.diff(vals)>0).any():raise ValueError('Ranked volumes must be positive and nonincreasing')
        ranks=np.arange(1,len(vals)+1)
        # Rasterize bars inside SVG only for dense profiles; underlying exact rows remain CSV/JSON.
        ax.bar(ranks,vals,width=1.0,color=color,linewidth=0,rasterized=len(vals)>1500)
        islog=vals.max()/vals.min()>=100
        if islog:
            ax.set_yscale('log');ax.set_ylim(.8,vals.max()*1.45)
            from matplotlib.ticker import LogLocator,NullFormatter
            ax.yaxis.set_major_locator(LogLocator(base=10,numticks=4));ax.yaxis.set_minor_formatter(NullFormatter())
        else:
            ax.set_ylim(0,max(vals)*1.20);ax.yaxis.set_major_locator(MaxNLocator(nbins=4,integer=True))
        ax.set_xlim(.5,len(vals)+.5 if len(vals)>1 else 1.5)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4,integer=True));ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
        ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
        ylabel='Accessions' if contributor else 'Root-WLID pairs'
        ax.set_ylabel(ylabel+(' (log)' if islog else ''),fontsize=9)
        ax.set_xlabel('PI / group rank (highest volume first)',fontsize=9)
        ax.tick_params(labelsize=9)
        ax.text(.99,.91,f"{len(vals):,} PI/group labels",ha='right',va='top',transform=ax.transAxes,fontsize=9,color=PALETTE['text'])
    else:
        ax.set_axis_off();ax.text(.5,.48,'No PI / group volume records available',ha='center',transform=ax.transAxes,fontsize=11)
    ax.set_title(title,loc='left',weight='bold',fontsize=12)
    fig.tight_layout(pad=.45)
    return savefig(fig,'rank_contributors_pi' if contributor else 'rank_accessors_pi')

def ext_volume_plot(data,key,rank=False):
    fig,ax=plt.subplots(figsize=(9.0,2.45))
    if data['n']:
        if rank:
            ax.bar([r['entity_rank'] for r in data['rows']],[r['dar_root_wlid_pairs'] for r in data['rows']],width=1)
            ax.set_xlabel('Unique '+data['label'].lower()+' ordered by volume',fontsize=10);ax.set_ylabel('Recorded DAR pairs',fontsize=10)
            if max(r['dar_root_wlid_pairs'] for r in data['rows'])/min(r['dar_root_wlid_pairs'] for r in data['rows'])>=100:ax.set_yscale('log');ax.set_ylabel('Recorded DAR pairs (log)',fontsize=10)
            ax.xaxis.set_major_locator(MaxNLocator(integer=True,nbins=5))
        else:
            ax.bar(range(len(data['labels'])),data['counts'],color=PALETTE['blue'])
            ax.set_xticks(range(len(data['labels'])));ax.set_xticklabels(data['labels'],rotation=30,ha='right',fontsize=8)
            ax.set_xlabel('Recorded DAR root-WLID pairs per entity',fontsize=10);ax.set_ylabel('Unique '+data['label'].lower(),fontsize=10)
            ax.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=4));ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    else:ax.text(.5,.5,'No observed entities',ha='center',va='center',transform=ax.transAxes);ax.set_axis_off()
    ax.set_title('DAR volume: '+data['label'],loc='left',fontsize=13,weight='bold')
    return ext_save(fig,('rank_volume_' if rank else 'hist_volume_')+key)

def ext_type_charts(data):
    result={};show=data['type_counts'][:8]
    fig,ax=plt.subplots(figsize=(6.7,3.25))
    if show:
        yy=np.arange(len(show));vv=[r['roots'] for r in show];ax.barh(yy,vv,color=PALETTE['teal']);ax.set_yticks(yy)
        ax.set_yticklabels(['\n'.join(textwrap.wrap(r['data_type'],25)) for r in show],fontsize=9);ax.invert_yaxis();ax.set_xlim(0,max(vv)*1.22)
        for y,v in zip(yy,vv):ax.text(v+max(vv)*.02,y,f'{v:,}',va='center',fontsize=9)
        ax.set_xlabel('Distinct accessions (categories overlap)',fontsize=10)
    else:ax.text(.5,.5,'No harmonized data-type observations',ha='center',transform=ax.transAxes);ax.set_axis_off()
    ax.set_title('Current reported data families',loc='left',weight='bold',fontsize=13);result['type_current']=ext_save(fig,'type_current')
    cats=data['categories'][:8];rows=pd.DataFrame(data['rows']);years=list(range(2006,2027))
    for measure,key,title in [('annual_roots','type_annual','Annual root-type observations'),('percent_of_annotated_cohort','type_share','Share of annotated cohort (%)')]:
        fig,ax=plt.subplots(figsize=(7.0,3.75))
        if cats and len(rows) and data['dated_roots']:
            aa=np.full((len(cats),len(years)),np.nan)
            for ri,cat in enumerate(cats):
                rr=rows[rows.data_type.eq(cat)].set_index('year')
                for ci,y in enumerate(years):
                    if y in rr.index:aa[ri,ci]=float(rr.loc[y,measure]) if pd.notna(rr.loc[y,measure]) else np.nan
            im=ax.imshow(np.ma.masked_invalid(aa),aspect='auto',cmap='Blues')
            ax.set_xticks([years.index(y) for y in [2006,2010,2014,2018,2022,2026]]);ax.set_xticklabels([2006,2010,2014,2018,2022,2026],fontsize=9)
            ax.set_yticks(range(len(cats)));ax.set_yticklabels(['\n'.join(textwrap.wrap(c,24)) for c in cats],fontsize=8)
            fig.colorbar(im,ax=ax,fraction=.03,pad=.025)
            ax.set_xlabel('Submission year' if data['basis'].startswith('verified') else 'Accession discovery cohort (proxy)',fontsize=10)
        else:ax.text(.5,.5,'No dated root-type observations',ha='center',transform=ax.transAxes);ax.set_axis_off()
        ax.set_title(title,loc='left',weight='bold',fontsize=13);result[key]=ext_save(fig,key)
    fig,ax=plt.subplots(figsize=(6.7,3.25));h=data['types_per_root']
    if h.get('n'):
        ax.bar(range(len(h['labels'])),h['counts'],color=PALETTE['blue']);ax.set_xticks(range(len(h['labels'])));ax.set_xticklabels(h['labels']);ax.set_ylabel('Accessions',fontsize=10);ax.set_xlabel('Number of harmonized families per annotated root',fontsize=10)
    else:ax.text(.5,.5,'No typed roots',ha='center',transform=ax.transAxes);ax.set_axis_off()
    ax.set_title('Modality breadth per accession',loc='left',weight='bold',fontsize=13);result['type_breadth']=ext_save(fig,'type_breadth')
    return result


def rep_plot(plot,key,title):
    kind=plot['kind'];df=pd.DataFrame(plot['records']);fig,ax=plt.subplots(figsize=(9.0,4.3));ax.set_title(title,loc='left',fontsize=14,weight='bold')
    if df.empty:ax.text(.5,.5,'No eligible observations',ha='center',va='center',transform=ax.transAxes);ax.set_axis_off()
    elif kind=='time':
        for c in plot['series']:ax.step(df.year,df[c],where='post',label=c)
        ax.set_xlabel('Calendar year');ax.set_ylabel('Unique publications');ax.legend(frameon=False)
    elif kind=='category':
        cats=plot['categories'];val=plot['value'];df=df.sort_values(val,ascending=False)
        # Full counts remain in CSV; report graphic shows up to 20 groups with explicit title note.
        df=df.head(20);labels=[' | '.join(str(r[c]) for c in cats) for r in df.to_dict('records')]
        ax.barh(np.arange(len(df)),df[val]);ax.set_yticks(np.arange(len(df)));ax.set_yticklabels(['\n'.join(textwrap.wrap(x,54)) for x in labels],fontsize=8);ax.invert_yaxis();ax.set_xlabel(val.replace('_',' '))
        if len(plot['records'])>20:ax.set_title(title+' (20 largest categories)',loc='left',fontsize=13)
    elif kind=='access':
        for c,label in [('submitted_requests','Submitted consent-group requests'),('approved_requests','Approved'),('downloaded_request_units','Downloaded request units')]:
            if c in df and df[c].notna().any():ax.plot(df.year,df[c],marker='o',label=label)
        ax.set_xlabel('Request-cohort year');ax.set_ylabel('Request units');ax.legend(frameon=False,fontsize=9)
        if df.mean_processing_days.notna().any():
            second=ax.twinx();second.plot(df.year,df.mean_processing_days,'--',color=PALETTE['amber']);second.set_ylabel('Mean request-to-decision days',color=PALETTE['amber'])
    elif kind=='coady_availability':
        for typ,g in df.groupby('study_type'):
            ax.step(g.year,g.available_studies,where='post',ls='--',label=typ+' available')
            if plot.get('deliveries_measured'):ax.step(g.year,g.cumulative_study_requester_deliveries,where='post',label=typ+' deliveries')
        ax.set_xlabel('Calendar year');ax.set_ylabel('Cumulative studies / study-requester deliveries');ax.legend(frameon=False,fontsize=9)
    elif kind=='geo':
        for typ,g in df.groupby('cohort'):ax.scatter(g.longitude,g.latitude,s=18+15*np.sqrt(g.investigators),alpha=.6,label=typ)
        ax.set_xlim(-180,180);ax.set_ylim(-60,85);ax.grid(alpha=.25);ax.set_xlabel('Longitude');ax.set_ylabel('Latitude');ax.legend(frameon=False,fontsize=9)
        ax.set_title(title+' | coordinate view; no geocoding performed',loc='left',fontsize=13)
    elif kind=='survival':
        for typ,g in df.groupby('study_type'):
            g=g.sort_values('years');ax.step(g.years,g.survival,where='post',label=typ)
            ax.fill_between(g.years.to_numpy(float),g.survival_low.to_numpy(float),g.survival_high.to_numpy(float),step='post',alpha=.12)
        ax.set_ylim(0,1.03);ax.set_xlim(left=0);ax.set_xlabel('Years since verified data release');ax.set_ylabel('Probability of no first '+plot['endpoint']);ax.legend(frameon=False)
        ax.text(.01,.015,'Risk sets and censor counts exported; no lifetime extrapolation.',transform=ax.transAxes,fontsize=8)
    elif kind=='percentiles':
        for typ,g in df.groupby('cohort'):ax.hist(g.citation_percentile,bins=np.arange(0,101,10),histtype='step',linewidth=2,label=typ)
        ax.set_xlabel('InCites percentile (lower = more cited)');ax.set_ylabel('Publications');ax.legend(frameon=False,fontsize=9)
    return ext_save(fig,'rep_'+key)


def rep_write_report(report,figures):
    import html as _html,base64 as _b64
    esc=lambda v:_html.escape(str(v))
    sections=[]
    for item in report['items']:
        key=item['id'];image=''
        if key in figures:
            path=OUT/figures[key]['svg'];image='<img alt="'+esc(item['title'])+'" src="data:image/svg+xml;base64,'+_b64.b64encode(path.read_bytes()).decode()+'">'
        table=report['tables'].get(key,[])
        markup=pd.DataFrame(table).head(30).to_html(index=False,escape=True,border=0) if table else '<p>No estimate produced.</p>'
        link='<p>Full table: <a href="rep_'+key+'.csv">CSV</a></p>' if table else ''
        sections.append('<section><h2>'+esc(item['paper'].title()+' '+item['location'])+'</h2><h3>'+esc(item['title'])+'</h3><span class="status">'+esc(item['status'])+'</span><p>'+esc(item['detail'])+'</p>'+image+markup+link+'<p><a href="'+esc(item['source_url'])+'">Primary paper</a></p></section>')
    body='<!doctype html><meta charset="utf-8"><title>Published-study analyses: dbGaP</title><style>body{font:16px Arial;margin:30px auto;max-width:1420px;padding:0 26px;color:#183047;background:#edf3f6}section{background:white;margin:22px 0;padding:25px;border:1px solid #cedae3;border-radius:8px;overflow-x:auto}img{display:block;max-width:950px;width:100%;height:auto}table{border-collapse:collapse;font-size:13px}td,th{padding:7px 10px;border-bottom:1px solid #d3dce3;text-align:left}th{background:#edf3f6}.status{display:inline-block;background:#e6f2f1;padding:7px 12px;font-weight:bold}h1{font-size:32px}p{max-width:1080px;line-height:1.5}</style>'
    body+='<h1>Published-study analyses and comparability</h1><p>'+esc(report['scope'])+'</p><p>'+esc(report['references']['scope'])+'</p>'
    if OUTPUT_LABEL:body+='<h2>'+esc(OUTPUT_LABEL)+'</h2>'
    body+=''.join(sections)
    p=OUT/'paper_replication_report.html';p.write_text(body);OUTPUTS[p.name]=hashfile(p)
    return p.name


if not REUSE:
    RESULTS['figures']={}
    for _k,_d in RESULTS['participation']['annual'].items():RESULTS['figures']['annual_'+_k]=ext_annual_plot(_d,_k)
    for _k,_d in RESULTS['participation']['volumes'].items():
        RESULTS['figures']['hist_volume_'+_k]=ext_volume_plot(_d,_k)
        RESULTS['figures']['rank_volume_'+_k]=ext_volume_plot(_d,_k,rank=True)
    RESULTS['figures'].update(ext_type_charts(RESULTS['data_type_evolution']))
    _paper_figs={}
    for _it in RESULTS['literature_replication']['items']:
        _key=_it['id']
        if _key in RESULTS['literature_replication']['plots']:
            _paper_figs[_key]=rep_plot(RESULTS['literature_replication']['plots'][_key],_key,_it['title'])
    RESULTS['replication_figures']=_paper_figs
    RESULTS['replication_report']=rep_write_report(RESULTS['literature_replication'],_paper_figs)
    _sem=RESULTS['semantics']
    _rows=_sem['specific_descriptor_counts'] if _sem['mesh_vocabulary_available'] else _sem['recorded_descriptor_counts']
    _title='Most-specific supported disease descriptors' if _sem['mesh_vocabulary_available'] else 'Source-recorded MeSH terms (not hierarchy-refined)'
    RESULTS['figures']['semantic_themes']=draw_semantic_bars(_rows,'label',_title,'semantic_themes')
    RESULTS['figures']['semantic_assays']=draw_semantic_bars(RESULTS['semantics']['assay_counts'],'family','Reported assay / data families','semantic_assays')
    RESULTS['figures']['semantic_cross']=draw_semantic_matrix(RESULTS['semantics'])
    RESULTS['figures']['rank_contributors_pi']=participant_rank_plot(RESULTS['participation']['contributor_volume'],'contributors')
    RESULTS['figures']['rank_accessors_pi']=participant_rank_plot(RESULTS['participation']['volumes']['people'],'accessors')
    for k in ['accessions','publications','dars']:
        for kind in ['annual','cumulative']:RESULTS['figures']['growth_'+k+'_'+kind]=draw_growth(RESULTS['growth'][k],k,kind)
    for k,v in RESULTS['profile']['histograms'].items():RESULTS['figures']['profile_'+k]=draw_profile_hist(v,k)
    RESULTS['figures']['consent_labels']=draw_consent_labels(RESULTS['profile']['consents'])
    RESULTS['figures']['version_matrix']=draw_version_matrix(RESULTS['profile']['versions'])
    if MODEL_ENABLED and RESULTS['comparison']['status']=='fitted':RESULTS['figures']['both_neither_adjusted']=draw_comparison(RESULTS['comparison'])
    RESULTS['figures']['paper_hist']=draw_hist(RESULTS['paper_hist'],'paper_hist','Publications per accession','Distinct successfully retrieved PMIDs',PALETTE['blue'])
    RESULTS['figures']['dar_hist']=draw_hist(RESULTS['dar_hist'],'dar_hist','DAR records per accession','Recorded accession-requester rows',PALETTE['teal'])
    RESULTS['figures']['rcr_hist']=draw_hist(RESULTS['rcr'],'rcr_hist','RCR distribution','RCR bins (zero retained; missing excluded)',PALETTE['teal'])
    REFERENCES.extend([
      {'key':'mesh_semantics','label':'NLM MeSH trees and XML','url':'https://www.nlm.nih.gov/databases/download/mesh.html','finding':'Explicit descriptors, concept synonyms and tree positions.','boundary':'Hierarchy context is not exact term identity.','reviewed':'2026-10-08'},
      {'key':'obi_semantics','label':'OBI assay ontology','url':'https://obi-ontology.org/','finding':'Defined assay classes identified by persistent IRIs.','boundary':'A local assay crosswalk is not participant-level data harmonization.','reviewed':'2026-10-08'},
      {'key':'rcdc_semantics','label':'NIH RCDC process','url':'https://report.nih.gov/funding/categorical-spending/rcdc-process','finding':'RCDC uses weighted concept/category fingerprints and review.','boundary':'These profiles are NOT official RCDC categories or scores.','reviewed':'2026-10-08'}])
    RESULTS['references']=REFERENCES
    RESULTS['methods']={'group_comparison':RESULTS['group_comparison']['note'], 'optional_model':RESULTS['comparison']['interpretation'], 'date_basis':ACCESSION_DATE_BASIS,
        'counts':'Root, PMID, recorded accession-requester row and NCT are distinct units.',
        'covariates':MODEL_COVARIATES,'missingness':'Descriptive panels show available-case denominators; optional model uses complete cases. Unavailable counts are never zero.',
        'assumptions':['Current metadata can postdate outcomes','Family structure may be incompletely recorded','Neither does not mean never used'],
        'confounders':['Study age and observation window','Program and disease mix','Size, modality and phenotype depth','Consent constraints','Metadata curation and missingness','Parent-child/shared-paper dependence'],
        'profile_sources':'Field-level sources in accession_profile_lineage.csv; conflicts in accession_profile_conflicts.csv.'}
    RESULTS['gaps']=GAPS;RESULTS['checks']=CHECKS
    RESULTS['input_provenance']=INPUTS
    write_json(OUT/'summary.json',RESULTS);OUTPUTS['summary.json']=hashfile(OUT/'summary.json')
    # Preserve exact input hashes and new output hashes; no raw-table overwrite.
    check('Inputs unchanged during analysis',all(hashfile(Path(i['path']))==i['sha256'] for i in INPUTS))
    write_json(OUT/'manifest.json',{'recipe':RECIPE,'signature':SIGNATURE,'inputs':INPUTS,'settings':SETTINGS,'outputs':OUTPUTS,
        'checks':CHECKS,'gaps':GAPS,'built_utc':datetime.now(timezone.utc).isoformat()})
else:
    print('Matched input/recipe/output hashes: no scientific files or figures rewritten.')
print(f"Ready for Notebook C: {OUT/'summary.json'}")
print('Fourteen-slide layout: native MeSH recovery, hierarchy profiles and assay harmonization are ready for C.')


## Optional actual-submitter input

`submission_entities.parquet` (or CSV) is optional and must contain `root_accession`, `entity_id`, `entity_name`, `institution_name`, `source_url`, and `role`. Allowed submission roles: `data_submitter`, `submitting_pi`, `submitting_group`. A `validated` flag, when supplied, must be true. Do not populate this table from grant PIs, manuscript authors or sponsors without independent submission evidence.

Otherwise B uses A's public study-PI/affiliation **labels as proxies**, clearly labelled on slide 4. Multiple spellings, departments, homonyms and institutional moves remain unresolved. Gini refers to observed label/identifier distributions, not population equity.

The slide-1 acknowledgement measure uses only `dbgap_public_attribution.parquet`. Optional `pubmed_grant_acknowledgements` is retained as a separate paper-funding measure. The two are not interchangeable.

No reset or deletion is needed for ordinary Run all. `ANALYSIS_MODE='rebuild'` recomputes these local outputs only.

## Interpretation and optional model

The visible comparison is descriptive and contains no adjusted coefficients. Set `MODEL_ENABLED=True` only to run the separately saved diagnostic model; failures remain in its audit, not in a blank slide panel.

Version checks save `version_audit.json`, `version_pair_evidence.csv`, `highest_observed_version_pairs.csv` and the exact matrix. Numeric suffixes are not reconstructed histories: observing only `phs001974.v10.p1` means one retained pair at coordinates (10,1), not v1.p1 and not proof of ten retained releases. The primary matrix uses each root's highest co-observed numeric pair. All observed pairs are exported separately. The `phs001974` check uses saved fields only; the example is never injected into data.

Consent labels use the selected SSTR's `short_name`/`name` and code, or explicit Authorized Access names when SSTR labels are unavailable. No universal `1=GRU` mapping is imposed. Full modifier-bearing labels and lookup provenance are exported. Each root can contribute to more than one consent or MeSH category.

Samples use `n_samples_total` or `study_stats.cnt_samples.loaded`; no sum over assays, duplicated versions or consent-group summaries. The collection notebook already captures these fields. No recollection or source-table edits are required.


## Optional reviewed semantic mappings

B writes `semantic_text_review_candidates.csv`. To include human-accepted mappings in a separate profile lane, place `semantic_mesh_adjudications.csv` beside the A files with **candidate_id, mesh_id, source_sha256, decision, reviewer**. `decision` must be `accepted`, the candidate and hash must match this run, and `SEMANTIC_INCLUDE_REVIEWED` must be enabled. Default is off. No LLM labels enter the reference annotations.

Native annotations and text candidates are not the same evidence. A shared ancestor, including Neoplasms, only describes broader topical context; it does not validate a dataset-trial relationship. `semantic_trial_pair_comparison.csv` preserves exact native overlap separately from hierarchy results. The retained NCT references may be publication-mediated and their roles are not inferred.

Assay crosswalks describe labels in saved metadata, not harmonization of units, feature definitions, samples or experimental design. The `sample_count` fields in detailed evidence are **not additive** across assays, versions or repeated parent/child reports. Optional independent `sra_experiment_metadata.parquet` must explicitly contain `root_accession` and library strategy; it is not collected through speculative BioProject joins.


## Interactive graph companion (local HTML; optional)

Builds a bounded, pannable, selectable graph from saved B results and original link tables. No collection, source edits, new trial associations, or funding allocation occurs here. Disable `BUILD_INTERACTIVE_EXPLORER` in Settings to skip. Open the reported `interactive_explorer/.../explorer.html` in a desktop browser. The existing 13-slide Notebook C remains compatible and unchanged.


In [ ]:
# @title Interactive dbGaP relationship explorer -- local files only
# Run after Notebook B, or point CATALOG_DIR to an extracted MeSH catalog.
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/dbgap_analysis')
ANALYSIS_DIR = ''                     # Default: DRIVE_ROOT/preliminary14_analysis
CATALOG_DIR = ''                      # Optional: directory containing datasets.jsonl and trials.jsonl
EXPLORER_OUTPUT = ''                  # Default: DRIVE_ROOT/interactive_explorer
MOUNT_DRIVE = False
DOWNLOAD_HTML = False
VERIFY_ANALYSIS_MANIFEST = True
DEFAULT_VIEW = 'mesh_trial'           # mesh_trial | grant_trial | dataset_similarity
DEFAULT_STUDY_LIMIT = 40              # Display cap only; all records remain filterable and exportable
GRAPH_HEIGHT = 580                    # CSS pixels; also bounded by window height
PAGE_MAX_WIDTH = 1840                 # Fits a 1920-pixel desktop with margin

import ast, copy, csv, hashlib, html, json, math, os, re, sys, unicodedata
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

EXPLORER_VERSION = 'dbgap-relationship-explorer-1.0'
GX_CODE_SHA256 = 'b5d5ed6b6bd3a30730e9971ebcb29e2f42a419a4dfa0adc710f293875eb83756'


def gx_text(v):
    if v is None: return ''
    if isinstance(v, float) and not math.isfinite(v): return ''
    return str(v).strip()


def gx_decode(v):
    if isinstance(v, (dict, list, tuple)): return v
    s = gx_text(v)
    if not s: return None
    for fn in (json.loads, ast.literal_eval):
        try: return fn(s)
        except (ValueError, TypeError, SyntaxError): pass
    return v


def gx_list(v):
    v = gx_decode(v)
    if v is None: return []
    if isinstance(v, (list, tuple, set)): return list(v)
    return [v]


def gx_strings(v):
    return sorted({gx_text(x) for x in gx_list(v) if not isinstance(x, (dict, list)) and gx_text(x) not in {'', 'null', 'None', 'nan', '<NA>'}})


def gx_bool(v):
    s = gx_text(v).lower()
    return True if s in {'true','1','1.0','yes'} else False if s in {'false','0','0.0','no'} else None


def gx_num(v):
    try:
        n = float(v)
        return n if math.isfinite(n) else None
    except (ValueError, TypeError): return None


def gx_root(v):
    m = re.fullmatch(r'(phs\d{6,})(?:\.v\d+\.p\d+(?:\.c\d+)?)?', gx_text(v).lower())
    return m.group(1) if m else None


def gx_nct(v):
    s = gx_text(v).upper()
    return s if re.fullmatch(r'NCT\d{8}', s) else None


def gx_mid(v):
    s = gx_text(v).rsplit('/', 1)[-1].removesuffix('.html')
    return s if re.fullmatch(r'D\d{6,9}', s) else None


def gx_pmid(v):
    s = gx_text(v)
    if re.fullmatch(r'\d+\.0', s): s = s[:-2]
    return str(int(s)) if s.isdigit() and int(s) > 0 else None


def gx_core(v):
    s = re.sub(r'\s+', '', gx_text(v)).upper()
    # Keep source core/project identifier; do not parse a full award into a core heuristically.
    return s if s and len(s) <= 64 and re.fullmatch(r'[A-Z0-9][A-Z0-9._/-]*', s) else None


def gx_first(row, keys):
    for k in keys:
        v = row.get(k)
        if v is not None and gx_text(v) not in {'', 'null', 'None', 'nan', '<NA>'}: return v
    return None


def gx_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''): h.update(block)
    return h.hexdigest()


def gx_clean(v):
    if isinstance(v, dict): return {str(k):gx_clean(x) for k,x in v.items()}
    if isinstance(v, (list,tuple,set)): return [gx_clean(x) for x in v]
    if v is None: return None
    if isinstance(v, float): return v if math.isfinite(v) else None
    if isinstance(v, (str,int,bool)): return v
    if hasattr(v, 'tolist'): return gx_clean(v.tolist())
    if hasattr(v, 'isoformat'): return v.isoformat()
    if gx_text(v) in {'NaT','<NA>'}: return None
    return str(v)


def gx_load(path, inventory):
    path = Path(path)
    before = gx_hash(path)
    if path.suffix == '.jsonl':
        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    elif path.suffix == '.json':
        rows = json.loads(path.read_text(encoding='utf-8'))
    elif path.suffix == '.csv':
        with path.open(encoding='utf-8-sig', newline='') as f: rows = list(csv.DictReader(f))
    elif path.suffix == '.parquet':
        try:
            import pandas as pd
            rows = gx_clean(pd.read_parquet(path).to_dict('records'))
        except ImportError as exc:
            raise RuntimeError('Reading Parquet needs pandas and pyarrow. In Colab run: %pip install pandas pyarrow. CSV and catalog JSONL need no extra packages.') from exc
    else: raise ValueError('Unsupported input: '+str(path))
    if gx_hash(path) != before: raise RuntimeError('Input changed while reading: '+str(path))
    inventory.append({'path':str(path.resolve()), 'sha256':before, 'rows':len(rows) if isinstance(rows,list) else None})
    return rows


def gx_pick(base, stems, inventory):
    for stem in stems:
        for suffix in ('.parquet', '.csv', '.jsonl'):
            p = Path(base)/(stem+suffix)
            if p.is_file(): return gx_load(p, inventory)
    return []


def gx_unique(rows, key, kind):
    out = {}
    for row in rows:
        ident = key(row)
        if not ident: continue
        if ident in out and out[ident] != row:
            raise ValueError('Conflicting duplicate '+kind+' identifier: '+str(ident))
        out[ident] = row
    return out


def gx_term_tokens(value):
    value = gx_decode(value)
    if isinstance(value, list):
        return [x for v in value for x in gx_term_tokens(v)]
    if not isinstance(value, dict): return []
    mid = gx_mid(gx_first(value, ['mesh_id','descriptor_id','code','id']))
    label = gx_text(gx_first(value, ['mesh_label','label','display','term','name']))
    return [(mid,label)] if mid else []


def gx_read_sources(base, analysis_dir='', catalog_dir='', verify=True):
    base = Path(base); inv=[]; warns=[]
    analysis = Path(analysis_dir) if analysis_dir else base/'preliminary14_analysis'
    explicit_catalog = bool(catalog_dir)
    cat = Path(catalog_dir) if explicit_catalog else base
    raw = any((base/('joined_accessions_dar'+s)).is_file() for s in ('.parquet','.csv'))
    if not explicit_catalog and not raw and not (cat/'datasets.jsonl').is_file():
        candidates=[]
        for name in ['clinical_trial_dataset_mesh_catalog','clinical_trial_dataset_catalog']:
            parent=base/name
            if parent.is_dir(): candidates += [p.parent for p in parent.glob('*/datasets.jsonl') if (p.parent/'trials.jsonl').is_file()]
        if candidates: cat=max(candidates,key=lambda p:(p/'datasets.jsonl').stat().st_mtime)
    catalog_mode = explicit_catalog or not raw
    if catalog_mode:
        if not (cat/'datasets.jsonl').is_file():
            raise FileNotFoundError('No saved roots or catalog found. Set DRIVE_ROOT to the A/B folder, or CATALOG_DIR to the extracted catalog directory.')
        src={'roots':gx_load(cat/'datasets.jsonl',inv), 'trials':gx_load(cat/'trials.jsonl',inv) if (cat/'trials.jsonl').is_file() else [],
             'pair_evidence':gx_load(cat/'link_evidence.jsonl',inv) if (cat/'link_evidence.jsonl').is_file() else [],
             'catalog_pairs':gx_load(cat/'trial_dataset_catalog.jsonl',inv) if (cat/'trial_dataset_catalog.jsonl').is_file() else [],
             'links':[], 'reporter':[], 'mxn':[], 'projects':[], 'auth':[], 'source_mode':'catalog'}
        # Only co-located source exports: never mix a historical catalog with current A/B by accident.
        source_folder=cat/'source_rows'
        for key,stems in [('reporter',['reporter_pub_links_full']),('mxn',['reporter_pmid_core_award_MxN']),('projects',['reporter_projects_full']),('links',['dbgap_study_pubmed_links_full'])]:
            _p=next((source_folder/(stem+'.jsonl') for stem in stems if (source_folder/(stem+'.jsonl')).is_file()),None)
            _rows=gx_load(_p,inv) if _p else gx_pick(source_folder,stems,inv) or gx_pick(cat,stems,inv)
            src[key]=[x.get('record',x) if isinstance(x,dict) else x for x in _rows]
        sem_base=cat
    else:
        src={'roots':gx_pick(base,['joined_accessions_dar','dbgap_root_accessions'],inv),
             'trials':gx_pick(base,['dbgap_ctgov_direct_links'],inv),
             'pair_evidence':[], 'catalog_pairs':[],
             'links':gx_pick(base,['dbgap_study_pubmed_links_full'],inv),
             'reporter':gx_pick(base,['reporter_pub_links_full'],inv),
             'mxn':gx_pick(base,['reporter_pmid_core_award_MxN'],inv),
             'projects':gx_pick(base,['reporter_projects_full'],inv),
             'auth':gx_pick(base,['dbgap_authorized_access_api_metadata'],inv), 'source_mode':'pipeline'}
        sem_base=analysis
        manifest=analysis/'manifest.json'
        if manifest.is_file():
            m=gx_load(manifest,inv)
            if verify:
                for name,sha in m.get('outputs',{}).items():
                    p=(analysis/name).resolve()
                    if not p.is_relative_to(analysis.resolve()): raise ValueError('Unsafe analysis manifest path')
                    if not p.is_file() or gx_hash(p)!=sha: raise ValueError('B output is missing/changed; rerun B: '+str(p))
                for r in m.get('inputs',[]):
                    p=Path(r['path'])
                    if not p.is_file() or gx_hash(p)!=r['sha256']: raise ValueError('B source snapshot is stale; rerun B: '+str(p))
        elif analysis.is_dir(): warns.append('No B manifest found: supplemental analysis-file consistency is not verified.')
        pe=base/'semantic_sources'/'dataset_trial_evidence.jsonl'
        if pe.is_file(): src['pair_evidence']=gx_load(pe,inv)
        ti=base/'semantic_sources'/'trial_info.jsonl'
        if ti.is_file(): src['trial_info']=gx_load(ti,inv)
    for key,stem in [('profiles','semantic_profiles'),('bridges','semantic_trial_bridges'),('pair_semantics','semantic_trial_pair_comparison'),('assays','semantic_assay_profiles')]:
        src[key]=gx_pick(sem_base,[stem],inv)
    src['inventory']=inv; src['warnings']=warns;src['catalog_path']=str(cat) if catalog_mode else None
    return src


def gx_build_bundle(src):
    warns=list(src.get('warnings',[])); mesh={};datasets={}; trials={};projects={};pairs={};grants={}
    def addmesh(mid,label='',status='source-recorded'):
        mid=gx_mid(mid)
        if not mid:return
        if mid not in mesh: mesh[mid]={'id':mid,'label':label or mid,'url':'https://id.nlm.nih.gov/mesh/'+mid,'status':status}
        elif mesh[mid]['label']==mid and label:mesh[mid]['label']=label
    roots=gx_unique(src['roots'],lambda r:gx_root(gx_first(r,['root_accession','latest_accession','accession'])),'dataset')
    for ident,r in sorted(roots.items()):
        title=gx_first(r,['study_title','study_name','saved_joined__study_name'])
        _title=gx_decode(title)
        title='; '.join(gx_strings(_title)) if isinstance(_title,(list,tuple)) else gx_text(title)
        if not title:title='; '.join(gx_strings(r.get('titles_seen'))) or ident
        acc=gx_text(gx_first(r,['latest_accession','accession','saved_joined__latest_accession']))
        if gx_root(acc)!=ident:acc=ident
        mids={};anno=r.get('dataset_mesh_annotations',[])
        for a in anno:
            if a.get('eligible_for_shared_match') is True and gx_mid(a.get('mesh_id')):
                mids[a['mesh_id']]=gx_text(a.get('mesh_label'));addmesh(a['mesh_id'],gx_text(a.get('mesh_label')),'verified in supplied catalog')
        # Pipeline has explicit source IDs but no vocabulary validation until B profiles are used.
        if not anno and src.get('source_mode')=='pipeline':
            for mid,label in gx_term_tokens(r.get('mesh_terms')):mids[mid]=label;addmesh(mid,label,'source ID; use B for pinned-vocabulary resolution')
        data_types=gx_strings(gx_first(r,['data_types','saved_joined__data_types']))
        consent=[]
        for x in gx_list(gx_first(r,['consent_groups','saved_joined__consent_groups'])):
            if isinstance(x,dict):
                name=gx_text(gx_first(x,['short_name','name']))
                if name:consent.append(name)
        if not consent:consent=gx_strings(r.get('consent_group_names'))
        institutions=[]
        for x in gx_list(r.get('public_investigator_attribution')):
            if isinstance(x,dict) and gx_text(x.get('organization_label')): institutions.append(gx_text(x['organization_label']))
        fields={}
        for name,keys in {'Subjects':['subjects','n_subjects_total'],'Samples':['samples','n_samples_total'],
            'Variables':['variables','num_variables'],'DAR records':['saved_dar_count','n_dar_total'],
            'Linked PubMed papers':['unique_pmids_in_saved_link_table','n_pmids_linked'],'Median RCR':['saved_median_rcr','median_rcr'],
            'Study status':['study_status','saved_joined__study_status'],'Study design':['study_design'],
            'Release date (source-recorded)':['release_date_recorded','release_date'],
            'Latest recorded accession':['latest_accession'],'Administering IC':['admin_ic']}.items():
            val=gx_first(r,keys)
            if val is not None:fields[name]=gx_clean(val)
        platforms=gx_strings(r.get('external_datasource_names'))
        for label,keys in [('BDC',['bdc_listed_recorded','platform_biodata_catalyst']),('SRA',['sra_available_recorded','any_consent_group_sra_available']),('AnVIL',['platform_anvil'])]:
            if gx_bool(gx_first(r,keys)) is True:platforms.append(label)
        d={'id':ident,'label':title,'latest_accession':acc,'url':'https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id='+acc,
           'mesh_ids':sorted(mids),'mesh_labels':mids,'specific_mesh_ids':sorted(mids),'data_types':data_types,'assay_families':[],
           'platforms':sorted(set(platforms)),'admin_ic':gx_strings(r.get('admin_ic')),'consent':sorted(set(consent)),
           'institutions':sorted(set(institutions)),'study_types':gx_strings(r.get('study_types')),'fields':fields,
           'mesh_status':gx_text(r.get('dataset_mesh_status')) or 'source metadata; see provenance',
           'collection':gx_bool(gx_first(r,['is_collection_recorded','is_collection'])),
           'subjects':gx_num(gx_first(r,['subjects','n_subjects_total'])),'dar_count':gx_num(gx_first(r,['saved_dar_count','n_dar_total'])),
           'paper_count':gx_num(gx_first(r,['unique_pmids_in_saved_link_table','n_pmids_linked']))}
        datasets[ident]=d
    for r in src.get('auth',[]):
        rid=gx_root(r.get('root_accession'))
        if rid not in datasets:continue
        d=datasets[rid]
        d['platforms']=sorted(set(d['platforms']+gx_strings(r.get('external_datasource_names'))))
        for label,key in [('BDC','platform_biodata_catalyst'),('SRA','any_consent_group_sra_available'),('AnVIL','platform_anvil'),('CRDC','platform_crdc'),('GDC','platform_gdc'),('CTDC','platform_ctdc'),('Kids First','platform_kids_first'),('NDA','platform_nda'),('EGA','platform_ega'),('ImmPort','platform_immport')]:
            if gx_bool(r.get(key)) is True:d['platforms']=sorted(set(d['platforms']+[label]))
        d['collection']=gx_bool(r.get('is_collection')) if 'is_collection' in r else d['collection']
    for t in src.get('trials',[])+src.get('trial_info',[]):
        ident=gx_nct(gx_first(t,['nct_id','entity_id']))
        if not ident:continue
        label=gx_text(gx_first(t,['ctgov_brief_title','brief_title','title'])) or ident
        midmap={}
        for a in t.get('trial_mesh_annotations',[]):
            if a.get('eligible_for_shared_match') is True and gx_mid(a.get('mesh_id')):
                midmap[a['mesh_id']]=gx_text(a.get('mesh_label'));addmesh(a['mesh_id'],gx_text(a.get('mesh_label')),'verified in supplied catalog')
        record={'id':ident,'label':label,'url':'https://clinicaltrials.gov/study/'+ident,'mesh_ids':sorted(midmap),
                'mesh_labels':midmap,'mesh_status':gx_text(t.get('trial_mesh_status')) or 'See saved source metadata',
                'status':gx_text(gx_first(t,['ctgov_overall_status','overall_status'])),'type':gx_text(gx_first(t,['ctgov_study_type','study_type'])),
                'conditions':gx_strings(gx_first(t,['ctgov_conditions','conditions'])),'sponsor':gx_strings(gx_first(t,['ctgov_lead_sponsor','lead_sponsor'])),
                'fields':{k:gx_clean(v) for k,v in t.items() if k.startswith('ctgov_') and v is not None and len(str(v))<1600}}
        if ident not in trials:trials[ident]=record
        else:
            old=trials[ident]
            for k,v in record.items():
                if v and not old.get(k):old[k]=v
                elif k in {'type','status','label'} and v and old.get(k) not in {v,ident,''}:
                    warns.append('Conflicting retained trial '+k+' for '+ident+'; first value displayed, source records unchanged.')
    def addpair(rid,nid,e=None):
        if rid not in datasets or not nid:return
        key=rid+'|'+nid
        if nid not in trials:trials[nid]={'id':nid,'label':nid,'url':'https://clinicaltrials.gov/study/'+nid,'mesh_ids':[],'mesh_labels':{},'fields':{},'type':'','status':'','conditions':[],'sponsor':[]}
        if key not in pairs:pairs[key]={'id':key,'root':rid,'trial':nid,'basis':[],'evidence':[],'bridges':[]}
        if e:
            field=gx_text(gx_first(e,['dbgap_evidence_field','source_field']))
            basis=gx_text(e.get('relationship_basis')) or ('publication-mediated NCT reference' if 'publication' in field.lower() else 'recorded reference; role unadjudicated')
            if basis not in pairs[key]['basis']:pairs[key]['basis'].append(basis)
            evidence={'field':field,'source':gx_text(gx_first(e,['source','source_url','linkage_method'])),'text':gx_text(gx_first(e,['dbgap_evidence_text','evidence_text','context'])),'pmid':gx_pmid(e.get('pmid'))}
            if evidence not in pairs[key]['evidence']:pairs[key]['evidence'].append(evidence)
    for outer in src.get('pair_evidence',[]):
        r=outer.get('record',outer)
        addpair(gx_root(gx_first(outer,['root_accession']) or r.get('root_accession')),gx_nct(outer.get('nct_id') or r.get('nct_id')),r)
    for r in src.get('trials',[])+src.get('catalog_pairs',[]):
        addpair(gx_root(r.get('root_accession')),gx_nct(r.get('nct_id')),r if 'dbgap_evidence_field' in r else None)
    for rid,r in roots.items():
        for n in gx_strings(r.get('nct_ids')):addpair(rid,gx_nct(n))
    # Recover targeted evidence from full stored publication objects, never from an unrelated paper.
    for rid,r in roots.items():
        raw=gx_decode(gx_first(r,['publications_raw','saved_joined__publications_raw']))
        pubs=(raw.get('data',{}).get('publications',[]) if isinstance(raw,dict) else [])
        for pub in pubs:
            for field in ['refNotes','refTitle']:
                val=gx_text(pub.get(field))
                for m in re.finditer(r'\bNCT\d{8}\b',val,re.I):
                    key=rid+'|'+m.group().upper()
                    if key not in pairs:continue
                    e={'field':'publications_raw.'+field,'source':'saved publication metadata','pmid':gx_pmid(pub.get('pmid')),
                       'text':val[max(0,m.start()-200):min(len(val),m.end()+220)]}
                    if e not in pairs[key]['evidence']:pairs[key]['evidence'].append(e)
                    if 'publication-mediated NCT reference' not in pairs[key]['basis']:pairs[key]['basis'].append('publication-mediated NCT reference')
    for profile in src.get('profiles',[]):
        ident=gx_text(profile.get('entity_id'));kind=profile.get('entity_type');target=datasets.get(ident) if kind=='dataset' else trials.get(ident)
        if target is None:continue
        ids=[m for m in gx_strings(profile.get('native_mesh_ids')) if gx_mid(m)];labs=gx_strings(profile.get('native_mesh_labels'))
        # Labels are zipped using original ordering, not separately sorted.
        labs=[gx_text(x) for x in gx_list(profile.get('native_mesh_labels'))];ids=[gx_mid(x) for x in gx_list(profile.get('native_mesh_ids'))]
        labels={mid:(labs[i] if i<len(labs) else mid) for i,mid in enumerate(ids) if mid}
        for mid,l in labels.items():addmesh(mid,l,'resolved against B vocabulary snapshot')
        target['mesh_ids']=sorted(labels);target['mesh_labels']=labels;target['mesh_status']=gx_text(profile.get('profile_state'))
        if kind=='dataset':target['specific_mesh_ids']=[m for m in gx_strings(profile.get('specific_mesh_ids')) if gx_mid(m)]
    for a in src.get('assays',[]):
        rid=gx_root(a.get('root_accession'))
        if rid in datasets:
            datasets[rid]['assay_families']=gx_strings(a.get('families'))
            datasets[rid]['data_types']=sorted(set(datasets[rid]['data_types']+gx_strings(a.get('labels'))+gx_strings(a.get('raw_labels'))))
    for p in pairs.values():
        ds=set(datasets[p['root']]['mesh_ids']);ts=set(trials[p['trial']]['mesh_ids'])
        for mid in sorted(ds&ts):p['bridges'].append({'mesh_id':mid,'label':mesh[mid]['label'],'type':'exact native descriptor','broad':False,'source':'independent saved annotations'})
        p['gap']=('Both annotation sets unavailable' if not ds and not ts else 'Dataset annotations unavailable' if not ds else 'Trial condition annotations unavailable' if not ts else 'No exact shared descriptor')
        if not p['basis']:p['basis']=['recorded reference; role unadjudicated']
    for b in src.get('bridges',[]):
        key=gx_text(b.get('root_accession'))+'|'+gx_text(b.get('nct_id'))
        if key not in pairs:continue
        mid=gx_mid(gx_first(b,['bridge_mesh_id','bridge_id','mesh_id']))
        if not mid:continue
        label=gx_text(gx_first(b,['bridge_label','mesh_label'])) or mid;addmesh(mid,label,'B hierarchy output')
        typ=gx_text(b.get('match_type'))
        if typ=='exact':continue   # Exact native evidence already computed; reviewed-only exact retained below.
        bridge={'mesh_id':mid,'label':label,'type':'hierarchical context','broad':gx_bool(b.get('broad_fallback')) is True,
                'source':'B semantic_trial_bridges','details':gx_clean(b)}
        if not any(x['mesh_id']==mid and x['type']==bridge['type'] for x in pairs[key]['bridges']):pairs[key]['bridges'].append(bridge)
    # Projects remain source-recorded associations, not an allocation of funding to datasets/trials.
    def award(root,core,pmid=None,appl=None,basis='publication-mediated project association',source=''):
        if root not in datasets or not core:return
        key=root+'|'+core
        if key not in grants:grants[key]={'root':root,'project':core,'pmids':set(),'appl_ids':set(),'basis':basis,'sources':set()}
        g=grants[key]
        if pmid:g['pmids'].add(pmid)
        if appl:g['appl_ids'].add(appl)
        if source:g['sources'].add(source)
        if core not in projects:projects[core]={'id':core,'label':core,'appl_ids':set(),'organizations':set(),'admin_ic':set(),'activity':set(),'title':set(),'url':'https://reporter.nih.gov/advanced-search'}
        if appl:projects[core]['appl_ids'].add(appl)
    for rid,r in roots.items():
        for c in gx_strings(r.get('core_projects_linked_through_papers')):award(rid,gx_core(c),source='catalog core_projects_linked_through_papers')
    for r in src.get('mxn',[]):award(gx_root(r.get('root_accession')),gx_core(r.get('core_project_num')),gx_pmid(r.get('pmid')),gx_pmid(r.get('appl_id')),source='reporter_pmid_core_award_MxN')
    bypub=defaultdict(set)
    for r in src.get('links',[]):
        rid=gx_root(r.get('root_accession'));pid=gx_pmid(r.get('pmid'))
        if rid in datasets and pid:bypub[pid].add(rid)
    for r in src.get('reporter',[]):
        pid=gx_pmid(r.get('pmid'))
        for rid in bypub.get(pid,()):award(rid,gx_core(r.get('core_project_num')),pid,gx_pmid(r.get('appl_id')),source='root-PMID plus RePORTER PMID-project')
    for r in src.get('projects',[]):
        p=projects.get(gx_core(r.get('core_project_num')))
        if p is None:continue
        for key,cols in [('organizations',['org_name']),('admin_ic',['admin_ic']),('activity',['activity_code']),('title',['project_title','title'])]:
            val=gx_first(r,cols)
            p[key].update(gx_strings(val))
    for g in grants.values():
        for k in ['pmids','appl_ids','sources']:g[k]=sorted(g[k])
    for p in projects.values():
        for k in ['appl_ids','organizations','admin_ic','activity','title']:p[k]=sorted(p[k])
    for p in pairs.values():p['evidence']=sorted(p['evidence'],key=lambda e:(not bool(e['pmid']),e['field'],e['text']))
    return {'version':EXPLORER_VERSION,'datasets':list(datasets.values()),'trials':list(trials.values()),'mesh':list(mesh.values()),
            'projects':list(projects.values()),'pairs':list(pairs.values()),'awards':list(grants.values()),
            'provenance':{'source_mode':src.get('source_mode'),'inputs':src.get('inventory',[]),'warnings':sorted(set(warns)),
            'linkage_note':'Project links are publication-mediated; NCT links may be publication mentions. Juxtaposition is not a funded causal pathway.',
            'reference':'https://www.report.nih.gov/faqs'},
            'counts':{'datasets':len(datasets),'trials':len(trials),'pairs':len(pairs),'projects':len(projects),'project_dataset_pairs':len(grants),
                      'exact_mesh_pairs':sum(any(b['type']=='exact native descriptor' for b in p['bridges']) for p in pairs.values()),
                      'project_links_with_PMID_evidence':sum(bool(g['pmids']) for g in grants.values())}}


# Self-contained HTML/JavaScript is embedded below. No CDN or graph package is required.

GX_HTML = '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>dbGaP relationship explorer</title>\n<style>\n:root{--ink:#1d3449;--muted:#607184;--teal:#137d83;--line:#dbe4ec;--blue:#245b89;--amber:#aa6b12;--paper:#fff;--back:#edf2f6}\n*{box-sizing:border-box}body{margin:0;font:14px/1.45 system-ui,-apple-system,"Segoe UI",sans-serif;background:var(--back);color:var(--ink)}\nbutton,input,select{font:inherit}button,select,input[type=search],input[type=number]{border:1px solid #bacbd8;border-radius:5px;background:white;color:var(--ink);padding:6px 9px}button{cursor:pointer}button:hover{background:#e9f3f6}button.primary{background:var(--teal);color:white;border-color:var(--teal)}button:disabled{opacity:.45;cursor:default}input[type=checkbox]{accent-color:var(--teal)}a{color:#1767a1;overflow-wrap:anywhere}.shell{max-width:__MAX_WIDTH__px;margin:12px auto;padding:0 16px}header{display:flex;justify-content:space-between;align-items:center;gap:20px;padding:10px 3px}h1{font-size:24px;letter-spacing:-.5px;margin:0}.eyebrow{text-transform:uppercase;letter-spacing:1.3px;font-size:11px;color:var(--teal);font-weight:700}.subtitle{font-size:13px;color:var(--muted);margin-top:3px}h2{font-size:17px;margin:5px 0 12px}h3{font-size:14px;margin:12px 0 6px}p{margin:6px 0 12px}.badge{display:inline-block;font-size:11px;padding:3px 7px;border-radius:4px;background:#e9f2f7;border:1px solid #c6d6e2;margin-right:5px}\n.topbar,.viewbar,.toolbar{display:flex;align-items:center;flex-wrap:wrap;gap:8px;background:white;border:1px solid var(--line);padding:9px 12px}.topbar{border-radius:9px 9px 0 0}.topbar input[type=search]{flex:1;min-width:230px}.viewbar{border-top:0;padding:8px 12px}.viewbar label{font-size:12px;white-space:nowrap}.toolbar{border-top:0;border-bottom:0;padding:7px 10px;background:#f8fbfd}.toolbar button{padding:5px 9px}.spacer{flex:1}.layout{display:grid;grid-template-columns:278px minmax(0,1fr);gap:12px;margin-top:10px}.sidebar{border:1px solid var(--line);border-radius:9px;background:white;min-width:0;max-height:calc(100vh - 164px);overflow:auto}.sidehead{position:sticky;top:0;z-index:2;background:#fff;padding:11px;border-bottom:1px solid var(--line)}.sidehead p{font-size:12px;margin:4px 0;color:var(--muted)}.facets details{border-bottom:1px solid var(--line);padding:8px 10px}.facets summary{font-size:13px;font-weight:650;cursor:pointer;list-style:none;display:flex;align-items:center;gap:6px}.facets summary:after{content:\'+\';margin-left:auto;color:#789}.facets details[open] summary:after{content:\'-\'}summary input{margin:0}.facetcount{font-size:11px;color:var(--muted);font-weight:400}.facetsearch{width:100%;margin:8px 0 4px;font-size:12px!important}.facetactions{display:flex;gap:5px;margin:4px 0}.facetactions button{font-size:11px;padding:2px 6px}.options{max-height:168px;overflow:auto}.opt{display:flex;align-items:center;gap:6px;font-size:12px;padding:3px 0;line-height:1.25}.opt label{display:flex;gap:5px;align-items:center;flex:1;min-width:0}.opt label span{overflow:hidden;text-overflow:ellipsis;white-space:nowrap;display:block}.opt small{color:var(--muted);margin-left:auto}.only{font-size:10px;padding:1px 4px;border:0;background:#edf4f6}.facetoff .options,.facetoff .facetactions,.facetoff .facetsearch{opacity:.35;pointer-events:none}.range{padding:9px 10px;border-bottom:1px solid var(--line)}.range input[type=number]{width:85px;font-size:12px}.main{min-width:0}.mapbox{background:white;border:1px solid var(--line);border-radius:9px;overflow:hidden;position:relative}.maphead{padding:10px 12px 7px}.maphead .description{font-size:12px;color:var(--muted);margin-top:4px}.colheads{display:grid;grid-template-columns:repeat(3,1fr);background:#eff5f8;border-top:1px solid var(--line);border-bottom:1px solid var(--line);padding:8px 12px;text-align:center;font-size:12px;font-weight:700;gap:8px}.colheads span{min-width:0}.viewport{width:100%;height:min(__GRAPH_HEIGHT__px,calc(100vh - 490px));min-height:360px;background:#fff;overflow:hidden;position:relative;touch-action:none}.viewport svg{width:100%;height:100%;display:block;user-select:none;touch-action:none;cursor:grab}.viewport svg.dragging{cursor:grabbing}.edge{fill:none;stroke-opacity:.20;cursor:pointer;transition:stroke-opacity .15s}.edge.active{stroke-opacity:.72}.edge.dim{stroke-opacity:.022}.node{cursor:pointer;transition:opacity .15s}.node.dim{opacity:.1}.node.active rect.mainrect{stroke:#0e3a52;stroke-width:2.4}.node text{font-size:12px;fill:var(--ink);paint-order:stroke;stroke:white;stroke-width:3px;stroke-linejoin:round}.node .shortid{font-size:10px;fill:#5b6b7c}.node rect.hit{fill:transparent}.floating{position:absolute;bottom:10px;left:10px;border:1px solid #d6e0e8;background:#fffffff2;border-radius:5px;padding:5px 9px;font-size:11px;color:#52697a;pointer-events:none}.legend{display:flex;flex-wrap:wrap;gap:13px;padding:8px 12px;border-top:1px solid var(--line);font-size:11px;color:#52697a}.dot{display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:4px}.caption{font-size:12px;background:#fff7e8;border-left:3px solid #b87e25;padding:8px 10px;margin-top:8px}.chips{display:flex;flex-wrap:wrap;gap:5px;margin-bottom:8px}.chip{font-size:11px;padding:3px 7px;background:#dfeff0;border:1px solid #bfd8d9;border-radius:16px}.chip button{border:0;padding:0 2px;background:none}.stats{font-size:12px;margin:8px 2px;color:#465e73}.records{background:white;border:1px solid var(--line);border-radius:9px;margin-top:10px;padding:10px}.records summary{cursor:pointer;font-size:13px;font-weight:700}.tablewrap{max-height:280px;overflow:auto;margin-top:8px}table{border-collapse:collapse;width:100%;font-size:12px}th{position:sticky;top:0;background:#eaf1f5;text-align:left;padding:7px;border-bottom:1px solid #c5d5e0}td{padding:7px;border-bottom:1px solid #e5edf3;vertical-align:top;max-width:330px;overflow-wrap:anywhere}.inspector{display:none;position:absolute;top:0;right:0;bottom:0;width:355px;background:#fffffff5;border-left:1px solid #adc3d2;box-shadow:-8px 0 28px #1d34492b;padding:15px;z-index:5;overflow:auto}.inspector.open{display:block}.inspector h2{font-size:18px;overflow-wrap:anywhere}.inspector .close{float:right;padding:1px 7px}.inspector .kind{text-transform:uppercase;font-size:11px;font-weight:700;color:var(--teal);letter-spacing:1px}.inspector table{font-size:11px}.inspector td:first-child{width:40%;font-weight:600}.inspector .note{padding:9px;border-left:3px solid #ae7627;background:#fff4de;font-size:12px}.inspector .actions{display:flex;flex-wrap:wrap;gap:5px;margin:8px 0}.inspector .actions a{padding:5px 7px;border:1px solid #bfd0dc;border-radius:5px;font-size:12px;text-decoration:none;background:#f2f7fa}.inspector pre{white-space:pre-wrap;font-size:11px;overflow-wrap:anywhere}.inspector ul{padding-left:17px;font-size:12px}.empty{position:absolute;top:25%;left:10%;right:10%;padding:22px;border:1px dashed #afc6d4;background:#f6fafc;border-radius:8px;pointer-events:none}.empty h2{font-size:19px}.hidden{display:none!important}.footer{font-size:11px;color:var(--muted);padding:12px 4px}.provenance pre{max-height:200px;overflow:auto;font-size:10px;background:white;padding:10px}.toast{position:fixed;bottom:20px;left:50%;transform:translateX(-50%);background:#163a52;color:white;padding:9px 17px;border-radius:6px;z-index:30;display:none}.togglecompact{font-size:12px}.sidebarcollapsed .layout{grid-template-columns:1fr}.sidebarcollapsed .sidebar{display:none}\n@media(max-width:1100px){.layout{grid-template-columns:235px minmax(0,1fr)}.shell{padding:0 8px}.colheads{font-size:11px}.inspector{width:310px}.toolbar{gap:4px}.toolbar button{font-size:12px}}\n@media(max-width:780px){.layout{grid-template-columns:1fr}.sidebar{max-height:260px}.viewport{height:540px;min-height:320px}.inspector{width:min(330px,92%)}header{align-items:start}.subtitle{max-width:440px}}\n</style></head><body><div class="shell" id="shell">\n<header><div><div class="eyebrow">Evidence-linked exploration</div><h1>dbGaP relationship explorer</h1><div class="subtitle">Saved records, selectable facets and traceable connections. No live data collection.</div></div><div><button id="togglefacets">Hide facets</button> <button id="reset" class="primary">Reset all</button></div></header>\n<div class="topbar"><label for="view"><b>View</b></label><select id="view"><option value="mesh_trial">Studies - MeSH - trials</option><option value="grant_trial">Projects - studies - trials</option><option value="dataset_similarity">Focal study - MeSH - related studies</option></select><input id="search" type="search" placeholder="Search accessions, titles, NCT IDs, MeSH terms, projects..."><button id="clearselection">Clear selection</button></div>\n<div class="viewbar"><label id="focallabel" class="hidden">Focal study <select id="focal"><option value="">Choose a study...</option></select></label><label><input id="hierarchy" type="checkbox" checked> Include saved hierarchy context</label><label><input id="broad" type="checkbox" checked> Broad fallbacks</label><label><input id="gaps" type="checkbox" checked> Missing/no-match routes</label><label id="pairpathslabel"><input id="pairpaths" type="checkbox"> Separate pair paths</label><label id="samepaperlabel" class="hidden"><input id="samepaper" type="checkbox"> Same supporting PMID only</label><label>MeSH filter <select id="termscope"><option value="bridge">Displayed bridge</option><option value="any">Any dataset/trial annotation</option></select></label><span class="spacer"></span><label>Show studies <select id="limit"><option>20</option><option>40</option><option>80</option><option>200</option><option value="0">All</option></select></label></div>\n<div class="layout"><aside class="sidebar"><div class="sidehead"><h2>Facets</h2><p>OR within a facet; AND between enabled facets. Uncheck a facet to suspend it.</p><button id="enableall">Enable all</button> <button id="disableall">Disable all</button></div><div class="facets" id="facets"></div><div class="range"><b>Dataset subjects</b><p><input id="subjectmin" type="number" min="0" placeholder="Min"> to <input id="subjectmax" type="number" min="0" placeholder="Max"></p><label><input id="includeunknown" type="checkbox" checked> Keep unknown subject counts</label></div></aside>\n<main class="main"><div class="chips" id="chips"></div><div class="mapbox"><div class="maphead"><b id="maptitle"></b><div class="description" id="mapdescription"></div></div><div class="toolbar"><button id="zoomout" title="Zoom out">-</button><span id="zoomlabel">100%</span><button id="zoomin" title="Zoom in">+</button><button id="fit">Fit all</button><button id="reading">Readable scale</button><button id="resetview">Reset view</button><button id="layoutreset">Reset node positions</button><label class="togglecompact"><input id="dragmode" type="checkbox"> Move nodes</label><label class="togglecompact"><input id="showlabels" type="checkbox" checked> Labels</label><label class="togglecompact"><input id="fullpaths" type="checkbox" checked> Highlight full recorded routes</label><span class="spacer"></span><button id="exportsvg">Export SVG</button></div><div class="colheads" id="colheads"></div><div class="viewport" id="viewport"><svg id="graph" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Interactive relationship graph" tabindex="0"><g id="scene"></g></svg><div class="floating">Drag background to pan &middot; Wheel to zoom &middot; Click a node for evidence &middot; Esc clears selection</div><div class="empty hidden" id="empty"></div><aside class="inspector" id="inspector" aria-label="Selected node details"></aside></div><div class="legend"><span><i class="dot" style="background:#245b89"></i>dbGaP study</span><span><i class="dot" style="background:#137d83"></i>MeSH descriptor</span><span><i class="dot" style="background:#78649b"></i>NIH project / award</span><span><i class="dot" style="background:#596e3b"></i>NCT registration</span><span><i class="dot" style="background:#939faa"></i>Missing/no-match status</span><span>Link width: display-scaled association count, not dollars, patients or conserved flow</span></div></div>\n<div class="stats" id="stats" aria-live="polite"></div><div class="caption" id="meaning"></div>\n<details class="records"><summary>Visible relationship records and source evidence</summary><div><button id="exportrows">Export filtered records (JSON)</button> <button id="exportall">Export complete graph data (JSON)</button></div><div class="tablewrap"><table><thead><tr id="recordhead"></tr></thead><tbody id="recordbody"></tbody></table></div><div class="subtitle" id="recordnote"></div></details>\n<details class="records provenance"><summary>Source provenance and interpretation</summary><p id="sourceinfo"></p><p>All views retain the supplied accession-NCT relationships. Shared terms do not validate trial origin or secondary reuse. Projects linked through papers are not necessarily the awards that generated a dataset or supported a particular trial.</p><p><a href="https://www.report.nih.gov/faqs" target="_blank" rel="noopener noreferrer">NIH RePORTER: how projects are linked to publications</a></p><pre id="provenance"></pre></details>\n</main></div><div class="footer">Interactive HTML is the exploration view. PowerPoint, PNG and static SVG cannot retain these controls. Browser filtering does not change any source file.</div></div><div class="toast" id="toast"></div>\n<script id="graph-data" type="application/json">__DATA__</script>\n<script>\n\'use strict\';\nconst DATA=JSON.parse(document.getElementById(\'graph-data\').textContent),NS=\'http://www.w3.org/2000/svg\';\nconst $=id=>document.getElementById(id),dm=new Map(DATA.datasets.map(x=>[x.id,x])),tm=new Map(DATA.trials.map(x=>[x.id,x])),mm=new Map(DATA.mesh.map(x=>[x.id,x])),gm=new Map(DATA.projects.map(x=>[x.id,x]));\nconst awardsByRoot=new Map();DATA.awards.forEach(g=>{if(!awardsByRoot.has(g.root))awardsByRoot.set(g.root,[]);awardsByRoot.get(g.root).push(g)});\nconst colors={dataset:\'#245b89\',mesh:\'#137d83\',trial:\'#596e3b\',project:\'#78649b\',gap:\'#939faa\'};\nconst VIEWINFO={mesh_trial:{title:\'dbGaP studies - semantic descriptors - referenced trial registrations\',heads:[\'dbGaP studies\',\'Shared MeSH / explicitly marked coverage gaps\',\'ClinicalTrials.gov registrations\'],description:\'Only saved accession-NCT pairs contribute. Hierarchy context is distinct from exact native agreement.\',meaning:\'The diagram arranges metadata relationships; it is not a chronological, causal or reuse pipeline. Shared middle nodes may aggregate multiple pairs: select a node and inspect the exact records below.\'},grant_trial:{title:\'NIH project associations - dbGaP studies - referenced trial registrations\',heads:[\'NIH projects linked through publications\',\'dbGaP studies\',\'Referenced NCT registrations\'],description:\'Left edge: project-paper-study association. Right edge: recorded study-NCT reference. No direct project-trial claim.\',meaning:\'Do not read this as grant funding -> data generation -> clinical impact. The two edges can arise from different publications. "Same supporting PMID only" restricts to shared saved PMID evidence, but still does not prove causal funding or trial origin. No award dollars are allocated or added.\'},dataset_similarity:{title:\'Focal study - shared native MeSH - other studies\',heads:[\'Focal dbGaP study\',\'Shared native MeSH descriptors\',\'Other dbGaP studies sharing those descriptors\'],description:\'The two study columns are analytical roles in the same catalog, not producer and reuser populations.\',meaning:\'Similarity is an undirected topical relationship. It does not establish data reuse, overlapping participants, institutional collaboration or funding. Only exact independently recorded native descriptor overlap is used in this view.\'}};\nconst facetDefs=[\n{key:\'dataset\',label:\'dbGaP studies\',open:true,opts:DATA.datasets.map(d=>[d.id,d.id+\' | \'+d.label]),values:r=>[r.root,...(r.other?[r.other]:[])]},\n{key:\'mesh\',label:\'MeSH descriptors\',open:true,opts:DATA.mesh.map(m=>[m.id,m.label+\' [\'+m.id+\']\']),values:r=>$(\'termscope\').value===\'bridge\'&&$(\'view\').value!==\'grant_trial\'?(r.mesh?[r.mesh]:[]):[...new Set([...(dm.get(r.root)?.mesh_ids||[]),...(tm.get(r.trial)?.mesh_ids||[]),...(dm.get(r.other)?.mesh_ids||[]),...(r.mesh?[r.mesh]:[])])]},\n{key:\'assay\',label:\'Assay / data labels\',opts:[],values:r=>dm.get(r.root)?.data_types||[]},\n{key:\'assayfamily\',label:\'Harmonized assay families\',opts:[],values:r=>dm.get(r.root)?.assay_families||[]},\n{key:\'platform\',label:\'Repository / platform listing\',opts:[],values:r=>dm.get(r.root)?.platforms||[]},\n{key:\'ic\',label:\'Dataset administering IC\',opts:[],values:r=>dm.get(r.root)?.admin_ic||[]},\n{key:\'consent\',label:\'Consent / permitted-use labels\',opts:[],values:r=>dm.get(r.root)?.consent||[]},\n{key:\'studytype\',label:\'Study types\',opts:[],values:r=>dm.get(r.root)?.study_types||[]},\n{key:\'institution\',label:\'Publicly attributed institutions\',opts:[],values:r=>dm.get(r.root)?.institutions||[]},\n{key:\'collection\',label:\'Collection flag\',opts:[[\'true\',\'Collection\'],[\'false\',\'Not marked as collection\'],[\'unknown\',\'Unknown\']],values:r=>[dm.get(r.root)?.collection==null?\'unknown\':String(dm.get(r.root).collection)]},\n{key:\'trial\',label:\'ClinicalTrials.gov registrations\',opts:DATA.trials.map(t=>[t.id,t.id+\' | \'+t.label]),values:r=>r.trial?[r.trial]:[]},\n{key:\'trialtype\',label:\'Trial study type\',opts:[],values:r=>tm.get(r.trial)?.type?[tm.get(r.trial).type]:[]},\n{key:\'trialstatus\',label:\'Trial status\',opts:[],values:r=>tm.get(r.trial)?.status?[tm.get(r.trial).status]:[]},\n{key:\'trialsponsor\',label:\'Trial lead sponsor\',opts:[],values:r=>tm.get(r.trial)?.sponsor||[]},\n{key:\'project\',label:\'NIH core project / award\',opts:DATA.projects.map(g=>[g.id,g.id]),values:r=>r.project?[r.project]:(awardsByRoot.get(r.root)||[]).map(g=>g.project)},\n{key:\'projectorg\',label:\'Project organizations\',opts:[],values:r=>r.project?gm.get(r.project)?.organizations||[]:(awardsByRoot.get(r.root)||[]).flatMap(g=>gm.get(g.project)?.organizations||[])},\n{key:\'projectic\',label:\'Project administering IC\',opts:[],values:r=>r.project?gm.get(r.project)?.admin_ic||[]:(awardsByRoot.get(r.root)||[]).flatMap(g=>gm.get(g.project)?.admin_ic||[])},\n{key:\'evidence\',label:\'Relationship evidence\',opts:[],values:r=>r.basis||[]},\n{key:\'match\',label:\'Semantic connection type\',opts:[],values:r=>[r.match||\'not a semantic bridge\']}\n];\nconst state={facets:{},selected:null,nodes:[],edges:[],records:[],filtered:[],base:[],manual:new Map(),transform:{x:0,y:0,k:1},bounds:{width:1440,height:660},drag:null,totalMatchingStudies:0};\nfunction uniq(a){return [...new Set(a)]}function nodeText(v){return v==null?\'Unknown\':Array.isArray(v)?v.join(\'; \'):typeof v===\'object\'?JSON.stringify(v):String(v)}\nfunction svgEl(tag,attrs={},txt=\'\'){const e=document.createElementNS(NS,tag);Object.entries(attrs).forEach(([k,v])=>e.setAttribute(k,v));if(txt)e.textContent=txt;return e}\nfunction h(tag,txt=\'\',cls=\'\'){let el=document.createElement(tag);el.textContent=txt;if(cls)el.className=cls;return el}\nfunction toast(txt){$(\'toast\').textContent=txt;$(\'toast\').style.display=\'block\';setTimeout(()=>$(\'toast\').style.display=\'none\',2400)}\nfunction safeLink(url){try{let u=new URL(url);return [\'https:\',\'http:\'].includes(u.protocol)?u.href:null}catch(e){return null}}\nfunction link(label,url){let a=h(\'a\',label);a.href=safeLink(url)||\'#\';a.target=\'_blank\';a.rel=\'noopener noreferrer\';return a}\nfunction download(name,body,type){const blob=new Blob([body],{type:type||\'application/json\'}),a=document.createElement(\'a\');a.href=URL.createObjectURL(blob);a.download=name;a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000)}\nfunction rawRecords(view){\n let rows=[];\n if(view===\'dataset_similarity\'){\n  const focal=dm.get($(\'focal\').value);if(!focal)return [];\n  const fids=new Set(focal.mesh_ids);\n  DATA.datasets.forEach(d=>{if(d.id===focal.id)return;d.mesh_ids.forEach(mid=>{if(fids.has(mid))rows.push({id:\'sim|\'+focal.id+\'|\'+d.id+\'|\'+mid,root:focal.id,other:d.id,mesh:mid,match:\'exact native descriptor\',basis:[\'shared native descriptor; not reuse\'],pair:null,weight:1})})});\n  return rows;\n }\n for(const p of DATA.pairs){\n  if(view===\'mesh_trial\'){\n   const b=p.bridges.filter(b=>b.type===\'exact native descriptor\'||($(\'hierarchy\').checked&&(!b.broad||$(\'broad\').checked)));\n   if(b.length)b.forEach((x,i)=>rows.push({id:p.id+\'|\'+x.mesh_id+\'|\'+x.type,root:p.root,trial:p.trial,mesh:x.mesh_id,match:x.type+(x.broad?\' (broad fallback)\':\'\'),basis:p.basis,pair:p.id,evidence:p.evidence,bridge:x,weight:1/b.length}));\n   else if($(\'gaps\').checked)rows.push({id:p.id+\'|gap\',root:p.root,trial:p.trial,mesh:null,gap:p.gap,match:p.gap,basis:p.basis,pair:p.id,evidence:p.evidence,weight:1});\n  }else{\n   const gs=awardsByRoot.get(p.root)||[],pms=new Set(p.evidence.map(e=>e.pmid).filter(Boolean));\n   for(const g of gs){const common=g.pmids.filter(x=>pms.has(x));if($(\'samepaper\').checked&&!common.length)continue;\n    rows.push({id:\'g|\'+g.project+\'|\'+p.id,root:p.root,trial:p.trial,project:g.project,pair:p.id,basis:uniq([g.basis,...p.basis]),evidence:p.evidence,award:g,common_pmids:common,match:\'not a semantic bridge\',weight:1})}\n   if(!gs.length&&$(\'gaps\').checked&&!$(\'samepaper\').checked)rows.push({id:\'g|missing|\'+p.id,root:p.root,trial:p.trial,project:null,gap:\'No recorded project association\',pair:p.id,basis:p.basis,evidence:p.evidence,weight:1});\n  }\n }\n return rows;\n}\nfunction facetOptions(){\n const roots=DATA.datasets.map(d=>({root:d.id}));const pr=DATA.pairs.map(p=>({root:p.root,trial:p.trial,basis:p.basis}));\n const all=state.base.concat(roots,pr,DATA.awards.map(g=>({root:g.root,project:g.project})));\n for(const d of facetDefs){if(!d.opts.length){const vals=uniq(all.flatMap(r=>d.values(r)).filter(Boolean));d.opts=vals.map(v=>[v,v])}d.opts.sort((a,b)=>a[1].localeCompare(b[1]));if(!state.facets[d.key])state.facets[d.key]={enabled:true,values:new Set(),none:false,search:\'\'};}\n}\nfunction renderFacets(){\n const holder=$(\'facets\');holder.replaceChildren();\n for(const def of facetDefs){const f=state.facets[def.key];const detail=document.createElement(\'details\');detail.dataset.key=def.key;detail.open=!!def.open;detail.classList.toggle(\'facetoff\',!f.enabled);\n  const summary=document.createElement(\'summary\'),toggle=document.createElement(\'input\');toggle.type=\'checkbox\';toggle.checked=f.enabled;toggle.title=\'Enable this facet\';toggle.addEventListener(\'click\',e=>e.stopPropagation());toggle.onchange=()=>{f.enabled=toggle.checked;detail.classList.toggle(\'facetoff\',!f.enabled);applyFilters(false)};\n  summary.append(toggle,h(\'span\',def.label),h(\'span\',String(def.opts.length),\'facetcount\'));detail.append(summary);\n  const search=document.createElement(\'input\');search.type=\'search\';search.className=\'facetsearch\';search.placeholder=\'Find a value...\';search.value=f.search;search.oninput=()=>{f.search=search.value;renderOptions()};detail.append(search);\n  const actions=h(\'div\',\'\',\'facetactions\');for(const [name,fn]of [[\'All\',()=>{f.values.clear();f.none=false}],[\'None\',()=>{f.values.clear();f.none=true}]]){const b=h(\'button\',name);b.onclick=()=>{fn();renderOptions();applyFilters(false)};actions.append(b)}detail.append(actions);\n  const options=h(\'div\',\'\',\'options\');detail.append(options);\n  function renderOptions(){options.replaceChildren();const candidates=def.opts.filter(([id,label])=>!f.search||(id+\' \'+label).toLowerCase().includes(f.search.toLowerCase()));if(!candidates.length)options.append(h(\'div\',\'No values available\',\'subtitle\'));\n   candidates.forEach(([id,label])=>{const row=h(\'div\',\'\',\'opt\'),l=document.createElement(\'label\'),box=document.createElement(\'input\');box.type=\'checkbox\';box.value=id;box.dataset.facet=def.key;box.checked=f.values.has(id);l.title=label;const tx=h(\'span\',label);l.append(box,tx);row.append(l);const only=h(\'button\',\'only\',\'only\');only.title=\'Use only this value\';only.onclick=()=>{f.values=new Set([id]);f.none=false;f.enabled=true;toggle.checked=true;detail.classList.remove(\'facetoff\');renderOptions();applyFilters(true)};row.append(only);box.onchange=()=>{f.none=false;if(box.checked)f.values.add(id);else f.values.delete(id);applyFilters(true)};options.append(row)});\n  }renderOptions();holder.append(detail);\n }\n}\nfunction matches(r,skip=null){\n const q=$(\'search\').value.trim().toLowerCase();\n if(q){const hay=[r.root,dm.get(r.root)?.label,r.trial,tm.get(r.trial)?.label,r.other,dm.get(r.other)?.label,r.project,r.mesh,mm.get(r.mesh)?.label,...(dm.get(r.root)?.mesh_ids||[]).map(m=>mm.get(m)?.label),...(dm.get(r.root)?.data_types||[])].join(\' \').toLowerCase();if(!hay.includes(q))return false;}\n for(const def of facetDefs){const f=state.facets[def.key];if(!f.enabled||skip===def.key)continue;if(f.none)return false;if(f.values.size&&!def.values(r).some(v=>f.values.has(v)))return false;}\n const lo=$(\'subjectmin\').value,hi=$(\'subjectmax\').value,n=dm.get(r.root)?.subjects;\n if((lo!==\'\'||hi!==\'\')&&n==null&&!$(\'includeunknown\').checked)return false;\n if(n!=null&&((lo!==\'\'&&n<Number(lo))||(hi!==\'\'&&n>Number(hi))))return false;\n return true;\n}\nfunction updateChips(){const el=$(\'chips\');el.replaceChildren();for(const d of facetDefs){const f=state.facets[d.key];if(!f.enabled||(!f.none&&!f.values.size))continue;for(const val of f.none?[\'(none)\']:[...f.values]){const label=d.opts.find(x=>x[0]===val)?.[1]||val;const chip=h(\'span\',d.label+\': \'+label.slice(0,64),\'chip\'),b=h(\'button\',\' x\');b.title=\'Remove filter\';b.onclick=()=>{f.none=false;f.values.delete(val);renderFacets();applyFilters(true)};chip.append(b);el.append(chip)}}}\nfunction rebuild(resetPosition=true){state.base=rawRecords($(\'view\').value);applyFilters(resetPosition)}\nfunction applyFilters(resetPosition=true){\n const priorSelected=state.selected;state.filtered=state.base.filter(r=>matches(r));const countKey=$(\'view\').value===\'dataset_similarity\'?\'other\':\'root\';\n const counts=new Map();state.filtered.forEach(r=>counts.set(r[countKey],(counts.get(r[countKey])||0)+1));\n const ids=[...counts.keys()].sort((a,b)=>(counts.get(b)-counts.get(a))||a.localeCompare(b)),limit=Number($(\'limit\').value),keep=new Set(limit?ids.slice(0,limit):ids);\n state.totalMatchingStudies=ids.length;state.records=state.filtered.filter(r=>keep.has(r[countKey]));\n buildGraph();state.selected=state.nodes.some(n=>n.key===priorSelected)?priorSelected:null;\n draw();if(resetPosition)readingView();else transform();highlight();updateChips();recordTable();\n const show=keep.size,filteredPairs=uniq(state.filtered.map(r=>r.pair).filter(Boolean)).length;\n $(\'stats\').textContent=`${show.toLocaleString()} of ${ids.length.toLocaleString()} matching ${countKey===\'other\'?\'neighbor studies\':\'studies\'} displayed | ${state.nodes.length.toLocaleString()} nodes | ${state.edges.length.toLocaleString()} links | ${filteredPairs.toLocaleString()} filtered accession-NCT pairs. Display caps never change the underlying data.`;\n if(state.selected)inspect(state.nodes.find(n=>n.key===state.selected));else $(\'inspector\').classList.remove(\'open\');\n $(\'empty\').classList.toggle(\'hidden\',!!state.records.length);$(\'empty\').replaceChildren();\n if(!state.records.length){$(\'empty\').append(h(\'h2\',$(\'view\').value===\'dataset_similarity\'&&!$(\'focal\').value?\'Choose a focal dbGaP study\':\'No relationships match these filters\'),h(\'p\',\'Use Reset all or clear individual facets. Missing evidence is not replaced with inferred relationships.\'));}\n window.GX_STATS={records:state.records.length,filtered:state.filtered.length,nodes:state.nodes.length,edges:state.edges.length,matchingStudies:ids.length};\n}\nfunction buildGraph(){\n const nodes=new Map(),edges=new Map(),view=$(\'view\').value;\n function node(key,type,id,col,label){if(!nodes.has(key))nodes.set(key,{key,type,id,col,label,records:new Set(),weight:0,x:0,y:0,h:30,w:16});return nodes.get(key)}\n function addlink(a,b,r){const key=a.key+\'>\'+b.key;if(!edges.has(key))edges.set(key,{key,source:a.key,target:b.key,records:new Set(),weight:0});let e=edges.get(key);e.records.add(r.id);e.weight+=r.weight;}\n for(const r of state.records){let ns;\n if(view===\'mesh_trial\')ns=[node(\'D:\'+r.root,\'dataset\',r.root,0,dm.get(r.root).label),r.mesh?node(\'M:\'+r.mesh+($(\'pairpaths\').checked?\'|\'+r.pair:\'\'),\'mesh\',r.mesh,1,mm.get(r.mesh)?.label||r.mesh):node(\'G:\'+r.gap+($(\'pairpaths\').checked?\'|\'+r.pair:\'\'),\'gap\',r.gap,1,r.gap),node(\'T:\'+r.trial,\'trial\',r.trial,2,tm.get(r.trial).label)];\n else if(view===\'grant_trial\')ns=[r.project?node(\'P:\'+r.project,\'project\',r.project,0,r.project):node(\'G:\'+r.gap,\'gap\',r.gap,0,r.gap),node(\'D:\'+r.root,\'dataset\',r.root,1,dm.get(r.root).label),node(\'T:\'+r.trial,\'trial\',r.trial,2,tm.get(r.trial).label)];\n else ns=[node(\'F:\'+r.root,\'dataset\',r.root,0,dm.get(r.root).label),node(\'M:\'+r.mesh,\'mesh\',r.mesh,1,mm.get(r.mesh)?.label||r.mesh),node(\'N:\'+r.other,\'dataset\',r.other,2,dm.get(r.other).label)];\n for(const n of ns){n.records.add(r.id);n.weight+=r.weight}addlink(ns[0],ns[1],r);addlink(ns[1],ns[2],r);\n }\n const gs=[0,1,2].map(col=>[...nodes.values()].filter(n=>n.col===col).sort((a,b)=>b.records.size-a.records.size||a.label.localeCompare(b.label)));\n // Fixed readable row height on a pannable virtual canvas; never force every label into 660 pixels.\n const height=Math.max(560,Math.max(...gs.map(g=>g.length),0)*49+80),width=1500;\n gs.forEach((g,c)=>{g.forEach((n,i)=>{n.x=[270,746,1218][c];n.y=40+i*49;n.h=29;const manual=state.manual.get(view+\'|\'+n.key);if(manual){n.x=manual.x;n.y=manual.y}})});\n state.nodes=[...nodes.values()];state.edges=[...edges.values()];state.bounds={width,height};\n}\nfunction edgePath(e,nm){let a=nm.get(e.source),b=nm.get(e.target),x1=a.x+a.w,x2=b.x,y1=a.y+a.h/2,y2=b.y+b.h/2,mx=(x1+x2)/2;return `M ${x1} ${y1} C ${mx} ${y1} ${mx} ${y2} ${x2} ${y2}`}\nfunction draw(){\n const scene=$(\'scene\');scene.replaceChildren();const nm=new Map(state.nodes.map(n=>[n.key,n]));\n const lines=svgEl(\'g\',{id:\'edges\'}),nodes=svgEl(\'g\',{id:\'nodes\'});scene.append(lines,nodes);\n state.edges.forEach(e=>{let a=nm.get(e.source),b=nm.get(e.target),color=a.type===\'gap\'||b.type===\'gap\'?colors.gap:(a.type===\'project\'?colors.project:colors.mesh);const p=svgEl(\'path\',{d:edgePath(e,nm),stroke:color,\'stroke-width\':Math.min(15,1.4+2*Math.sqrt(e.weight)),class:\'edge\',\'data-edge\':e.key});p.append(svgEl(\'title\',{},`${e.records.size} displayed relationship record(s). Click an endpoint to inspect evidence.`));p.onclick=ev=>{ev.stopPropagation();selectNode(e.source)};lines.append(p)});\n state.nodes.forEach(n=>{const g=svgEl(\'g\',{class:\'node\',\'data-key\':n.key,transform:`translate(${n.x},${n.y})`,\'tabindex\':\'0\',\'role\':\'button\',\'aria-label\':n.id+\' \'+n.label}),left=n.col===0;\n g.append(svgEl(\'rect\',{class:\'hit\',x:left?-255:-3,y:-5,width:280,height:42}),svgEl(\'rect\',{class:\'mainrect\',width:n.w,height:n.h,rx:3,fill:colors[n.type]}));\n const maxChars=n.type===\'mesh\'?39:34,shown=n.label.length>maxChars?n.label.slice(0,maxChars-1)+\'...\':n.label,x=left?-9:25;\n const labelGroup=svgEl(\'g\',{class:\'labels\'});labelGroup.append(svgEl(\'text\',{x,y:11,\'text-anchor\':left?\'end\':\'start\'},shown));if(n.id!==n.label)labelGroup.append(svgEl(\'text\',{x,y:26,\'text-anchor\':left?\'end\':\'start\',class:\'shortid\'},n.id));g.append(labelGroup);\n g.append(svgEl(\'title\',{},n.id+\'\\n\'+n.label+\'\\n\'+n.records.size+\' displayed association records\'));\n g.onclick=e=>{e.stopPropagation();if(!state.suppressClick)selectNode(n.key)};g.onkeydown=e=>{if(e.key===\'Enter\'||e.key===\' \'){e.preventDefault();selectNode(n.key)}};\n g.onpointerdown=e=>{if($(\'dragmode\').checked&&e.button===0){e.preventDefault();e.stopPropagation();const p=clientToWorld(e.clientX,e.clientY);state.drag={kind:\'node\',id:e.pointerId,key:n.key,x:p.x,y:p.y,startx:n.x,starty:n.y,moved:false};$(\'graph\').setPointerCapture(e.pointerId)}};nodes.append(g);\n });toggleLabels();transform();\n}\nfunction applyPosition(n){let g=[...document.querySelectorAll(\'.node\')].find(e=>e.dataset.key===n.key);if(g)g.setAttribute(\'transform\',`translate(${n.x},${n.y})`);const nm=new Map(state.nodes.map(x=>[x.key,x]));document.querySelectorAll(\'.edge\').forEach((el,i)=>{const e=state.edges[i];if(e.source===n.key||e.target===n.key)el.setAttribute(\'d\',edgePath(e,nm))})}\nfunction toggleLabels(){document.querySelectorAll(\'.labels\').forEach(el=>{el.style.display=$(\'showlabels\').checked?\'\':\'none\'})}\nfunction transform(){const t=state.transform;$(\'scene\').setAttribute(\'transform\',`translate(${t.x} ${t.y}) scale(${t.k})`);$(\'zoomlabel\').textContent=Math.round(t.k*100)+\'%\'}\nfunction readingView(){const w=$(\'viewport\').clientWidth;state.transform={x:8,y:10,k:Math.min(1,w/state.bounds.width)};transform()}\nfunction fitAll(){const w=$(\'viewport\').clientWidth,h=$(\'viewport\').clientHeight;let minx=0,maxx=state.bounds.width,miny=0,maxy=state.bounds.height;for(const n of state.nodes){minx=Math.min(minx,n.x-260);maxx=Math.max(maxx,n.x+280);miny=Math.min(miny,n.y-10);maxy=Math.max(maxy,n.y+50)}const k=Math.min(w/(maxx-minx+25),h/(maxy-miny+25));state.transform={x:12-minx*k,y:12-miny*k,k};transform()}\nfunction zoom(factor,cx=$(\'viewport\').clientWidth/2,cy=$(\'viewport\').clientHeight/2){const t=state.transform,k=Math.max(.03,Math.min(5,t.k*factor)),r=k/t.k;t.x=cx-(cx-t.x)*r;t.y=cy-(cy-t.y)*r;t.k=k;transform()}\nfunction clientToWorld(x,y){const box=$(\'graph\').getBoundingClientRect(),t=state.transform;return{x:(x-box.left-t.x)/t.k,y:(y-box.top-t.y)/t.k}}\nfunction highlight(){\n const sel=state.nodes.find(n=>n.key===state.selected);let activeEdges=new Set(),activeNodes=new Set();\n if(sel){activeNodes.add(sel.key);for(const e of state.edges){if($(\'fullpaths\').checked?[...e.records].some(id=>sel.records.has(id)):(e.source===sel.key||e.target===sel.key)){activeEdges.add(e.key);activeNodes.add(e.source);activeNodes.add(e.target)}}}\n document.querySelectorAll(\'.edge\').forEach(e=>{e.classList.toggle(\'dim\',!!sel&&!activeEdges.has(e.dataset.edge));e.classList.toggle(\'active\',!!sel&&activeEdges.has(e.dataset.edge))});\n document.querySelectorAll(\'.node\').forEach(e=>{e.classList.toggle(\'dim\',!!sel&&!activeNodes.has(e.dataset.key));e.classList.toggle(\'active\',!!sel&&activeNodes.has(e.dataset.key))});\n window.GX_HIGHLIGHT={selected:state.selected,activeNodes:activeNodes.size,dimmed:sel?state.nodes.length-activeNodes.size:0};\n}\nfunction selectNode(key){state.selected=state.selected===key?null:key;highlight();const n=state.nodes.find(n=>n.key===state.selected);if(n)inspect(n);else $(\'inspector\').classList.remove(\'open\');recordTable()}\nfunction clearSelection(){state.selected=null;highlight();$(\'inspector\').classList.remove(\'open\');recordTable()}\nfunction inspect(n){\n const pane=$(\'inspector\');pane.replaceChildren();pane.classList.add(\'open\');const close=h(\'button\',\'x\',\'close\');close.onclick=clearSelection;pane.append(close,h(\'div\',n.type===\'project\'?\'NIH project association\':n.type===\'gap\'?\'Coverage / comparison status\':n.type===\'dataset\'?\'dbGaP study accession\':n.type===\'trial\'?\'NCT registration\':\'MeSH descriptor\',\'kind\'),h(\'h2\',n.label),h(\'p\',n.id,\'subtitle\'));\n const obj=n.type===\'dataset\'?dm.get(n.id):n.type===\'trial\'?tm.get(n.id):n.type===\'mesh\'?mm.get(n.id):n.type===\'project\'?gm.get(n.id):null;\n const actions=h(\'div\',\'\',\'actions\');if(obj?.url)actions.append(link(n.type===\'dataset\'?\'Open NLM / NCBI dbGaP record\':n.type===\'trial\'?\'Open ClinicalTrials.gov\':n.type===\'mesh\'?\'Open NLM MeSH record\':\'Open NIH RePORTER search\',obj.url));\n if(n.type===\'project\'&&obj?.appl_ids?.length)obj.appl_ids.slice(0,10).forEach(id=>actions.append(link(\'Application \'+id,\'https://reporter.nih.gov/project-details/\'+id)));\n const copy=h(\'button\',\'Copy identifier\');copy.onclick=async()=>{try{await navigator.clipboard.writeText(n.id);toast(\'Identifier copied\')}catch(e){window.prompt(\'Copy identifier\',n.id)}};actions.append(copy);pane.append(actions);\n if(n.type===\'dataset\')pane.append(h(\'div\',\'A root accession can describe a collection or evolving study; it is not a count of independent participants or files.\',\'note\'));\n if(n.type===\'project\')pane.append(h(\'div\',\'Association via one or more dataset-linked publications, not verified data-generation funding. Core IDs can span multiple award years. No dollars are attributed.\',\'note\'));\n if(n.type===\'trial\')pane.append(h(\'div\',\'A recorded NCT reference may occur in linked publication metadata. Trial origin and reuse are not inferred.\',\'note\'));\n if(n.type===\'mesh\')pane.append(h(\'div\',\'Exact native annotations and broader hierarchy context are distinct. Shared terms do not establish a scientific use pathway.\',\'note\'));\n if(n.type===\'gap\')pane.append(h(\'div\',\'This is a status label, not an ontology term. No external catalog identifier is invented.\',\'note\'));\n const fields=obj?.fields||{};const tb=document.createElement(\'table\');for(const[k,v]of Object.entries(fields)){const tr=document.createElement(\'tr\');tr.append(h(\'td\',k),h(\'td\',nodeText(v)));tb.append(tr)}if(Object.keys(fields).length)pane.append(h(\'h3\',\'Saved record details\'),tb);\n for(const[k,label]of [[\'mesh_labels\',\'Native MeSH\'],[\'data_types\',\'Reported data / assay labels\'],[\'assay_families\',\'Harmonized assay families\'],[\'platforms\',\'Listed repositories / platforms\'],[\'consent\',\'Consent labels\'],[\'conditions\',\'Registry condition text\'],[\'organizations\',\'Recorded project organizations\'],[\'admin_ic\',\'Administering IC\']]){\n  const vals=obj?.[k];if(vals&&(Array.isArray(vals)?vals.length:Object.keys(vals).length)){pane.append(h(\'h3\',label));const p=h(\'p\',Array.isArray(vals)?vals.join(\'; \'):Object.entries(vals).map(([id,l])=>l+\' [\'+id+\']\').join(\'; \'));p.style.fontSize=\'12px\';pane.append(p)}}\n const records=state.records.filter(r=>n.records.has(r.id));pane.append(h(\'h3\',records.length+\' displayed relationship records\'));\n const groups=uniq(records.flatMap(r=>r.basis||[]));if(groups.length)pane.append(h(\'p\',groups.join(\'; \'),\'subtitle\'));\n const evidence=uniq(records.flatMap(r=>(r.evidence||[]).filter(e=>e.pmid).map(e=>e.pmid)));if(evidence.length){pane.append(h(\'h3\',\'Publications containing NCT evidence\'));const ul=document.createElement(\'ul\');for(const pid of evidence){const li=document.createElement(\'li\');li.append(link(\'PMID \'+pid,\'https://pubmed.ncbi.nlm.nih.gov/\'+pid+\'/\'));ul.append(li)}pane.append(ul)}\n const ev=records.flatMap(r=>r.evidence||[]).filter(e=>e.pmid&&e.text);if(ev.length){pane.append(h(\'h3\',\'Example saved evidence context\'));pane.append(h(\'p\',ev[0].field,\'subtitle\'),h(\'pre\',ev[0].text));}\n const details=document.createElement(\'details\');details.append(h(\'summary\',\'Full selected-node records\'));const p=h(\'pre\',JSON.stringify(records.slice(0,25),null,2));details.append(p);pane.append(details,h(\'p\',\'The detail preview shows up to 25 records. Export filtered records for the complete list.\',\'subtitle\'));\n}\nfunction recordTable(){\n const view=$(\'view\').value,head=$(\'recordhead\'),body=$(\'recordbody\');head.replaceChildren();body.replaceChildren();\n const titles=view===\'grant_trial\'?[\'Project (paper-linked)\',\'dbGaP accession\',\'NCT\',\'Shared evidence PMIDs\',\'Relationship evidence\']:view===\'dataset_similarity\'?[\'Focal study\',\'Other study\',\'Shared native MeSH\',\'Interpretation\']:[\'dbGaP accession\',\'MeSH / status\',\'NCT\',\'Semantic layer\',\'Reference basis\'];titles.forEach(t=>head.append(h(\'th\',t)));\n const selected=state.nodes.find(n=>n.key===state.selected),records=selected?state.records.filter(r=>selected.records.has(r.id)):state.records;\n for(const r of records.slice(0,200)){const tr=document.createElement(\'tr\');const vals=view===\'grant_trial\'?[r.project||r.gap,r.root,r.trial,(r.common_pmids||[]).join(\'; \')||\'Not established\',(r.basis||[]).join(\'; \')]:view===\'dataset_similarity\'?[r.root,r.other,mm.get(r.mesh)?.label||r.mesh,\'Topical overlap; not reuse\']:[r.root,r.mesh?(mm.get(r.mesh)?.label||r.mesh):r.gap,r.trial,r.match,(r.basis||[]).join(\'; \')];vals.forEach(v=>tr.append(h(\'td\',v)));body.append(tr)}\n $(\'recordnote\').textContent=`${records.length} displayed ${selected?\'selected-node \':\'\'}records; table preview limited to 200. Export includes all ${state.filtered.length} filtered records before the display cap.`;\n}\nfunction viewChanged(){const view=$(\'view\').value,info=VIEWINFO[view];$(\'maptitle\').textContent=info.title;$(\'mapdescription\').textContent=info.description;$(\'meaning\').textContent=info.meaning;$(\'colheads\').replaceChildren(...info.heads.map(x=>h(\'span\',x)));$(\'focallabel\').classList.toggle(\'hidden\',view!==\'dataset_similarity\');$(\'pairpathslabel\').classList.toggle(\'hidden\',view!==\'mesh_trial\');$(\'samepaperlabel\').classList.toggle(\'hidden\',view!==\'grant_trial\');$(\'termscope\').value=view===\'grant_trial\'?\'any\':\'bridge\';state.selected=null;rebuild(true)}\nfunction resetAll(){for(const f of Object.values(state.facets)){f.enabled=true;f.values.clear();f.none=false;f.search=\'\'}$(\'search\').value=\'\';$(\'subjectmin\').value=\'\';$(\'subjectmax\').value=\'\';$(\'includeunknown\').checked=true;$(\'hierarchy\').checked=true;$(\'broad\').checked=true;$(\'gaps\').checked=true;$(\'samepaper\').checked=false;$(\'pairpaths\').checked=false;$(\'showlabels\').checked=true;$(\'fullpaths\').checked=true;$(\'dragmode\').checked=false;$(\'focal\').value=\'\';$(\'limit\').value=String(DATA.settings.default_limit);$(\'view\').value=DATA.settings.default_view;state.manual.clear();state.selected=null;renderFacets();viewChanged()}\n$(\'graph\').onpointerdown=e=>{if(e.button!==0||e.target.closest(\'.node,.edge\'))return;const t=state.transform;state.drag={kind:\'pan\',id:e.pointerId,x:e.clientX,y:e.clientY,startx:t.x,starty:t.y,moved:false};$(\'graph\').setPointerCapture(e.pointerId);$(\'graph\').classList.add(\'dragging\')};\n$(\'graph\').onpointermove=e=>{const d=state.drag;if(!d||d.id!==e.pointerId)return;if(Math.abs(e.clientX-d.x)+Math.abs(e.clientY-d.y)>3)d.moved=true;if(d.kind===\'pan\'){state.transform.x=d.startx+e.clientX-d.x;state.transform.y=d.starty+e.clientY-d.y;transform()}else{const p=clientToWorld(e.clientX,e.clientY),n=state.nodes.find(n=>n.key===d.key);n.x=d.startx+p.x-d.x;n.y=d.starty+p.y-d.y;state.manual.set($(\'view\').value+\'|\'+n.key,{x:n.x,y:n.y});applyPosition(n);d.moved=true}};\nfunction endDrag(e){if(!state.drag)return;const moved=state.drag.moved;state.drag=null;state.suppressClick=moved;setTimeout(()=>state.suppressClick=false,60);$(\'graph\').classList.remove(\'dragging\');try{$(\'graph\').releasePointerCapture(e.pointerId)}catch(x){}}\n$(\'graph\').onpointerup=endDrag;$(\'graph\').onpointercancel=endDrag;\n$(\'graph\').onclick=e=>{if(e.target===$(\'graph\')&&!state.suppressClick)clearSelection()};\n$(\'graph\').addEventListener(\'wheel\',e=>{e.preventDefault();const box=$(\'graph\').getBoundingClientRect();zoom(Math.exp(-e.deltaY*.0015),e.clientX-box.left,e.clientY-box.top)},{passive:false});\n$(\'zoomin\').onclick=()=>zoom(1.25);$(\'zoomout\').onclick=()=>zoom(.8);$(\'fit\').onclick=fitAll;$(\'reading\').onclick=readingView;$(\'resetview\').onclick=readingView;$(\'layoutreset\').onclick=()=>{state.manual.clear();rebuild(true)};\n$(\'clearselection\').onclick=clearSelection;$(\'reset\').onclick=resetAll;$(\'view\').onchange=viewChanged;$(\'focal\').onchange=()=>rebuild(true);$(\'limit\').onchange=()=>applyFilters(true);$(\'termscope\').onchange=()=>applyFilters(true);\nfor(const k of [\'hierarchy\',\'broad\',\'gaps\',\'samepaper\',\'pairpaths\'])$(k).onchange=()=>rebuild(true);\n$(\'showlabels\').onchange=toggleLabels;$(\'fullpaths\').onchange=highlight;\nlet timer;for(const k of [\'search\',\'subjectmin\',\'subjectmax\'])$(k).oninput=()=>{clearTimeout(timer);timer=setTimeout(()=>applyFilters(true),150)};$(\'includeunknown\').onchange=()=>applyFilters(true);\n$(\'togglefacets\').onclick=()=>{$(\'shell\').classList.toggle(\'sidebarcollapsed\');$(\'togglefacets\').textContent=$(\'shell\').classList.contains(\'sidebarcollapsed\')?\'Show facets\':\'Hide facets\';readingView()};\n$(\'enableall\').onclick=()=>{for(const f of Object.values(state.facets))f.enabled=true;renderFacets();applyFilters(true)};$(\'disableall\').onclick=()=>{for(const f of Object.values(state.facets))f.enabled=false;renderFacets();applyFilters(true)};\n$(\'exportrows\').onclick=()=>download(\'dbgap_filtered_relationships.json\',JSON.stringify({version:DATA.version,view:$(\'view\').value,records:state.filtered,view_controls:{hierarchy:$(\'hierarchy\').checked,broad:$(\'broad\').checked,gaps:$(\'gaps\').checked,samepaper:$(\'samepaper\').checked,pairpaths:$(\'pairpaths\').checked,term_scope:$(\'termscope\').value},filters:Object.fromEntries(Object.entries(state.facets).map(([k,v])=>[k,{enabled:v.enabled,values:[...v.values],none:v.none}])),search:$(\'search\').value,provenance:DATA.provenance},null,2));\n$(\'exportall\').onclick=()=>download(\'dbgap_explorer_data.json\',JSON.stringify(DATA,null,2));\n$(\'exportsvg\').onclick=()=>{const svg=$(\'graph\').cloneNode(true);let minx=0,miny=0,maxx=state.bounds.width,maxy=state.bounds.height;for(const n of state.nodes){minx=Math.min(minx,n.x-270);miny=Math.min(miny,n.y-10);maxx=Math.max(maxx,n.x+290);maxy=Math.max(maxy,n.y+50)}svg.setAttribute(\'width\',maxx-minx);svg.setAttribute(\'height\',maxy-miny);svg.setAttribute(\'viewBox\',`${minx} ${miny} ${maxx-minx} ${maxy-miny}`);svg.querySelector(\'#scene\').removeAttribute(\'transform\');const st=svgEl(\'style\',{},\'.edge{fill:none;stroke-opacity:.25}.edge.dim{stroke-opacity:.03}.edge.active{stroke-opacity:.75}.node.dim{opacity:.1}.node text{font:12px Arial;fill:#1d3449;paint-order:stroke;stroke:white;stroke-width:3px}.node .shortid{font:10px Arial}.node rect.hit{fill:transparent}\');svg.insertBefore(st,svg.firstChild);download(\'dbgap_relationships.svg\',new XMLSerializer().serializeToString(svg),\'image/svg+xml\')};\ndocument.addEventListener(\'keydown\',e=>{if(e.key===\'Escape\')clearSelection();if([\'INPUT\',\'SELECT\',\'TEXTAREA\'].includes(e.target.tagName))return;if(e.key===\'+\'||e.key===\'=\')zoom(1.2);if(e.key===\'-\')zoom(.8);if(e.key===\'0\')fitAll();if(e.key===\'ArrowLeft\'){state.transform.x+=40;transform()}if(e.key===\'ArrowRight\'){state.transform.x-=40;transform()}if(e.key===\'ArrowUp\'){state.transform.y+=40;transform()}if(e.key===\'ArrowDown\'){state.transform.y-=40;transform()}});\nDATA.datasets.slice().sort((a,b)=>a.id.localeCompare(b.id)).forEach(d=>{const o=h(\'option\',d.id+\' | \'+d.label);o.value=d.id;$(\'focal\').append(o)});\n$(\'view\').value=DATA.settings.default_view;$(\'limit\').value=String(DATA.settings.default_limit);state.base=rawRecords($(\'view\').value);facetOptions();renderFacets();viewChanged();\n$(\'sourceinfo\').textContent=`Loaded ${DATA.counts.datasets} dbGaP studies, ${DATA.counts.trials} NCT records, ${DATA.counts.pairs} accession-NCT pairs and ${DATA.counts.project_dataset_pairs} recorded project-study associations. Source mode: ${DATA.provenance.source_mode}.`;\n$(\'provenance\').textContent=JSON.stringify(DATA.provenance,null,2);\n// Test/accessibility hooks expose state, not mutation of source records.\nwindow.GX={state,DATA,setFacet:(key,values,enabled=true)=>{state.facets[key]={enabled,values:new Set(values),none:false,search:\'\'};renderFacets();applyFilters(true)},selectNode,resetAll,viewChanged,fitAll,readingView};\n</script></body></html>\n'

def gx_write(path, data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if isinstance(data,str):data=data.encode('utf-8')
    if path.is_file() and path.read_bytes()==data:return
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_bytes(data);os.replace(tmp,path)


def gx_render(bundle, height=660, width=1840):
    data=json.dumps(gx_clean(bundle),ensure_ascii=True,separators=(',',':'),allow_nan=False).replace('<','\\u003c').replace('>','\\u003e').replace('&','\\u0026')
    return GX_HTML.replace('__MAX_WIDTH__',str(int(width))).replace('__GRAPH_HEIGHT__',str(int(height))).replace('__DATA__',data)


def gx_build_explorer(base, analysis_dir='', catalog_dir='', output_dir='', verify=True, view='mesh_trial', limit=40, height=660, width=1840):
    if view not in {'mesh_trial','grant_trial','dataset_similarity'}:raise ValueError('Invalid DEFAULT_VIEW')
    if limit not in {0,20,40,80,200}:raise ValueError('DEFAULT_STUDY_LIMIT must be 0, 20, 40, 80, or 200')
    if not 360<=int(height)<=1100 or not 800<=int(width)<=2400:raise ValueError('Invalid viewport dimensions')
    src=gx_read_sources(base,analysis_dir,catalog_dir,verify)
    bundle=gx_build_bundle(src)
    bundle['settings']={'default_view':view,'default_limit':limit,'graph_height':height,'page_max_width':width}
    # The renderer identity is frozen independently of acquisition/analysis recipes.
    bundle['provenance']['builder_recipe_sha256']=GX_CODE_SHA256
    bundle['provenance']['renderer_sha256']=hashlib.sha256(GX_HTML.encode()).hexdigest()
    signature=hashlib.sha256(json.dumps(gx_clean(bundle),sort_keys=True,ensure_ascii=True).encode()).hexdigest()
    out=Path(output_dir) if output_dir else Path(base)/'interactive_explorer'
    run=out/signature[:16];run.mkdir(parents=True,exist_ok=True)
    files={'explorer.html':gx_render(bundle,height,width),'graph_data.json':json.dumps(gx_clean(bundle),indent=2,ensure_ascii=True,allow_nan=False)}
    for name,content in files.items():gx_write(run/name,content)
    checks=[{'check':'Dataset identifiers unique','passed':len({d['id'] for d in bundle['datasets']})==len(bundle['datasets'])},
            {'check':'All pair endpoints exist','passed':all(p['root'] in {d['id'] for d in bundle['datasets']} and p['trial'] in {t['id'] for t in bundle['trials']} for p in bundle['pairs'])},
            {'check':'All source hashes unchanged','passed':all(gx_hash(r['path'])==r['sha256'] for r in src['inventory'])},
            {'check':'No source collection performed','passed':True}]
    if not all(c['passed'] for c in checks):raise ValueError('Graph validation failed: '+str(checks))
    manifest={'version':EXPLORER_VERSION,'signature':signature,'counts':bundle['counts'],'checks':checks,
              'inputs':src['inventory'],'outputs':{name:gx_hash(run/name) for name in files},'warnings':bundle['provenance']['warnings']}
    gx_write(run/'manifest.json',json.dumps(manifest,indent=2))
    gx_write(out/'latest.json',json.dumps({'html':str((run/'explorer.html').resolve()),'signature':signature,'manifest':str((run/'manifest.json').resolve())},indent=2))
    print('Explorer:',run/'explorer.html')
    print('Saved records:',bundle['counts'])
    print('Default view is capped at',limit or 'all','matching studies for readability; set Show studies = All to remove the display cap.')
    for w in bundle['provenance']['warnings']:print('NOTE:',w)
    if bundle['counts']['project_dataset_pairs'] and not bundle['counts']['project_links_with_PMID_evidence']:
        print('NOTE: The catalog supplies project-study summaries but no per-project PMID evidence. Same supporting PMID mode will correctly be empty until source link rows are available.')
    return run/'explorer.html',bundle,manifest


def gx_main():
    if MOUNT_DRIVE and DRIVE_ROOT.startswith('/content/drive/'):
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except ImportError:pass
    target,bundle,manifest=gx_build_explorer(DRIVE_ROOT,ANALYSIS_DIR,CATALOG_DIR,EXPLORER_OUTPUT,
         VERIFY_ANALYSIS_MANIFEST,DEFAULT_VIEW,DEFAULT_STUDY_LIMIT,GRAPH_HEIGHT,PAGE_MAX_WIDTH)
    try:
        from IPython.display import display,HTML
        display(HTML('<p><b>Interactive explorer created.</b> Open the downloaded HTML in Chrome, Edge, or Firefox. The 1920-pixel layout is easier to use outside the narrow Colab output pane.</p>'))
        if DOWNLOAD_HTML:
            from google.colab import files as colab_files
            colab_files.download(str(target))
    except ImportError:pass
    return target


if globals().get('BUILD_INTERACTIVE_EXPLORER', True) and __name__ == '__main__' and not globals().get('GX_SUPPRESS_MAIN',False):
    gx_main()
